In [ ]:
# -*- coding: utf-8 -*-
"""SpVAR_Colombia_agro_manufactura_datos_reales (2).ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1EwcoOkvO-zSTNkgHJyQ_M-svG1yJLm_k

# SpVAR departamental: actividad agropecuaria ↔ industria manufacturera (Colombia)

**Modelo VAR Espacial (SpVAR)** estimado por Máxima Verosimilitud, con selección automática de especificación
espacial (OLS, SAR, SEM, SAC, SDM) y de matriz de pesos **W** (Queen, Rook, k-vecinos, distancia inversa),
covariables climáticas reales, descomposición de los 4 mecanismos de transmisión, mapas, auditoría e interfaz Gradio.

**Datos reales, descargados automáticamente (no hay que subir archivos)**
- **Y1 · agro**: valor agregado de *Agricultura, ganadería, caza, silvicultura y pesca*.
- **Y2 · ind**: valor agregado de *Industrias manufactureras*.
- Ambas de **DANE, Cuentas Nacionales Departamentales, base 2015, precios constantes** (volumen encadenado), los
  33 departamentos, 2005 en adelante. Vienen del mismo anexo oficial: misma metodología, sin reserva estadística
  ni cambios de clasificación. El cuaderno lee además el anexo *por departamento* y compara ambos (validación cruzada).
- **Clima**: precipitación y temperatura anual de **NASA POWER** (MERRA-2), API pública sin clave, promediadas
  sobre varios puntos interiores de cada departamento; entran como anomalías estandarizadas.
- **Geometría**: geoBoundaries ADM1 (o el MGN del DANE si se indica `RUTA_MGN`).

Si una descarga falla, el cuaderno se detiene y explica la causa: **no usa datos sintéticos de respaldo**.
`MODO_DATOS='demo'` sigue disponible solo para validar el método con un DGP conocido.

**Por qué manufactura total y no solo alimentos:** la industria de alimentos por departamento solo existe en la
Encuesta Anual Manufacturera, con reserva estadística en varios departamentos y diseños de archivo que cambian
cada año. La manufactura total de las cuentas departamentales es completa y comparable; incluye la agroindustria,
pero también otras ramas, y así se reporta en las interpretaciones.

**Especificación por ecuación** (k = variable propia, j = la otra; Y en tasas de crecimiento):

$$y_{k,t} = \rho_k W y_{k,t} + \phi_{kk} y_{k,t-1} + \phi_{kj} y_{j,t-1} + \beta_{kj} y_{j,t}^{\dagger}
  + \gamma_k' S_t + W\big[\psi_{kk} y_{k,t-1} + \psi_{kj} y_{j,t-1} + \theta_{kj} y_{j,t}^{\dagger} + \delta_k' S_t\big]
  + \alpha_t + u_{k,t},\qquad u_{k,t} = \lambda_k W u_{k,t} + \varepsilon_{k,t}$$

(† solo en la ecuación "aguas abajo" según el orden recursivo de identificación agro → ind; S = covariables climáticas.)

## Celda 1 · Instalación automática de librerías (solo se ejecuta en Colab)
"""

import sys, subprocess, importlib, importlib.util

EN_COLAB = 'google.colab' in sys.modules
PAQUETES = {               # módulo -> paquete pip
    'geopandas': 'geopandas', 'libpysal': 'libpysal', 'esda': 'esda', 'spreg': 'spreg',
    'folium': 'folium', 'mapclassify': 'mapclassify', 'rasterio': 'rasterio',
    'statsmodels': 'statsmodels', 'pyflakes': 'pyflakes', 'ee': 'earthengine-api',
    'openpyxl': 'openpyxl', 'plotly': 'plotly', 'gradio': 'gradio',
}

def instalar_faltantes(paquetes=PAQUETES):
    def _falta(mod):
        try:
            return importlib.util.find_spec(mod) is None
        except (ValueError, ImportError):
            return mod not in sys.modules
    faltan = [pip for mod, pip in paquetes.items() if _falta(mod)]
    if faltan and EN_COLAB:
        print('Instalando:', ' '.join(faltan))
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *faltan], check=False)
        importlib.invalidate_caches()
    elif faltan:
        print('Fuera de Colab; paquetes no instalados (el pipeline usará rutas alternativas):', faltan)
    else:
        print('Todas las librerías están disponibles.')

instalar_faltantes()

"""## Celda 2 · Importaciones y CONFIGURACIÓN (edite aquí)"""

import os, json, ast, math, time, warnings, itertools, unicodedata
from dataclasses import dataclass, field
import numpy as np
import pandas as pd
from scipy import optimize, stats, linalg
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 40)

def _try_import(nombre):
    try:
        return importlib.import_module(nombre)
    except Exception:
        return None

gpd = _try_import('geopandas')
libpysal = _try_import('libpysal')
folium = _try_import('folium')
spreg = _try_import('spreg')
sm_tsa = _try_import('statsmodels.tsa.stattools')
try:
    from IPython.display import display, HTML
except Exception:                                        # ejecución como script
    display = print
    HTML = str

CONFIG = dict(
    # ---------------- Datos ----------------
    MODO_DATOS='dane',                      # 'dane' (descarga oficial) | 'csv' | 'demo' (solo validación)
    RUTA_CSV='/content/panel_departamental.csv',
    COL_DPTO='cod_dpto', COL_ANIO='anio', COL_AGRO='agro', COL_IND='ind_alim',
    ANIOS=(2005, 2025),                     # rango de años (niveles); se pierden 2 por diferencia + rezago
    TRANSFORMACION='auto',                  # 'dlog' | 'dasinh' | 'auto' (asinh si hay ceros)
    ORDEN=('agro', 'ind'),                  # identificación recursiva: el 1º afecta contemporáneamente al 2º
    # ---------------- Satélite ----------------
    FUENTE_SAT='nasa_power',                # 'nasa_power' | 'gee' | 'rasterio' | 'csv' | 'demo' | 'ninguna'
    GEE_PROYECTO='su-proyecto-gcp',         # ID del proyecto Google Cloud registrado en Earth Engine
    RASTER_DIR='/content/rasters', RASTER_PATRON='ndvi_{anio}.tif', RASTER_ESCALA=1.0,
    RUTA_CSV_SAT='/content/ndvi_departamental.csv',   # columnas: cod_dpto, anio, ndvi[, ndvi_crop, crop_frac]
    VARS_SAT=['precip_anom', 'temp_anom'],  # covariables climáticas que entran a ambas ecuaciones
    RUTA_MGN='/content/MGN_DPTO.zip',       # opcional: capa de departamentos del MGN (DANE) para la app
    # ---------------- Especificación ----------------
    EFECTOS_TIEMPO=True,                    # dummies de año (choques comunes nacionales)
    EFECTOS_DPTO=False,                     # dummies departamentales (cuidado: sesgo de Nickell con T corto)
    W_CANDIDATAS=['queen', 'rook', 'knn4', 'knn6', 'idw1', 'idw2'],
    MODELOS=['OLS', 'SAR', 'SEM', 'SAC', 'SDM'],
    CRITERIO='BIC',                         # 'BIC' | 'AIC' | 'ELHORST'
    ALFA=0.05,
    # ---------------- Inferencia y dinámica ----------------
    N_SIM_IMPACTOS=1000, N_SIM_IRF=300, HORIZONTE_IRF=8,
    DPTO_CHOQUE='50',                       # código DANE del departamento para el choque localizado (50=Meta)
    N_PERM=499, SEMILLA=2026,
    JACKKNIFE=True,
    VALIDAR_SPREG=True,                     # comparación con spreg (con límite de 90 s)
    FONDO_MAPA=None,                        # None = sin fondo (recomendado en Colab) | 'Esri.WorldGrayCanvas'
    OUT_DIR='/content/spvar_resultados' if EN_COLAB else './spvar_resultados',
)
os.makedirs(CONFIG['OUT_DIR'], exist_ok=True)
RNG = np.random.default_rng(CONFIG['SEMILLA'])
NOMBRES_VAR = {'agro': 'Agropecuario', 'ind': 'Industria manufacturera'}
print('Modo:', CONFIG['MODO_DATOS'], '| Satélite:', CONFIG['FUENTE_SAT'], '| Salidas en', CONFIG['OUT_DIR'])

"""## Celda 3 · Geometría departamental (geoBoundaries ADM1 → códigos DANE)
Si la descarga falla, se usa una tabla embebida de centroides y una contigüidad Queen derivada de la topología
de geoBoundaries, de modo que el modelo siempre corre (los mapas pasan a modo de puntos).
"""

URL_GEO = ('https://media.githubusercontent.com/media/wmgeolab/geoBoundaries/9469f09/releaseData/'
           'gbOpen/COL/ADM1/geoBoundaries-COL-ADM1_simplified.geojson')

# ISO 3166-2 -> (código DANE, nombre, lat, lon del centroide de área)
TABLA_DPTOS = {
    'CO-ANT': ('05', 'Antioquia', 6.926, -75.571), 'CO-ATL': ('08', 'Atlántico', 10.677, -74.965),
    'CO-DC':  ('11', 'Bogotá D.C.', 4.314, -74.182), 'CO-BOL': ('13', 'Bolívar', 8.744, -74.507),
    'CO-BOY': ('15', 'Boyacá', 5.778, -73.100), 'CO-CAL': ('17', 'Caldas', 5.342, -75.306),
    'CO-CAQ': ('18', 'Caquetá', 0.800, -73.962), 'CO-CAU': ('19', 'Cauca', 2.396, -76.826),
    'CO-CES': ('20', 'Cesar', 9.537, -73.518), 'CO-COR': ('23', 'Córdoba', 8.359, -75.795),
    'CO-CUN': ('25', 'Cundinamarca', 4.824, -74.098), 'CO-CHO': ('27', 'Chocó', 5.940, -76.938),
    'CO-HUI': ('41', 'Huila', 2.571, -75.586), 'CO-LAG': ('44', 'La Guajira', 11.476, -72.430),
    'CO-MAG': ('47', 'Magdalena', 10.249, -74.261), 'CO-MET': ('50', 'Meta', 3.348, -72.955),
    'CO-NAR': ('52', 'Nariño', 1.570, -77.871), 'CO-NSA': ('54', 'Norte de Santander', 8.095, -72.881),
    'CO-QUI': ('63', 'Quindío', 4.455, -75.690), 'CO-RIS': ('66', 'Risaralda', 5.084, -75.842),
    'CO-SAN': ('68', 'Santander', 6.696, -73.487), 'CO-SUC': ('70', 'Sucre', 9.065, -75.112),
    'CO-TOL': ('73', 'Tolima', 4.034, -75.256), 'CO-VAC': ('76', 'Valle del Cauca', 3.858, -76.518),
    'CO-ARA': ('81', 'Arauca', 6.570, -70.969), 'CO-CAS': ('85', 'Casanare', 5.403, -71.602),
    'CO-PUT': ('86', 'Putumayo', 0.458, -75.864), 'CO-SAP': ('88', 'San Andrés y Providencia', 12.910, -81.562),
    'CO-AMA': ('91', 'Amazonas', -1.542, -71.499), 'CO-GUA': ('94', 'Guainía', 2.726, -68.813),
    'CO-GUV': ('95', 'Guaviare', 1.924, -72.128), 'CO-VAU': ('97', 'Vaupés', 0.646, -70.561),
    'CO-VID': ('99', 'Vichada', 4.713, -69.415),
}
# Contigüidad Queen de respaldo (vértices compartidos en geoBoundaries); San Andrés es isla (se trata en W).
QUEEN_RESPALDO = {
    '05': ['17', '13', '23', '27', '68', '66', '15'], '08': ['13', '47'], '11': ['50', '25', '41'],
    '13': ['05', '08', '20', '47', '70', '23', '68'], '15': ['17', '50', '25', '05', '54', '68', '81', '85'],
    '17': ['25', '73', '05', '66', '15'], '18': ['91', '50', '97', '86', '95', '19', '41'],
    '19': ['73', '76', '86', '52', '18', '41'], '20': ['13', '47', '44', '54', '68'], '23': ['05', '13', '70'],
    '25': ['17', '50', '73', '11', '15', '85', '41'], '27': ['05', '76', '66'],
    '41': ['50', '25', '73', '11', '19', '18'], '44': ['20', '47'], '47': ['08', '13', '20', '44'],
    '50': ['25', '99', '95', '11', '15', '85', '18', '41'], '52': ['86', '19'], '54': ['20', '68', '15'],
    '63': ['73', '76', '66'], '66': ['17', '73', '05', '27', '76', '63'], '68': ['05', '13', '20', '54', '15'],
    '70': ['13', '23'], '73': ['17', '25', '76', '63', '66', '19', '41'], '76': ['73', '27', '63', '66', '19'],
    '81': ['99', '15', '85'], '85': ['50', '25', '99', '81', '15'], '86': ['91', '52', '19', '18'], '88': [],
    '91': ['97', '86', '18'], '94': ['99', '97', '95'], '95': ['50', '99', '97', '94', '18'],
    '97': ['91', '95', '94', '18'], '99': ['50', '95', '94', '81', '85'],
}

def _norm(s):
    return unicodedata.normalize('NFKD', str(s)).encode('ascii', 'ignore').decode().lower().strip()

def cargar_geometria():
    """Devuelve (gdf o None, DataFrame base con cod_dpto, nombre, lat, lon) ordenado por código DANE."""
    base = pd.DataFrame([(v[0], v[1], v[2], v[3], k) for k, v in TABLA_DPTOS.items()],
                        columns=['cod_dpto', 'nombre', 'lat', 'lon', 'iso'])
    base = base.sort_values('cod_dpto').reset_index(drop=True)
    if gpd is None:
        print('geopandas no disponible: se usan centroides embebidos (mapas en modo puntos).')
        return None, base
    try:
        g = gpd.read_file(URL_GEO)
        g = g.rename(columns={'shapeISO': 'iso'})[['iso', 'geometry']]
        g = g.merge(base, on='iso', how='inner').sort_values('cod_dpto').reset_index(drop=True)
        if len(g) != 33:
            raise ValueError(f'Se esperaban 33 departamentos y llegaron {len(g)}')
        g = g.set_crs(4326, allow_override=True)
        try:
            cent = g.to_crs(9377).centroid.to_crs(4326)      # MAGNA-SIRGAS Origen-Nacional
        except Exception:
            cent = g.to_crs(3116).centroid.to_crs(4326)      # MAGNA-SIRGAS Colombia Bogotá
        g['lat'], g['lon'] = cent.y.values, cent.x.values
        print(f'Geometría cargada: {len(g)} departamentos (geoBoundaries ADM1).')
        return g, pd.DataFrame(g.drop(columns='geometry'))
    except Exception as e:
        print('No se pudo descargar la geometría, se usa el respaldo embebido:', e)
        return None, base

GDF, DPTOS = cargar_geometria()
DPTOS.head()

"""## Celda 4 · Constructor de matrices de pesos espaciales candidatas
Todas las W se estandarizan por filas. Las islas (San Andrés en contigüidad) se conectan a su vecino más
cercano por distancia geodésica, práctica estándar para evitar filas nulas.
"""

def haversine_km(lat, lon):
    la, lo = np.radians(lat)[:, None], np.radians(lon)[:, None]
    d = np.sin((la - la.T) / 2) ** 2 + np.cos(la) * np.cos(la.T) * np.sin((lo - lo.T) / 2) ** 2
    return 2 * 6371.0 * np.arcsin(np.sqrt(np.clip(d, 0, 1)))

def estandarizar_filas(B):
    B = np.asarray(B, float).copy()
    np.fill_diagonal(B, 0.0)
    s = B.sum(1, keepdims=True)
    if np.any(s == 0):
        raise ValueError('W con filas nulas (islas sin tratar)')
    return B / s

def conectar_islas(B, D):
    B = B.copy()
    for i in np.where(B.sum(1) == 0)[0]:
        d = D[i].copy(); d[i] = np.inf
        j = int(np.argmin(d)); B[i, j] = B[j, i] = 1.0
    return B

def contiguidad(codigos, tipo='queen'):
    """Matriz binaria de contigüidad. Usa libpysal+geometría; si no, el respaldo Queen embebido."""
    n = len(codigos)
    if GDF is not None and libpysal is not None:
        g = GDF.set_index('cod_dpto').loc[list(codigos)].reset_index()
        from libpysal.weights import Queen, Rook
        cls = Queen if tipo == 'queen' else Rook
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            w = cls.from_dataframe(g, use_index=False)
        B, _ = w.full()
        return np.asarray(B, float), 'libpysal'
    if tipo == 'rook':
        return None, 'sin geometría'          # sin polígonos no se distingue rook de queen
    idx = {c: i for i, c in enumerate(codigos)}
    B = np.zeros((n, n))
    for c in codigos:
        for v in QUEEN_RESPALDO.get(c, []):
            if v in idx:
                B[idx[c], idx[v]] = B[idx[v], idx[c]] = 1.0
    return B, 'respaldo'

def construir_W(codigos, lat, lon, candidatas):
    """Devuelve dict nombre -> {'W': estandarizada, 'B': cruda (para jackknife), 'desc': texto}."""
    D = haversine_km(np.asarray(lat), np.asarray(lon))
    salida = {}
    for nombre in candidatas:
        try:
            if nombre in ('queen', 'rook'):
                B, fuente = contiguidad(codigos, nombre)
                if B is None:
                    print(f'  · {nombre}: omitida ({fuente})'); continue
                B = conectar_islas(B, D)
                desc = f'Contigüidad {nombre} ({fuente})'
            elif nombre.startswith('knn'):
                k = int(nombre[3:])
                B = np.zeros_like(D)
                for i in range(len(D)):
                    d = D[i].copy(); d[i] = np.inf
                    B[i, np.argsort(d)[:k]] = 1.0
                desc = f'{k} vecinos más cercanos'
            elif nombre.startswith('idw'):
                a = float(nombre[3:])
                with np.errstate(divide='ignore'):
                    B = np.where(D > 0, 1.0 / D ** a, 0.0)
                desc = f'Distancia inversa (potencia {a:g}, densa)'
            else:
                raise ValueError(nombre)
            np.fill_diagonal(B, 0.0)
            salida[nombre] = {'W': estandarizar_filas(B), 'B': B, 'D': D, 'desc': desc}
        except Exception as e:
            print(f'  · {nombre}: error -> {e}')
    # Elimina candidatas idénticas (p. ej. rook == queen)
    unicas = {}
    for k, v in salida.items():
        if not any(np.allclose(v['W'], u['W']) for u in unicas.values()):
            unicas[k] = v
        else:
            print(f'  · {k}: idéntica a otra candidata, se descarta')
    return unicas

def resumen_W(Ws):
    filas = []
    for k, v in Ws.items():
        B = v['B'] > 0
        ev = np.linalg.eigvals(v['W'])
        filas.append({'W': k, 'descripción': v['desc'], 'vecinos_medios': B.sum(1).mean(),
                      'vecinos_min': B.sum(1).min(), 'vecinos_max': B.sum(1).max(),
                      'simétrica_binaria': bool(np.allclose(B, B.T)),
                      'autovalor_min_real': ev.real.min()})
    return pd.DataFrame(filas)

"""## Celda 5 · Datos económicos reales: descarga automática del DANE (o CSV propio / demo de validación)"""

# Parámetros verdaderos del DGP de demostración (se usan luego en la auditoría de recuperación de parámetros)
DGP_VERDADERO = {
    'W': 'queen',
    'agro': {'modelo': 'SAR', 'rho': 0.35, 'lam': 0.0, 'sigma': 0.030, 'const': 0.010,
             'coef': {'agro_L1': 0.30, 'ind_L1': 0.05, 'ndvi_anom': 0.012}},
    'ind':  {'modelo': 'SDM', 'rho': 0.25, 'lam': 0.0, 'sigma': 0.025, 'const': 0.002,
             'coef': {'ind_L1': 0.20, 'agro_L1': 0.10, 'agro_t': 0.35, 'ndvi_anom': 0.0,
                      'W_ind_L1': 0.12, 'W_agro_L1': 0.12, 'W_agro_t': 0.25, 'W_ndvi_anom': 0.004}},
}
EVENTOS_ENOS = {2007: -0.5, 2009: -0.8, 2010: 1.2, 2011: 1.0, 2015: -1.3, 2016: -0.9, 2019: -0.4,
                2020: 0.5, 2021: 0.7, 2022: 0.8, 2023: -1.0}   # + La Niña (más verde) / − El Niño (seco)

def _zscore_por_dpto(df, col):
    g = df.groupby('cod_dpto')[col]
    return (df[col] - g.transform('mean')) / g.transform('std').replace(0, np.nan)

def simular_ndvi(codigos, lat, W, anios, rng):
    """NDVI anual sintético con componente ENOS común y ruido espacialmente correlacionado."""
    N = len(codigos)
    base = np.clip(0.55 + 0.04 * np.cos(np.radians(lat) * 8) + rng.normal(0, 0.06, N), 0.3, 0.85)
    crop = np.clip(rng.beta(2, 6, N), 0.01, 0.6)
    Ainv = np.linalg.inv(np.eye(N) - 0.5 * W)
    filas = []
    for a in anios:
        u = Ainv @ rng.normal(0, 0.6, N)
        ndvi = base + 0.03 * (EVENTOS_ENOS.get(a, rng.normal(0, 0.3)) + u)
        filas += [{'cod_dpto': c, 'anio': a, 'ndvi': ndvi[i],
                   'ndvi_crop': ndvi[i] - 0.03 + rng.normal(0, 0.01),
                   'crop_frac': crop[i] * (1 + 0.005 * (a - anios[0]))} for i, c in enumerate(codigos)]
    return pd.DataFrame(filas)

def simular_panel_demo(dptos, rng, anios=CONFIG['ANIOS'], quemado=30):
    """Genera niveles de agro e industria alimentaria desde un SpVAR recursivo (agro → ind) con W queen."""
    cods = dptos['cod_dpto'].tolist()
    Wd = construir_W(cods, dptos['lat'], dptos['lon'], [DGP_VERDADERO['W']])[DGP_VERDADERO['W']]['W']
    N = len(cods)
    a0, a1 = anios
    anios_niv = list(range(a0, a1 + 1))
    sat = simular_ndvi(cods, dptos['lat'].values, Wd, anios_niv, rng)
    sat['ndvi_anom'] = _zscore_por_dpto(sat, 'ndvi')
    Z = sat.pivot(index='anio', columns='cod_dpto', values='ndvi_anom')[cods]
    pa, pi = DGP_VERDADERO['agro'], DGP_VERDADERO['ind']
    Aa_inv = np.linalg.inv(np.eye(N) - pa['rho'] * Wd)
    Ai_inv = np.linalg.inv(np.eye(N) - pi['rho'] * Wd)
    ya, yi = np.full(N, 0.03), np.full(N, 0.03)
    niv_a = np.exp(rng.normal(np.log(2000), 1.0, N))      # miles de millones COP constantes
    niv_i = niv_a * np.exp(rng.normal(-0.3, 0.8, N))
    registros = [(a0, niv_a.copy(), niv_i.copy())]
    for t in range(quemado + len(anios_niv) - 1):
        anio = a0 + 1 + t - quemado
        z = Z.loc[anio].values if anio in Z.index else rng.normal(0, 1, N)
        ca, ci = pa['coef'], pi['coef']
        alfa_a, alfa_i = rng.normal(0, 0.01), rng.normal(0, 0.008)
        xa = pa['const'] + alfa_a + ca['agro_L1'] * ya + ca['ind_L1'] * yi + ca['ndvi_anom'] * z
        ya_n = Aa_inv @ (xa + rng.normal(0, pa['sigma'], N))
        xi = (pi['const'] + alfa_i + ci['ind_L1'] * yi + ci['agro_L1'] * ya + ci['agro_t'] * ya_n
              + ci['ndvi_anom'] * z + Wd @ (ci['W_ind_L1'] * yi + ci['W_agro_L1'] * ya
                                             + ci['W_agro_t'] * ya_n + ci['W_ndvi_anom'] * z))
        yi_n = Ai_inv @ (xi + rng.normal(0, pi['sigma'], N))
        ya, yi = ya_n, yi_n
        if anio > a0:
            niv_a, niv_i = niv_a * np.exp(ya), niv_i * np.exp(yi)
            registros.append((anio, niv_a.copy(), niv_i.copy()))
    filas = [{'cod_dpto': c, 'anio': a, 'agro': na[i], 'ind': ni[i]}
             for a, na, ni in registros for i, c in enumerate(cods)]
    return pd.DataFrame(filas), sat.drop(columns='ndvi_anom')

# =====================================================================================
# Datos reales: DANE, Cuentas Nacionales Departamentales (base 2015) — descarga automática
# =====================================================================================
import re, hashlib, urllib.request, urllib.parse

DANE_PAGINA = ('https://www.dane.gov.co/index.php/estadisticas-por-tema/cuentas-nacionales/'
               'cuentas-nacionales-departamentales')
DANE_RESPALDO = {   # URL verificadas en la página oficial del DANE (20-oct-2026)
    'actividad': 'https://www.dane.gov.co/files/operaciones/PIB/anex-PIBDep-Activecono-2025pr.xlsx',
    'departamento': 'https://www.dane.gov.co/files/operaciones/PIB/anex-PIBDep-departamento-2025pr.xlsx',
}
DANE_PATRON = {'actividad': r'anex-PIBDep-Activecono-[\w\-]+\.xlsx', 'departamento': r'anex-PIBDep-departamento-[\w\-]+\.xlsx'}
RX_AGRO = r'agricultura.*ganader'
RX_MANUF = r'industrias? manufacturer'
HOJAS_EXCLUIR = ('variaci', 'participaci', 'contribuci', 'deflactor', 'indice', 'per capita', 'percapita', 'ficha',
                 'contenido', 'nota', 'tasa')
_UA = {'User-Agent': 'Mozilla/5.0 (SpVAR-Colombia; actividad academica)'}

def _plano(s):
    s = unicodedata.normalize('NFKD', str(s)).encode('ascii', 'ignore').decode().lower()
    return re.sub(r'\s+', ' ', re.sub(r'[^a-z0-9 ]', ' ', s)).strip()

_NOMBRE_COD = {_plano(n): c for c, n in DPTOS[['cod_dpto', 'nombre']].itertuples(index=False)}
_NOMBRE_COD.update({_plano(k): v for k, v in {
    'Bogotá, D. C.': '11', 'Bogotá D.C.': '11', 'Bogotá': '11', 'Santafé de Bogotá': '11', 'Bogota DC': '11',
    'Archipiélago de San Andrés, Providencia y Santa Catalina': '88', 'San Andrés': '88',
    'San Andrés, Providencia y Santa Catalina': '88', 'Archipiélago de San Andrés': '88',
    'Valle': '76', 'Guajira': '44', 'N. de Santander': '54', 'Norte Santander': '54', 'Quindio': '63'}.items()})

def cod_departamento(v, permitir_numero=False):
    """Nombre o código → código DANE de 2 dígitos (None si no es un departamento)."""
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return None
    s = str(v).strip()
    if permitir_numero and re.fullmatch(r'\d{1,2}(\.0)?', s):
        c = f'{int(float(s)):02d}'
        return c if c in _NOMBRE_COD.values() else None
    p = _plano(s)
    if p in _NOMBRE_COD:
        return _NOMBRE_COD[p]
    cand = {c for n, c in _NOMBRE_COD.items() if len(n) >= 5 and (p.startswith(n + ' ') or p == n)}
    return cand.pop() if len(cand) == 1 else None

def _anio(c):
    if isinstance(c, (int, np.integer)) and 1990 <= c <= 2040:
        return int(c), ''
    if isinstance(c, float) and np.isfinite(c) and c.is_integer() and 1990 <= c <= 2040:
        return int(c), ''
    m = re.fullmatch(r'\s*((?:19|20)\d{2})\s*(p|pr|py)?\s*\*?\s*', str(c), flags=re.I) if isinstance(c, str) else None
    return (int(m.group(1)), (m.group(2) or '').lower()) if m else (None, '')

def _descargar(url, carpeta, intentos=4):
    os.makedirs(carpeta, exist_ok=True)
    destino = os.path.join(carpeta, url.rsplit('/', 1)[-1])
    ultimo = None
    for k in range(intentos):
        try:
            req = urllib.request.Request(url, headers=_UA)
            with urllib.request.urlopen(req, timeout=90) as r:
                datos = r.read()
            if not datos.startswith(b'PK'):
                raise ValueError('la respuesta no es un archivo XLSX (¿página de error?)')
            with open(destino, 'wb') as fh:
                fh.write(datos)
            return destino, hashlib.sha256(datos).hexdigest()
        except Exception as e:
            ultimo = e
            time.sleep(2 ** k)
    raise RuntimeError(f'No se pudo descargar {url}: {ultimo}')

def _descubrir(tipo):
    """Busca en la página oficial la versión más reciente del anexo; si falla, usa la URL verificada."""
    try:
        req = urllib.request.Request(DANE_PAGINA, headers=_UA)
        html_ = urllib.request.urlopen(req, timeout=60).read().decode('utf-8', 'ignore')
        hrefs = {urllib.parse.urljoin(DANE_PAGINA, h) for h in re.findall(r'href=["\']([^"\']+)["\']', html_)
                 if re.search(DANE_PATRON[tipo], h)}
        if hrefs:
            return sorted(hrefs, key=lambda u: re.findall(r'(\d{4})', u)[-1] if re.findall(r'(\d{4})', u) else '')[-1]
    except Exception as e:
        print(f'  (no se pudo leer la página del DANE: {e}; se usa la URL verificada)')
    return DANE_RESPALDO[tipo]

def departamento_en_texto(texto):
    """Departamento mencionado dentro de un texto (p. ej. «Departamento de Antioquia»); el nombre más largo gana
    (Norte de Santander antes que Santander, Valle del Cauca antes que Cauca)."""
    p = ' ' + _plano(texto) + ' '
    hits = [(len(n), c) for n, c in _NOMBRE_COD.items() if len(n) >= 4 and f' {n} ' in p]
    if not hits:
        return None
    largo = max(h[0] for h in hits)
    cods = {c for l, c in hits if l == largo}
    return cods.pop() if len(cods) == 1 else None

def _precios(t):
    return ('corrientes' if 'corriente' in t else
            'constantes' if ('constante' in t or 'encadenad' in t or 'volumen' in t) else None)

_NO_NIVEL = ('variacion', 'participacion', 'contribucion', 'deflactor', 'per capita', 'indice de', 'tasa de')

def _filas_anios(df):
    """Índices de todas las filas de encabezado con ≥ 5 años (una hoja puede tener varios bloques)."""
    out = {}
    for i in range(len(df)):
        cc = {j: _anio(v) for j, v in enumerate(df.iloc[i].tolist())}
        cc = {j: a for j, a in cc.items() if a[0]}
        if len(cc) >= 5:
            out[i] = cc
    return out

def leer_tablas_anuales(ruta):
    """Lector de tablas con años en columnas. Cada hoja se divide en bloques (uno por fila de años). Para cada
    bloque se detectan: precios y departamento en el título propio del bloque o de la hoja; departamento como
    columna o como fila de sección; actividad como columna o en el título. Guarda la procedencia de cada valor."""
    libros = pd.read_excel(ruta, sheet_name=None, header=None)
    regs, diag = [], []
    for hoja, df in libros.items():
        if any(t in _plano(hoja) for t in HOJAS_EXCLUIR):
            continue
        cabeceras = _filas_anios(df)
        if not cabeceras:
            continue
        inicios = sorted(cabeceras)
        titulo_hoja = ' '.join(str(v) for v in df.iloc[:inicios[0]].values.ravel() if not pd.isna(v))
        for b, h0 in enumerate(inicios):
            cols = cabeceras[h0]
            fin = inicios[b + 1] if b + 1 < len(inicios) else len(df)
            # título del bloque: filas sin cifras inmediatamente encima de su cabecera
            k = h0 - 1
            limite = inicios[b - 1] if b else -1
            while k > limite and not any(isinstance(x, (int, float, np.integer, np.floating)) and not pd.isna(x)
                                         for x in df.iloc[k, min(cols):].tolist()):
                k -= 1
            titulo_bloque = ' '.join(str(v) for v in df.iloc[k + 1:h0].values.ravel() if not pd.isna(v))
            tb, th = _plano(titulo_bloque), _plano(hoja + ' ' + titulo_hoja)
            if any(t in tb or (b == 0 and t in th) for t in _NO_NIVEL):
                continue
            precios = _precios(tb) or _precios(th)
            primera = min(cols)
            cab = [_plano(v) if not pd.isna(v) else '' for v in df.iloc[h0, :primera]]
            cuerpo = df.iloc[h0 + 1:fin]
            col_cod = next((j for j, h in enumerate(cab) if 'depart' in h and ('cod' in h or 'divipola' in h)), None)
            puntajes = {j: sum(1 for v in cuerpo.iloc[:, j].dropna() if isinstance(v, str) and cod_departamento(v))
                        for j in range(primera)}
            col_dep = col_cod if col_cod is not None else (max(puntajes, key=puntajes.get) if puntajes and
                                                           max(puntajes.values()) >= 5 else None)
            dep_titulo = (cod_departamento(hoja) or departamento_en_texto(titulo_bloque)
                          or departamento_en_texto(hoja) or departamento_en_texto(titulo_hoja))
            cols_txt = [j for j in range(primera) if j != col_dep and j != col_cod]
            tt = tb + ' ' + th
            act_titulo = 'agropecuario' if re.search(RX_AGRO, tt) else 'manufactura' if re.search(RX_MANUF, tt) else None
            dep_actual = dep_titulo
            n0 = len(regs)
            for r, fila in cuerpo.iterrows():
                v = fila.tolist()
                if col_dep is not None:
                    d = cod_departamento(v[col_dep], permitir_numero=(col_dep == col_cod))
                    if d:
                        dep_actual = d
                    elif not pd.isna(v[col_dep]) and str(v[col_dep]).strip():
                        dep_actual = None                  # total nacional u otra agregación
                else:
                    txt = [str(v[j]) for j in cols_txt if not pd.isna(v[j])]
                    numeros = [v[j] for j in cols if isinstance(v[j], (int, float)) and not pd.isna(v[j])]
                    if len(txt) == 1 and not numeros and not re.search(r'\d', txt[0]) and cod_departamento(txt[0]):
                        dep_actual = cod_departamento(txt[0])   # departamento como fila de sección
                        continue
                if dep_actual is None:
                    continue
                etiqueta = ' '.join(str(v[j]) for j in cols_txt if not pd.isna(v[j])).strip()
                ep = _plano(etiqueta)
                act = ('agropecuario' if re.search(RX_AGRO, ep) else 'manufactura' if re.search(RX_MANUF, ep) else
                       (act_titulo if (not etiqueta or cod_departamento(etiqueta)) else None))
                if act is None:
                    continue
                for j, (a, marca) in cols.items():
                    x = v[j]
                    if isinstance(x, (int, float, np.integer, np.floating)) and np.isfinite(x):
                        regs.append({'cod_dpto': dep_actual, 'anio': a, 'marca': marca, 'actividad': act,
                                     'precios': precios, 'valor': float(x), 'hoja': hoja, 'fila': int(r) + 1,
                                     'columna': int(j) + 1})
            if len(regs) > n0:
                diag.append({'hoja': hoja, 'bloque': b, 'precios': precios, 'valores': len(regs) - n0,
                             'departamentos': len({x['cod_dpto'] for x in regs[n0:]})})
    return pd.DataFrame(regs), diag

def _estructura(ruta):
    """Resumen compacto para diagnóstico: por hoja, título, cabeceras de años y primeras etiquetas."""
    print(f'\n===== DIAGNÓSTICO de {os.path.basename(ruta)} (copie este bloque si necesita ayuda) =====')
    for hoja, df in pd.read_excel(ruta, sheet_name=None, header=None).items():
        cab = _filas_anios(df)
        tit = ' | '.join(str(v) for v in df.iloc[:min(cab) if cab else 6].values.ravel() if not pd.isna(v))[:260]
        print(f'· Hoja «{hoja}» {df.shape}; filas de años: {list(cab)[:6]}; título: {tit}')
        if cab:
            h0 = min(cab); primera = min(cab[h0])
            etq = [' / '.join(str(x) for x in df.iloc[r, :primera] if not pd.isna(x))[:60]
                   for r in range(h0 + 1, min(h0 + 9, len(df)))]
            print('    cabecera:', [str(x)[:25] for x in df.iloc[h0, :primera + 2].tolist()])
            print('    filas:', etq)

def serie_dane(largo, actividad):
    s = largo[(largo.actividad == actividad) & (largo.precios == 'constantes')]
    if s.empty:
        return s
    cobertura = s.groupby('hoja').cod_dpto.nunique()
    if cobertura.max() >= 5:          # diseño con departamentos en filas: la hoja de mayor cobertura
        s = s[s.hoja == cobertura.idxmax()]
    # diseño con una hoja por departamento: se combinan todas las hojas
    return s.drop_duplicates(['cod_dpto', 'anio'])

def descargar_panel_dane():
    carpeta = os.path.join(CONFIG['OUT_DIR'], 'datos_dane')
    fuentes, series, info = {}, {}, {}
    for tipo in ('actividad', 'departamento'):
        try:
            url = _descubrir(tipo)
            ruta, sha = _descargar(url, carpeta)
            largo, diag = leer_tablas_anuales(ruta)
            fuentes[tipo] = {'url': url, 'archivo': ruta, 'sha256': sha, 'hojas_leidas': diag}
            print(f'✔ {os.path.basename(ruta)} · sha256 {sha[:12]}… · {len(largo)} valores en {len(diag)} bloques de {len({d["hoja"] for d in diag})} hojas')
            if not largo.empty:
                series[tipo] = {a: serie_dane(largo, a) for a in ('agropecuario', 'manufactura')}
            vacias = [a for a, x in series.get(tipo, {}).items() if x.empty]
            if largo.empty or vacias:
                print(f'  ⚠ {os.path.basename(ruta)}: no se extrajo {vacias or "ninguna serie"} a precios constantes; '
                      f'hojas leídas: {diag[:6]}')
                _estructura(ruta)
        except Exception as e:
            print(f'✘ anexo «{tipo}»: {e}')
    ok = {t: s for t, s in series.items() if all(len(x) for x in s.values())}
    if not ok:
        raise RuntimeError('No fue posible extraer las series del DANE. Copie el bloque «DIAGNÓSTICO» impreso arriba '
                           'para ajustar el lector. No se usan datos sintéticos de respaldo.')
    tipo = 'actividad' if 'actividad' in ok else 'departamento'
    sel = ok[tipo]
    # Validación cruzada entre los dos anexos oficiales cuando ambos se leyeron
    if len(ok) == 2:
        for a in ('agropecuario', 'manufactura'):
            m = ok['actividad'][a].merge(ok['departamento'][a], on=['cod_dpto', 'anio'], suffixes=('_a', '_d'))
            dif = (m.valor_a - m.valor_d).abs() / m.valor_a.abs().clip(lower=1e-9)
            info[f'coincidencia_{a}'] = f'{len(m)} pares, diferencia relativa máxima {dif.max():.2e}'
            print(f'  Validación cruzada {a}: {info[f"coincidencia_{a}"]}')
            if len(m) and dif.max() > 1e-3:
                print('  ⚠ Los dos anexos difieren: revise las hojas leídas antes de interpretar.')
    agro = sel['agropecuario'].rename(columns={'valor': 'agro'})
    ind = sel['manufactura'].rename(columns={'valor': 'ind'})
    df = agro[['cod_dpto', 'anio', 'agro', 'marca']].merge(ind[['cod_dpto', 'anio', 'ind']], on=['cod_dpto', 'anio'])
    df = df[df.anio.between(*CONFIG['ANIOS'])].sort_values(['cod_dpto', 'anio']).reset_index(drop=True)
    # Controles de calidad
    nd, na = df.cod_dpto.nunique(), df.anio.nunique()
    faltan = sorted(set(DPTOS.cod_dpto) - set(df.cod_dpto))
    assert na >= 8, f'Solo {na} años: insuficiente para el SpVAR'
    no_pos = df[(df.agro <= 0) | (df.ind <= 0)]
    if len(no_pos):
        print(f'  ⚠ {len(no_pos)} observaciones ≤ 0 (se tratarán en la Celda 7):',
              no_pos[['cod_dpto', 'anio']].drop_duplicates('cod_dpto').values.tolist()[:8])
    prelim = sorted(df.loc[df.marca != '', 'anio'].unique().tolist())
    PROCEDENCIA.update({'fuente': 'DANE · Cuentas Nacionales Departamentales, base 2015, precios constantes '
                                   '(volumen encadenado, miles de millones de pesos)',
                        'anexo_usado': fuentes[tipo], 'otros_anexos': {k: v for k, v in fuentes.items() if k != tipo},
                        'agro': 'Agricultura, ganadería, caza, silvicultura y pesca', 'ind': 'Industrias manufactureras',
                        'departamentos': nd, 'anios': (int(df.anio.min()), int(df.anio.max())),
                        'anios_preliminares': prelim, 'departamentos_faltantes': faltan, **info})
    df[['cod_dpto', 'anio', 'agro', 'ind']].to_csv(os.path.join(CONFIG['OUT_DIR'], 'panel_dane.csv'), index=False)
    print(f'Panel DANE: {nd} departamentos × {na} años ({df.anio.min()}–{df.anio.max()}); '
          f'preliminares/provisionales: {prelim or "ninguno"}; sin dato: {faltan or "ninguno"}.')
    return df[['cod_dpto', 'anio', 'agro', 'ind']]

PROCEDENCIA = {}

def cargar_datos_economicos():
    modo = CONFIG['MODO_DATOS']
    if modo == 'dane':
        return descargar_panel_dane(), None
    if modo == 'demo':
        df, sat_demo = simular_panel_demo(DPTOS, np.random.default_rng(CONFIG['SEMILLA']))
        PROCEDENCIA['fuente'] = 'SIMULACIÓN (DGP conocido) — no son datos de Colombia'
        print(f'Panel DEMO simulado: {df.cod_dpto.nunique()} dptos × {df.anio.nunique()} años.')
        return df, sat_demo
    df = pd.read_csv(CONFIG['RUTA_CSV'], dtype={CONFIG['COL_DPTO']: str})
    df = df.rename(columns={CONFIG['COL_DPTO']: 'cod_dpto', CONFIG['COL_ANIO']: 'anio',
                            CONFIG['COL_AGRO']: 'agro', CONFIG['COL_IND']: 'ind'})
    df['cod_dpto'] = df['cod_dpto'].str.extract(r'(\d+)')[0].str.zfill(2)
    df = df[df.anio.between(*CONFIG['ANIOS'])][['cod_dpto', 'anio', 'agro', 'ind']]
    PROCEDENCIA['fuente'] = f'CSV del usuario: {CONFIG["RUTA_CSV"]}'
    print(f'Panel CSV: {df.cod_dpto.nunique()} dptos × {df.anio.nunique()} años, {len(df)} filas.')
    return df, None

DF_ECON, _SAT_DEMO = cargar_datos_economicos()
DF_ECON.head()

"""## Celda 6 · Módulo climático: precipitación y temperatura (NASA POWER) por departamento
* **nasa_power** (por defecto): API pública de la NASA (MERRA-2), sin clave. Para cada departamento se consultan
  hasta 5 puntos interiores del polígono (malla de 0,75°) y se promedian; precipitación en mm/año y temperatura
  media anual en °C. Las respuestas se guardan en caché (`datos_clima/`).
* Alternativas: **gee** (NDVI MODIS, requiere cuenta de Earth Engine), **rasterio**, **csv**, **demo**.

Al modelo entran las **anomalías estandarizadas** por departamento (z-score), que aíslan choques climáticos
(p. ej. El Niño / La Niña) de las diferencias estructurales de clima. Si el clima del último año aún no está
publicado, el panel se recorta a los años con clima completo (no se imputa).
"""

def ndvi_gee(gdf, anios, proyecto):
    import ee
    try:
        ee.Initialize(project=proyecto)
    except Exception:
        ee.Authenticate()
        ee.Initialize(project=proyecto)
    g = gdf[['cod_dpto', 'geometry']].copy()
    g['geometry'] = g.geometry.simplify(0.01, preserve_topology=True)
    fc = ee.FeatureCollection(json.loads(g.to_json()))
    modis_ndvi = ee.ImageCollection('MODIS/061/MOD13Q1').select('NDVI')
    lc_ic = ee.ImageCollection('MODIS/061/MCD12Q1').select('LC_Type1')
    ult_lc = ee.Date(lc_ic.aggregate_max('system:time_start')).get('year').getInfo()
    filas = []
    for a in anios:
        if a < 2001:
            continue
        ndvi = modis_ndvi.filterDate(f'{a}-01-01', f'{a + 1}-01-01').mean().multiply(0.0001)
        alc = min(max(a, 2001), ult_lc)
        lc = lc_ic.filterDate(f'{alc}-01-01', f'{alc + 1}-01-01').first()
        cultivo = lc.eq(12).Or(lc.eq(14))
        img = (ndvi.rename('ndvi')
               .addBands(ndvi.updateMask(cultivo).rename('ndvi_crop'))
               .addBands(cultivo.rename('crop_frac')))
        red = img.reduceRegions(collection=fc, reducer=ee.Reducer.mean(), scale=1000, tileScale=4)
        info = red.select(['cod_dpto', 'ndvi', 'ndvi_crop', 'crop_frac'], None, False).getInfo()
        for f in info['features']:
            p = f['properties']
            filas.append({'cod_dpto': p['cod_dpto'], 'anio': a, 'ndvi': p.get('ndvi'),
                          'ndvi_crop': p.get('ndvi_crop'), 'crop_frac': p.get('crop_frac')})
        print(f'  GEE {a}: {len(info["features"])} departamentos')
    return pd.DataFrame(filas)

def ndvi_rasterio(gdf, anios, carpeta, patron, escala=1.0):
    import rasterio, rasterio.mask
    filas = []
    for a in anios:
        ruta = os.path.join(carpeta, patron.format(anio=a))
        if not os.path.exists(ruta):
            print('  falta', ruta); continue
        with rasterio.open(ruta) as src:
            g = gdf.to_crs(src.crs)
            for cod, geom in zip(g.cod_dpto, g.geometry):
                arr, _ = rasterio.mask.mask(src, [geom.__geo_interface__], crop=True, filled=False)
                v = arr[0].astype(float).compressed()
                v = v[np.isfinite(v)]
                filas.append({'cod_dpto': cod, 'anio': a, 'ndvi': v.mean() * escala if v.size else np.nan})
    return pd.DataFrame(filas)

def _puntos_departamento(geom, max_puntos=5, paso=0.75):
    """Punto representativo + puntos de una malla regular (paso ≈ resolución de MERRA-2) dentro del polígono."""
    from shapely.geometry import Point
    pts = [geom.representative_point()]
    x0, y0, x1, y1 = geom.bounds
    cand = [Point(x, y) for x in np.arange(x0 + paso / 2, x1, paso) for y in np.arange(y0 + paso / 2, y1, paso)
            if geom.contains(Point(x, y))]
    if cand:
        pts += [cand[i] for i in np.unique(np.linspace(0, len(cand) - 1, min(max_puntos - 1, len(cand))).astype(int))]
    return [(round(p.y, 4), round(p.x, 4)) for p in pts]

def _power_punto(lat, lon, a0, a1, carpeta):
    """Serie mensual NASA POWER (PRECTOTCORR mm/día, T2M °C) de un punto; con caché en disco."""
    ruta = os.path.join(carpeta, f'power_{lat}_{lon}_{a0}_{a1}.json')
    if os.path.exists(ruta):
        with open(ruta) as fh:
            return json.load(fh)
    ultimo = None
    for fin in (a1, a1 - 1, a1 - 2):                          # el último año puede no estar publicado aún
        url = ('https://power.larc.nasa.gov/api/temporal/monthly/point?parameters=PRECTOTCORR,T2M'
               f'&community=AG&longitude={lon}&latitude={lat}&start={a0}&end={fin}&format=JSON')
        for k in range(3):
            try:
                with urllib.request.urlopen(urllib.request.Request(url, headers=_UA), timeout=90) as r:
                    js = json.load(r)['properties']['parameter']
                with open(ruta, 'w') as fh:
                    json.dump(js, fh)
                return js
            except urllib.error.HTTPError as e:
                ultimo = e
                if e.code in (400, 422):
                    break                                     # rango de fechas no disponible: probar fin − 1
                time.sleep(2 ** k)
            except Exception as e:
                ultimo = e
                time.sleep(2 ** k)
    raise RuntimeError(f'NASA POWER ({lat}, {lon}): {ultimo}')

def clima_nasa_power(anios):
    """Precipitación anual (mm/año) y temperatura media (°C) por departamento: promedio de varios puntos
    interiores del polígono. Fuente: NASA POWER (MERRA-2), API pública sin clave."""
    from concurrent.futures import ThreadPoolExecutor
    carpeta = os.path.join(CONFIG['OUT_DIR'], 'datos_clima'); os.makedirs(carpeta, exist_ok=True)
    a0, a1 = int(min(anios)), int(max(anios))
    if GDF is not None:
        puntos = {c: _puntos_departamento(g) for c, g in zip(GDF.cod_dpto, GDF.geometry)}
    else:
        puntos = {c: [(round(la, 4), round(lo, 4))] for c, la, lo in zip(DPTOS.cod_dpto, DPTOS.lat, DPTOS.lon)}
    tareas = [(c, la, lo) for c, ps in puntos.items() for la, lo in ps]
    print(f'NASA POWER: {len(tareas)} puntos en {len(puntos)} departamentos ({a0}–{a1})…')
    with ThreadPoolExecutor(max_workers=6) as ex:
        res = list(ex.map(lambda t: (t[0], _power_punto(t[1], t[2], a0, a1, carpeta)), tareas))
    filas = []
    for c, js in res:
        for a in range(a0, a1 + 1):
            pr = [js['PRECTOTCORR'].get(f'{a}{m:02d}', -999) for m in range(1, 13)]
            tm = [js['T2M'].get(f'{a}{m:02d}', -999) for m in range(1, 13)]
            pr = np.where(np.array(pr, float) <= -990, np.nan, pr)
            tm = np.where(np.array(tm, float) <= -990, np.nan, tm)
            filas.append({'cod_dpto': c, 'anio': a,
                          'precip': np.mean(pr) * 365.25 if np.isfinite(pr).all() else np.nan,
                          'temp': np.mean(tm) if np.isfinite(tm).all() else np.nan})
    df = pd.DataFrame(filas).groupby(['cod_dpto', 'anio'], as_index=False).mean()
    PROCEDENCIA['clima'] = {'fuente': 'NASA POWER, API temporal/monthly/point (MERRA-2), comunidad AG',
                            'variables': 'PRECTOTCORR (mm/día → mm/año), T2M (°C, media anual)',
                            'agregacion': 'promedio de hasta 5 puntos interiores por departamento (malla de 0,75°)',
                            'puntos': {c: len(p) for c, p in puntos.items()},
                            'consultado': time.strftime('%Y-%m-%d')}
    return df

def obtener_satelite(anios):
    fuente = CONFIG['FUENTE_SAT']
    try:
        if fuente == 'ninguna':
            return None
        if fuente == 'nasa_power':
            sat = clima_nasa_power(anios)
        elif fuente == 'demo':
            sat = _SAT_DEMO if _SAT_DEMO is not None else simular_ndvi(
                DPTOS.cod_dpto.tolist(), DPTOS.lat.values,
                construir_W(DPTOS.cod_dpto.tolist(), DPTOS.lat, DPTOS.lon, ['knn4'])['knn4']['W'], anios, RNG)
        elif fuente == 'gee':
            if GDF is None:
                raise RuntimeError('GEE requiere la geometría (geopandas).')
            sat = ndvi_gee(GDF, anios, CONFIG['GEE_PROYECTO'])
        elif fuente == 'rasterio':
            sat = ndvi_rasterio(GDF, anios, CONFIG['RASTER_DIR'], CONFIG['RASTER_PATRON'], CONFIG['RASTER_ESCALA'])
        elif fuente == 'csv':
            sat = pd.read_csv(CONFIG['RUTA_CSV_SAT'], dtype={'cod_dpto': str})
            sat['cod_dpto'] = sat['cod_dpto'].str.zfill(2)
        else:
            raise ValueError(fuente)
    except Exception as e:
        print(f'Módulo climático/satelital ({fuente}) falló: {e}\n -> se continúa SIN covariables climáticas.')
        CONFIG['VARS_SAT'] = []
        return None
    sat = sat.copy()
    if 'ndvi_crop' in sat:
        sat['ndvi_crop'] = sat['ndvi_crop'].fillna(sat['ndvi'])
    for c in ['ndvi', 'ndvi_crop', 'precip', 'temp']:
        if c in sat:
            sat[c + '_anom'] = _zscore_por_dpto(sat, c)
    print(f'Clima/satélite ({fuente}): {sat.cod_dpto.nunique()} dptos × {sat.anio.nunique()} años; '
          f'columnas {[c for c in sat.columns if c not in ("cod_dpto", "anio")]}')
    return sat

ANIOS_NIV = sorted(DF_ECON.anio.unique())
DF_SAT = obtener_satelite(ANIOS_NIV)
faltan_sat = [v for v in CONFIG['VARS_SAT'] if DF_SAT is None or v not in DF_SAT.columns]
if faltan_sat:
    print('Covariables climáticas no disponibles y retiradas:', faltan_sat)
    CONFIG['VARS_SAT'] = [v for v in CONFIG['VARS_SAT'] if v not in faltan_sat]
if CONFIG['VARS_SAT'] and DF_SAT is not None:
    # Años con clima completo para todos los departamentos del panel: el panel se recorta (no se imputa)
    comp = DF_SAT.dropna(subset=CONFIG['VARS_SAT']).groupby('anio').cod_dpto.nunique()
    ok = comp[comp >= DF_ECON.cod_dpto.nunique()].index
    if len(ok) and DF_ECON.anio.max() > ok.max():
        print(f'El clima completo llega hasta {ok.max()}: el panel se recorta a ese año '
              f'(se excluyen {sorted(set(DF_ECON.anio[DF_ECON.anio > ok.max()]))}).')
        DF_ECON = DF_ECON[DF_ECON.anio <= ok.max()].copy()
        ANIOS_NIV = sorted(DF_ECON.anio.unique())

"""## Celda 7 · Construcción del panel SpVAR (crecimientos, rezagos, balanceo) y de las matrices W definitivas"""

def construir_panel(df, sat):
    df = df.copy()
    if sat is not None and CONFIG['VARS_SAT']:
        df = df.merge(sat[['cod_dpto', 'anio'] + CONFIG['VARS_SAT']], on=['cod_dpto', 'anio'], how='left')
    anios = sorted(df.anio.unique())
    # Balanceo: se excluyen departamentos con datos faltantes o no positivos
    piv = df.pivot_table(index='cod_dpto', columns='anio', values=['agro', 'ind'], aggfunc='first')
    malos = set(piv.index[piv.isna().any(axis=1)])
    trans = CONFIG['TRANSFORMACION']
    hay_ceros = (df[['agro', 'ind']] <= 0).any().any()
    if trans == 'auto':
        trans = 'dasinh' if hay_ceros else 'dlog'
    if trans == 'dlog':
        malos |= set(df.loc[(df.agro <= 0) | (df.ind <= 0), 'cod_dpto'])
    if CONFIG['VARS_SAT']:
        malos |= set(df.loc[df[CONFIG['VARS_SAT']].isna().any(axis=1) & (df.anio > anios[0]), 'cod_dpto'])
    malos |= set(df.cod_dpto) - set(DPTOS.cod_dpto)
    if malos:
        print('Departamentos excluidos por datos incompletos/no válidos:', sorted(malos))
    df = df[~df.cod_dpto.isin(malos)].sort_values(['cod_dpto', 'anio'])
    f = np.log if trans == 'dlog' else np.arcsinh
    for v in ['agro', 'ind']:
        df[v + '_niv'] = df[v]
        df[v] = df.groupby('cod_dpto')[v].transform(lambda s: f(s).diff())
        df[v + '_L1'] = df.groupby('cod_dpto')[v].shift(1)
    df = df.dropna(subset=['agro', 'ind', 'agro_L1', 'ind_L1'])
    codigos = [c for c in DPTOS.cod_dpto if c in set(df.cod_dpto)]
    df = df.set_index(['anio', 'cod_dpto']).sort_index()
    anios_m = sorted(df.index.get_level_values(0).unique())
    completo = pd.MultiIndex.from_product([anios_m, codigos], names=['anio', 'cod_dpto'])
    df = df.reindex(completo)
    assert not df[['agro', 'ind', 'agro_L1', 'ind_L1']].isna().any().any(), 'Panel no balanceado'
    print(f'Panel de estimación: N={len(codigos)} dptos, T={len(anios_m)} años '
          f'({anios_m[0]}–{anios_m[-1]}), NT={len(df)}; transformación={trans}')
    return df.reset_index(), codigos, anios_m, trans

PANEL, CODIGOS, ANIOS, TRANSF = construir_panel(DF_ECON, DF_SAT)
N, T = len(CODIGOS), len(ANIOS)
DPT = DPTOS.set_index('cod_dpto').loc[CODIGOS].reset_index()
WS = construir_W(CODIGOS, DPT.lat.values, DPT.lon.values, CONFIG['W_CANDIDATAS'])
display(resumen_W(WS))

def matrices_ecuacion(panel, k):
    """y, X y nombres de la ecuación k. Orden de filas: año-mayor (bloques de N), coherente con I_T ⊗ W."""
    j = 'ind' if k == 'agro' else 'agro'
    aguas_abajo = CONFIG['ORDEN'].index(k) == 1
    cols = {'const': np.ones(len(panel)), f'{k}_L1': panel[f'{k}_L1'].values, f'{j}_L1': panel[f'{j}_L1'].values}
    if aguas_abajo:
        cols[f'{j}_t'] = panel[j].values
    for s in CONFIG['VARS_SAT']:
        cols[s] = panel[s].values
    dinamicos = [c for c in cols if c != 'const']           # candidatos a término Durbin W·x
    if CONFIG['EFECTOS_TIEMPO']:
        for a in ANIOS[1:]:
            cols[f'T_{a}'] = (panel.anio.values == a).astype(float)
    if CONFIG['EFECTOS_DPTO']:
        for c in CODIGOS[1:]:
            cols[f'D_{c}'] = (panel.cod_dpto.values == c).astype(float)
    X = np.column_stack(list(cols.values()))
    nombres = list(cols)
    return panel[k].values.astype(float), X, nombres, [nombres.index(c) for c in dinamicos]

ECUACIONES = {k: matrices_ecuacion(PANEL, k) for k in ['agro', 'ind']}
for k, (y, X, nm, dz) in ECUACIONES.items():
    print(f'{NOMBRES_VAR[k]:>22}: y={y.shape}, X={X.shape}, regresores dinámicos={[nm[i] for i in dz]}')

"""## Celda 8 · Motor econométrico espacial (Máxima Verosimilitud exacta, panel apilado I_T ⊗ W)
Un solo motor estima las cinco especificaciones con la **misma** función de verosimilitud

$$\ell = -\tfrac{NT}{2}\ln(2\pi\sigma^2) - \tfrac{1}{2\sigma^2}\varepsilon'\varepsilon + T\ln|I-\rho W| + T\ln|I-\lambda W|,
  \quad \varepsilon = (I-\lambda W)\big[(I-\rho W)y - X\beta\big]$$

por lo que AIC/BIC y las pruebas LR son estrictamente comparables entre modelos y entre matrices W.
Los log-determinantes se calculan con los autovalores de W (exactos para N=33); la covarianza se obtiene
de la Hessiana numérica de la log-verosimilitud completa. Se incluyen los multiplicadores de Lagrange
(LM-lag, LM-error, robustos y SARMA, Anselin et al. 1996) y la I de Moran con permutaciones por período.
"""

@dataclass
class Resultado:
    modelo: str
    nombre_W: str
    nombres: list
    params: np.ndarray            # [beta..., (rho), (lambda), sigma2]
    nombres_params: list
    vcov: np.ndarray
    ll: float
    n: int
    k_par: int
    rho: float = 0.0
    lam: float = 0.0
    sigma2: float = np.nan
    resid: np.ndarray = None      # innovaciones estructurales ε
    yhat: np.ndarray = None       # predicción de forma reducida
    convergio: bool = True
    vcov_ok: bool = True
    durbin: list = field(default_factory=list)

    @property
    def beta(self):
        return self.params[:len(self.nombres)]
    @property
    def se(self):
        return np.sqrt(np.clip(np.diag(self.vcov), 0, None))
    @property
    def aic(self):
        return -2 * self.ll + 2 * self.k_par
    @property
    def bic(self):
        return -2 * self.ll + self.k_par * np.log(self.n)
    def coef(self, nombre, defecto=0.0):
        return float(self.params[self.nombres_params.index(nombre)]) if nombre in self.nombres_params else defecto
    def tabla(self):
        z = self.params / np.where(self.se > 0, self.se, np.nan)
        p = 2 * stats.norm.sf(np.abs(z))
        return pd.DataFrame({'coef': self.params, 'se': self.se, 'z': z, 'p': p,
                             'sig': [estrellas(x) for x in p]}, index=self.nombres_params)

def estrellas(p):
    return '' if not np.isfinite(p) else '***' if p < .01 else '**' if p < .05 else '*' if p < .10 else ''

def hessiana_numerica(f, x, paso_rel=1e-4):
    x = np.asarray(x, float)
    n = len(x)
    h = paso_rel * np.abs(x) + 1e-6
    H = np.zeros((n, n))
    f0 = f(x)
    for i in range(n):
        ei = np.zeros(n); ei[i] = h[i]
        H[i, i] = (f(x + ei) - 2 * f0 + f(x - ei)) / h[i] ** 2
        for j in range(i + 1, n):
            ej = np.zeros(n); ej[j] = h[j]
            H[i, j] = H[j, i] = (f(x + ei + ej) - f(x + ei - ej) - f(x - ei + ej) + f(x - ei - ej)) / (4 * h[i] * h[j])
    return H

class MotorEspacial:
    def __init__(self, W, N, T, nombre_W=''):
        self.W, self.N, self.T, self.nombre_W = np.asarray(W, float), N, T, nombre_W
        self.eig = np.linalg.eigvals(self.W).astype(complex)
        reales = self.eig.real[np.abs(self.eig.imag) < 1e-8]
        lo = 1.0 / reales.min() if reales.min() < 0 else -1.0
        self.lo, self.hi = max(lo, -0.999) + 1e-3, min(1.0 / reales.max(), 0.999) - 1e-3
        self.trWW = np.trace(self.W.T @ self.W + self.W @ self.W)

    # --- utilidades -------------------------------------------------------------------------
    def lag(self, v):
        v = np.asarray(v, float)
        if v.ndim == 1:
            return (v.reshape(self.T, self.N) @ self.W.T).ravel()
        k = v.shape[1]
        return np.einsum('ij,tjk->tik', self.W, v.reshape(self.T, self.N, k)).reshape(-1, k)

    def logdet(self, a):
        return 0.0 if a == 0 else float(self.T * np.sum(np.log(1.0 - a * self.eig)).real)

    def resolver(self, a, v):
        """(I_T ⊗ (I − aW))^{-1} v"""
        if a == 0:
            return v
        M = np.eye(self.N) - a * self.W
        return np.linalg.solve(M, np.asarray(v).reshape(self.T, self.N).T).T.ravel()

    @staticmethod
    def _ols(y, X):
        b, *_ = np.linalg.lstsq(X, y, rcond=None)
        return b, y - X @ b

    def _ll_conc(self, e, rho, lam):
        n = len(e); s2 = e @ e / n
        return -n / 2 * (np.log(2 * np.pi * s2) + 1) + self.logdet(rho) + self.logdet(lam)

    def ll_completa(self, theta, y, X, con_rho, con_lam):
        k = X.shape[1]; i = k
        b = theta[:k]
        rho = theta[i] if con_rho else 0.0; i += con_rho
        lam = theta[i] if con_lam else 0.0; i += con_lam
        s2 = theta[i]
        if s2 <= 0 or not (self.lo - 1e-3 < rho < self.hi + 1e-3) or not (self.lo - 1e-3 < lam < self.hi + 1e-3):
            return -1e12
        r = y - rho * self.lag(y) - X @ b
        e = r - lam * self.lag(r)
        n = len(y)
        return -n / 2 * np.log(2 * np.pi * s2) - e @ e / (2 * s2) + self.logdet(rho) + self.logdet(lam)

    # --- estimación -------------------------------------------------------------------------
    def ajustar(self, y, X, nombres, modelo, durbin_idx=()):
        nombres = list(nombres)
        durbin = []
        if modelo == 'SDM':
            Xd = self.lag(X[:, list(durbin_idx)])
            durbin = ['W_' + nombres[i] for i in durbin_idx]
            X = np.column_stack([X, Xd]); nombres = nombres + durbin
        n = len(y); Wy = self.lag(y); WX = self.lag(X)
        con_rho, con_lam = modelo in ('SAR', 'SAC', 'SDM'), modelo in ('SEM', 'SAC')
        rho = lam = 0.0; ok = True
        if modelo == 'OLS':
            b, e = self._ols(y, X)
        elif modelo in ('SAR', 'SDM'):
            _, e0 = self._ols(y, X); _, eL = self._ols(Wy, X)
            res = optimize.minimize_scalar(lambda r: -self._ll_conc(e0 - r * eL, r, 0.0),
                                           bounds=(self.lo, self.hi), method='bounded', options={'xatol': 1e-9})
            rho, ok = float(res.x), bool(res.success)
            b, e = self._ols(y - rho * Wy, X)
        elif modelo == 'SEM':
            def f(l):
                _, e = self._ols(y - l * Wy, X - l * WX); return -self._ll_conc(e, 0.0, l)
            res = optimize.minimize_scalar(f, bounds=(self.lo, self.hi), method='bounded', options={'xatol': 1e-9})
            lam, ok = float(res.x), bool(res.success)
            b, e = self._ols(y - lam * Wy, X - lam * WX)
        elif modelo == 'SAC':
            WWy = self.lag(Wy)
            def f(p):
                r, l = p
                ys = y - r * Wy - l * (Wy - r * WWy)
                _, e = self._ols(ys, X - l * WX); return -self._ll_conc(e, r, l)
            rejilla = np.linspace(max(self.lo, -0.9), min(self.hi, 0.9), 9)
            inicio = min(itertools.product(rejilla, rejilla), key=f)
            res = optimize.minimize(f, inicio, method='L-BFGS-B', bounds=[(self.lo, self.hi)] * 2)
            (rho, lam), ok = map(float, res.x), bool(res.success)
            b, e = self._ols(y - rho * Wy - lam * (Wy - rho * WWy), X - lam * WX)
        else:
            raise ValueError(modelo)
        s2 = e @ e / n
        theta = np.r_[b, [rho] * con_rho, [lam] * con_lam, s2]
        nombres_p = nombres + ['rho'] * con_rho + ['lambda'] * con_lam + ['sigma2']
        ll = self.ll_completa(theta, y, X, con_rho, con_lam)
        H = hessiana_numerica(lambda t: self.ll_completa(t, y, X, con_rho, con_lam), theta)
        V, vok = invertir_informacion(-H)
        yhat = self.resolver(rho, X @ b)
        r = Resultado(modelo, self.nombre_W, nombres, theta, nombres_p, V, ll, n, len(theta),
                      rho, lam, s2, e, yhat, ok, vok, durbin)
        return r

    # --- diagnósticos -----------------------------------------------------------------------
    def pruebas_LM(self, y, X):
        """Multiplicadores de Lagrange sobre residuos MCO (versión panel con I_T ⊗ W)."""
        n = len(y)
        b, e = self._ols(y, X)
        s2 = e @ e / n
        We, Wy = self.lag(e), self.lag(y)
        Tw = self.T * self.trWW
        WXb = self.lag(X @ b)
        _, MWXb = self._ols(WXb, X)
        D = (WXb @ MWXb + Tw * s2) / s2
        d_err, d_lag = e @ We / s2, e @ Wy / s2
        out = {
            'LM_error': d_err ** 2 / Tw,
            'LM_lag': d_lag ** 2 / D,
            'RLM_error': (d_err - Tw / D * d_lag) ** 2 / (Tw * (1 - Tw / D)),
            'RLM_lag': (d_lag - d_err) ** 2 / (D - Tw),
        }
        out['SARMA'] = out['RLM_lag'] + out['LM_error']
        filas = {k: {'estadistico': v, 'gl': 2 if k == 'SARMA' else 1,
                     'p': stats.chi2.sf(v, 2 if k == 'SARMA' else 1)} for k, v in out.items()}
        return filas

    def moran_residuos(self, e, n_perm=499, rng=None):
        rng = rng or np.random.default_rng(0)
        E = np.asarray(e).reshape(self.T, self.N)
        E = E - E.mean(1, keepdims=True)
        def I(M):
            return np.sum(M * (M @ self.W.T)) / np.sum(M * M)
        obs = I(E)
        sims = np.array([I(np.take_along_axis(E, rng.permuted(np.tile(np.arange(self.N), (self.T, 1)), axis=1), 1))
                         for _ in range(n_perm)])
        p = (1 + np.sum(np.abs(sims - sims.mean()) >= abs(obs - sims.mean()))) / (n_perm + 1)
        return {'I': obs, 'E[I]': sims.mean(), 'z': (obs - sims.mean()) / sims.std(), 'p_perm': p}

def invertir_informacion(Inf):
    Inf = (Inf + Inf.T) / 2
    try:
        L = np.linalg.cholesky(Inf)
        Li = linalg.solve_triangular(L, np.eye(len(Inf)), lower=True)
        return Li.T @ Li, True
    except np.linalg.LinAlgError:
        return np.linalg.pinv(Inf), False

"""## Celda 9 · Pipeline automático de selección: (matriz W × especificación) para cada ecuación
Para cada ecuación y cada W candidata: (i) MCO + pruebas LM + Moran; (ii) SAR, SEM, SAC, SDM por MV;
(iii) pruebas LR anidadas (SDM→SAR: θ=0; SDM→SEM: factor común θ=−ρβ; SAC→SAR/SEM; SAR/SEM→MCO);
(iv) árbol de decisión de Elhorst (2010, 2014); (v) pesos de Akaike/BIC sobre todo el espacio (W, modelo).
La selección final minimiza el criterio de `CONFIG['CRITERIO']` conjuntamente sobre W y modelo
(con 'ELHORST' se aplica el árbol en cada W y se elige la W con mejor criterio de información).
"""

def arbol_elhorst(lm, ajustes, alfa):
    sig = {k: v['p'] < alfa for k, v in lm.items()}
    if not (sig['LM_lag'] or sig['LM_error']):
        return 'OLS', 'Ningún LM significativo: no hay evidencia de dependencia espacial.'
    sdm, sar, sem = ajustes.get('SDM'), ajustes.get('SAR'), ajustes.get('SEM')
    if sdm is None:
        return ('SAR' if lm['RLM_lag']['estadistico'] >= lm['RLM_error']['estadistico'] else 'SEM'), 'Sin SDM.'
    gl = len(sdm.durbin)
    p_sar = stats.chi2.sf(2 * (sdm.ll - sar.ll), gl)
    p_sem = stats.chi2.sf(2 * (sdm.ll - sem.ll), gl)
    rej_sar, rej_sem = p_sar < alfa, p_sem < alfa
    if sig['RLM_lag'] and sig['RLM_error'] and 'SAC' in ajustes:
        m = min(['SAC', 'SDM'], key=lambda k: ajustes[k].aic)
        return m, 'Ambos LM robustos significativos: SAC vs SDM por AIC.'
    if rej_sar and rej_sem:
        return 'SDM', f'LR rechaza θ=0 (p={p_sar:.3f}) y factor común (p={p_sem:.3f}).'
    if not rej_sar and rej_sem:
        return 'SAR', f'LR no rechaza θ=0 (p={p_sar:.3f}); rechaza factor común (p={p_sem:.3f}).'
    if rej_sar and not rej_sem:
        return 'SEM', f'LR no rechaza factor común (p={p_sem:.3f}); rechaza θ=0 (p={p_sar:.3f}).'
    m = 'SAR' if lm['RLM_lag']['estadistico'] >= lm['RLM_error']['estadistico'] else 'SEM'
    return m, f'Ninguna restricción rechazada; desempate por LM robusto → {m}.'

def pruebas_LR(aj):
    pares = [('SDM', 'SAR', 'θ = 0 (Durbin)'), ('SDM', 'SEM', 'θ = −ρβ (factor común)'),
             ('SAC', 'SAR', 'λ = 0'), ('SAC', 'SEM', 'ρ = 0'), ('SAR', 'OLS', 'ρ = 0'), ('SEM', 'OLS', 'λ = 0')]
    filas = []
    for g, r, h0 in pares:
        if g in aj and r in aj:
            lr = max(2 * (aj[g].ll - aj[r].ll), 0.0)
            gl = aj[g].k_par - aj[r].k_par
            filas.append({'general': g, 'restringido': r, 'H0': h0, 'LR': lr, 'gl': gl, 'p': stats.chi2.sf(lr, gl)})
    return filas

def seleccionar(ecuaciones, Ws, modelos, criterio='BIC', alfa=0.05):
    tab_aj, tab_lm, tab_lr, tab_mor, ajustes, arboles = [], [], [], [], {}, {}
    for k, (y, X, nombres, dz) in ecuaciones.items():
        for wn, wd in Ws.items():
            mot = MotorEspacial(wd['W'], N, T, wn)
            lm = mot.pruebas_LM(y, X)
            for prueba, v in lm.items():
                tab_lm.append({'ecuacion': k, 'W': wn, 'prueba': prueba, **v})
            aj = {}
            for m in modelos:
                t0 = time.time()
                try:
                    aj[m] = mot.ajustar(y, X, nombres, m, dz)
                except Exception as e:
                    print(f'   ! {k}/{wn}/{m}: {e}'); continue
                r = aj[m]
                tab_aj.append({'ecuacion': k, 'W': wn, 'modelo': m, 'logLik': r.ll, 'k': r.k_par,
                               'AIC': r.aic, 'BIC': r.bic, 'rho': r.rho if m in ('SAR', 'SAC', 'SDM') else np.nan,
                               'lambda': r.lam if m in ('SEM', 'SAC') else np.nan, 'sigma2': r.sigma2,
                               'convergió': r.convergio, 'vcov_PD': r.vcov_ok, 'seg': time.time() - t0})
            for f in pruebas_LR(aj):
                tab_lr.append({'ecuacion': k, 'W': wn, **f})
            mor = mot.moran_residuos(aj['OLS'].resid, CONFIG['N_PERM'], RNG)
            tab_mor.append({'ecuacion': k, 'W': wn, **mor})
            arboles[(k, wn)] = arbol_elhorst(lm, aj, alfa)
            ajustes[(k, wn)] = aj
            print(f'{k:>5} | {wn:<6} | Elhorst → {arboles[(k, wn)][0]:<4} | '
                  + ' '.join(f'{m}:{aj[m].bic:8.1f}' for m in aj))
    TA = pd.DataFrame(tab_aj)
    TA['elhorst'] = [arboles[(e, w)][0] == m for e, w, m in zip(TA.ecuacion, TA.W, TA.modelo)]
    for k in ecuaciones:
        s = TA.ecuacion == k
        for c in ['AIC', 'BIC']:
            d = TA.loc[s, c] - TA.loc[s, c].min()
            TA.loc[s, f'peso_{c}'] = np.exp(-d / 2) / np.exp(-d / 2).sum()
    eleccion = {}
    for k in ecuaciones:
        s = TA[(TA.ecuacion == k) & TA.convergió]
        c = 'AIC' if criterio == 'AIC' else 'BIC'
        if criterio == 'ELHORST':
            s = s[s.elhorst]
        fila = s.loc[s[c].idxmin()]
        ic = TA[(TA.ecuacion == k) & TA.convergió]
        fila_ic = ic.loc[ic[c].idxmin()]
        eleccion[k] = {'W': fila.W, 'modelo': fila.modelo, 'criterio': c,
                       'peso': fila[f'peso_{c}'], 'arbol': arboles[(k, fila.W)],
                       'coincide_IC_arbol': arboles[(k, fila.W)][0] == fila.modelo,
                       'mejor_IC_global': (fila_ic.W, fila_ic.modelo)}
    return (TA, pd.DataFrame(tab_lm), pd.DataFrame(tab_lr), pd.DataFrame(tab_mor), ajustes, arboles, eleccion)

t0 = time.time()
TAB_AJ, TAB_LM, TAB_LR, TAB_MORAN, AJUSTES, ARBOLES, ELECCION = seleccionar(
    ECUACIONES, WS, CONFIG['MODELOS'], CONFIG['CRITERIO'], CONFIG['ALFA'])
print(f'\nEstimaciones: {len(TAB_AJ)} en {time.time() - t0:.1f} s')
for k, e in ELECCION.items():
    print(f"► {NOMBRES_VAR[k]}: {e['modelo']} con W={e['W']} ({e['criterio']}, peso={e['peso']:.2f}); "
          f"árbol Elhorst en esa W → {e['arbol'][0]} ({e['arbol'][1]})")
FINAL = {k: AJUSTES[(k, e['W'])][e['modelo']] for k, e in ELECCION.items()}
display(TAB_AJ.pivot_table(index=['ecuacion', 'W'], columns='modelo', values='BIC').round(1))

"""## Celda 10 · Ensamble del SpVAR y descomposición de los 4 mecanismos de transmisión
| Mecanismo | Parámetros | Impactos (LeSage–Pace) |
|---|---|---|
| **a) Autodinámica espacial** | φ_kk (y_k,t−1), ψ_kk (W y_k,t−1) | directo e indirecto de y_k,t−1 |
| **b) Cruce local** | β_kj (y_j,t, contemporáneo), φ_kj (y_j,t−1) | impactos **directos** de y_j |
| **c) Relación espacial directa** | ρ_k (W y_k,t), λ_k (W u_k,t) | multiplicador espacial (I−ρW)⁻¹ |
| **d) Relación espacial cruzada** | θ_kj (W y_j,t), ψ_kj (W y_j,t−1) | impactos **indirectos** de y_j |

Los impactos promedio se calculan como Directo = tr(S)/N, Total = 1'S1/N, Indirecto = Total − Directo con
S = (I−ρW)⁻¹(βI + θW); su inferencia se obtiene por simulación de la distribución asintótica de los parámetros.
"""

def sorteos_parametros(res, n, rng):
    V = (res.vcov + res.vcov.T) / 2
    w, Q = np.linalg.eigh(V)
    raiz = Q * np.sqrt(np.clip(w, 0, None))
    return res.params + rng.standard_normal((n, len(res.params))) @ raiz.T

def impactos(res, W, regresores, n_sim, rng):
    eig = np.linalg.eigvals(W).astype(complex)
    Nn = len(W)
    def calc(theta):
        p = dict(zip(res.nombres_params, theta.T))
        rho = np.clip(p.get('rho', np.zeros(theta.shape[0])), -0.99, 0.99)
        trM = np.sum(1 / (1 - np.outer(rho, eig)), 1).real / Nn
        trMW = np.sum(eig / (1 - np.outer(rho, eig)), 1).real / Nn
        out = {}
        for x in regresores:
            b = p[x]; th = p.get('W_' + x, 0 * b)
            d = b * trM + th * trMW
            tot = (b + th) / (1 - rho)
            out[x] = (d, tot - d, tot)
        return out
    punto = calc(res.params[None, :])
    sims = calc(sorteos_parametros(res, n_sim, rng))
    filas = []
    for x in regresores:
        for i, tipo in enumerate(['Directo', 'Indirecto', 'Total']):
            est, sd = float(punto[x][i][0]), float(np.std(sims[x][i]))
            z = est / sd if sd > 0 else np.nan
            p = 2 * stats.norm.sf(abs(z)) if sd > 0 else np.nan
            lo, hi = np.percentile(sims[x][i], [5, 95])
            filas.append({'regresor': x, 'efecto': tipo, 'estimacion': est, 'se_sim': sd, 'z': z, 'p': p,
                          'IC90_inf': lo, 'IC90_sup': hi, 'sig': estrellas(p)})
    return pd.DataFrame(filas)

def matriz_S(res, W, x):
    """Matriz N×N de efectos ∂y/∂x' para el regresor x (impactos por departamento)."""
    M = np.linalg.inv(np.eye(len(W)) - res.rho * W)
    return M @ (res.coef(x) * np.eye(len(W)) + res.coef('W_' + x) * W)

def tabla_mecanismos(res, k, imp):
    j = 'ind' if k == 'agro' else 'agro'
    t = res.tabla()
    def fila(mec, desc, nombre):
        if nombre not in t.index:
            return None
        r = t.loc[nombre]
        return {'ecuacion': k, 'mecanismo': mec, 'termino': desc, 'parametro': nombre,
                'estimacion': r.coef, 'se': r.se, 'p': r.p, 'sig': r.sig, 'tipo': 'coeficiente'}
    def fila_imp(mec, desc, x, efecto):
        s = imp[(imp.regresor == x) & (imp.efecto == efecto)]
        if s.empty:
            return None
        r = s.iloc[0]
        return {'ecuacion': k, 'mecanismo': mec, 'termino': desc, 'parametro': f'{efecto}[{x}]',
                'estimacion': r.estimacion, 'se': r.se_sim, 'p': r.p, 'sig': r.sig, 'tipo': 'impacto'}
    A, B, C, D, S = ('a) Autodinámica espacial', 'b) Cruce local', 'c) Relación espacial directa',
                     'd) Relación espacial cruzada', 'Canal climático')
    filas = [
        fila(A, f'φ persistencia temporal propia ({k}_t-1)', f'{k}_L1'),
        fila(A, f'ψ rezago espacio-temporal propio (W·{k}_t-1)', f'W_{k}_L1'),
        fila_imp(A, f'Efecto local de {k}_t-1', f'{k}_L1', 'Directo'),
        fila_imp(A, f'Difusión espacial de {k}_t-1 hacia vecinos', f'{k}_L1', 'Indirecto'),
        fila(B, f'β efecto contemporáneo local de {j}_t', f'{j}_t'),
        fila(B, f'φ efecto local rezagado de {j}_t-1', f'{j}_L1'),
        fila_imp(B, f'Impacto directo total de {j}_t', f'{j}_t', 'Directo'),
        fila_imp(B, f'Impacto directo total de {j}_t-1', f'{j}_L1', 'Directo'),
        fila(C, f'ρ dependencia espacial contemporánea de {k} (W·{k}_t)', 'rho'),
        fila(C, f'λ dependencia espacial en choques de {k}', 'lambda'),
        fila(D, f'θ spillover contemporáneo de {j} vecino (W·{j}_t)', f'W_{j}_t'),
        fila(D, f'ψ spillover rezagado de {j} vecino (W·{j}_t-1)', f'W_{j}_L1'),
        fila_imp(D, f'Impacto indirecto total de {j}_t', f'{j}_t', 'Indirecto'),
        fila_imp(D, f'Impacto indirecto total de {j}_t-1', f'{j}_L1', 'Indirecto'),
    ]
    for s in CONFIG['VARS_SAT']:
        filas += [fila(S, f'γ efecto local de {s}', s), fila(S, f'δ efecto de {s} vecino (W·{s})', f'W_{s}'),
                  fila_imp(S, f'Impacto directo de {s}', s, 'Directo'),
                  fila_imp(S, f'Impacto indirecto de {s}', s, 'Indirecto')]
    return pd.DataFrame([f for f in filas if f is not None])

W_FINAL = {k: WS[ELECCION[k]['W']]['W'] for k in FINAL}
REGRESORES = {k: [ECUACIONES[k][2][i] for i in ECUACIONES[k][3]] for k in FINAL}
IMPACTOS = {k: impactos(FINAL[k], W_FINAL[k], REGRESORES[k], CONFIG['N_SIM_IMPACTOS'], RNG) for k in FINAL}
MECANISMOS = pd.concat([tabla_mecanismos(FINAL[k], k, IMPACTOS[k]) for k in FINAL], ignore_index=True)
for k in FINAL:
    print(f'\n=== {NOMBRES_VAR[k]} · {FINAL[k].modelo} · W={FINAL[k].nombre_W} ===')
    t = FINAL[k].tabla()
    display(t[~t.index.str.match(r'^(T_|D_)')].round(4))
display(MECANISMOS.round(4))

"""## Celda 11 · Dinámica del sistema: forma reducida, estabilidad y respuestas impulso espacio-temporales
Apilando Y_t = [y_up,t ; y_down,t] (2N×1):  A₀ Y_t = A₁ Y_{t−1} + B ε_t  ⇒  Y_t = Φ Y_{t−1} + Γ ε_t,
con Φ = A₀⁻¹A₁ y Γ = A₀⁻¹B. El SpVAR es estable si el radio espectral de Φ es < 1.
Se calculan IRF agregadas (choque de 1 d.e. en todos los departamentos), un choque de **sequía satelital**
(NDVI −1 d.e.) y un choque **localizado** en `DPTO_CHOQUE` para visualizar la difusión espacial.
"""

def _dict_param(res, theta=None):
    theta = res.params if theta is None else theta
    p = dict(zip(res.nombres_params, theta))
    return {'rho': p.get('rho', 0.0), 'lam': p.get('lambda', 0.0), 'sigma': np.sqrt(max(p['sigma2'], 1e-12)), 'c': p}

def sistema(par, Wd):
    up, dn = CONFIG['ORDEN']
    I = np.eye(N); c = lambda k, n: par[k]['c'].get(n, 0.0)
    A = {k: I - par[k]['rho'] * Wd[k] for k in (up, dn)}
    Cc = c(dn, f'{up}_t') * I + c(dn, f'W_{up}_t') * Wd[dn]
    A0 = np.block([[A[up], np.zeros((N, N))], [-Cc, A[dn]]])
    A1 = np.block([[c(k, f'{m}_L1') * I + c(k, f'W_{m}_L1') * Wd[k] for m in (up, dn)] for k in (up, dn)])
    Bm = linalg.block_diag(*[np.linalg.inv(I - par[k]['lam'] * Wd[k]) for k in (up, dn)])
    Phi = np.linalg.solve(A0, A1)
    Gam = np.linalg.solve(A0, Bm)
    Dsat = {s: np.vstack([c(k, s) * I + c(k, f'W_{s}') * Wd[k] for k in (up, dn)]) for s in CONFIG['VARS_SAT']}
    Gsat = {s: np.linalg.solve(A0, D) for s, D in Dsat.items()}
    return Phi, Gam, Gsat

def irf(Phi, R0, H):
    out = [R0]
    for _ in range(H):
        out.append(Phi @ out[-1])
    return np.array(out)                                      # (H+1, 2N, ...)

def irf_agregadas(par, Wd, H):
    up, dn = CONFIG['ORDEN']
    Phi, Gam, Gsat = sistema(par, Wd)
    choques = {f'ε_{up}': Gam[:, :N] @ np.full(N, par[up]['sigma']),
               f'ε_{dn}': Gam[:, N:] @ np.full(N, par[dn]['sigma'])}
    for s, G in Gsat.items():
        choques[f'clima({s} = −1 d.e.)'] = G @ np.full(N, -1.0)
    out = {}
    for nom, r0 in choques.items():
        R = irf(Phi, r0, H)
        out[nom] = {up: R[:, :N].mean(1), dn: R[:, N:].mean(1)}
    return out, Phi

ORDEN = CONFIG['ORDEN']
PAR = {k: _dict_param(FINAL[k]) for k in FINAL}
PHI, GAM, GSAT = sistema(PAR, W_FINAL)
RADIO_ESPECTRAL = float(np.max(np.abs(np.linalg.eigvals(PHI))))
print(f'Radio espectral de Φ = {RADIO_ESPECTRAL:.4f} -> {"ESTABLE" if RADIO_ESPECTRAL < 1 else "NO ESTACIONARIO"}')

H = CONFIG['HORIZONTE_IRF']
IRF_PUNTO, _ = irf_agregadas(PAR, W_FINAL, H)
sorteos = {k: sorteos_parametros(FINAL[k], CONFIG['N_SIM_IRF'], RNG) for k in FINAL}
IRF_SIM, explosivos = {n: {v: [] for v in ORDEN} for n in IRF_PUNTO}, 0
for s in range(CONFIG['N_SIM_IRF']):
    par_s = {k: _dict_param(FINAL[k], sorteos[k][s]) for k in FINAL}
    for k in par_s:
        par_s[k]['rho'] = float(np.clip(par_s[k]['rho'], -0.99, 0.99))
        par_s[k]['lam'] = float(np.clip(par_s[k]['lam'], -0.99, 0.99))
    out, Phi_s = irf_agregadas(par_s, W_FINAL, H)
    if np.max(np.abs(np.linalg.eigvals(Phi_s))) >= 1:
        explosivos += 1; continue
    for n in out:
        for v in ORDEN:
            IRF_SIM[n][v].append(out[n][v])
filas = []
for n in IRF_PUNTO:
    for v in ORDEN:
        sim = np.array(IRF_SIM[n][v])
        lo, hi = np.percentile(sim, [5, 95], axis=0)
        for h in range(H + 1):
            filas.append({'choque': n, 'respuesta': v, 'h': h, 'irf': IRF_PUNTO[n][v][h], 'p05': lo[h], 'p95': hi[h],
                          'acumulada': IRF_PUNTO[n][v][:h + 1].sum()})
TAB_IRF = pd.DataFrame(filas)
print(f'Sorteos explosivos descartados en bandas IRF: {explosivos}/{CONFIG["N_SIM_IRF"]}')

# Choque localizado (1 d.e. del choque estructural de la variable aguas arriba en DPTO_CHOQUE)
i0 = CODIGOS.index(CONFIG['DPTO_CHOQUE']) if CONFIG['DPTO_CHOQUE'] in CODIGOS else 0
R_LOC = irf(PHI, GAM[:, i0] * PAR[ORDEN[0]]['sigma'], H)          # (H+1, 2N)
print(f'Choque localizado en {DPT.nombre[i0]} ({CODIGOS[i0]})')
display(TAB_IRF[TAB_IRF.h <= 3].pivot_table(index=['choque', 'respuesta'], columns='h', values='irf').round(5))

"""## Celda 12 · Resúmenes interpretativos automatizados"""

def _sig_txt(p, alfa=CONFIG['ALFA']):
    if not np.isfinite(p):
        return 'sin inferencia disponible'
    return f'significativo (p={p:.3f})' if p < alfa else f'no significativo (p={p:.3f})'

def interpretar():
    up, dn = ORDEN
    lin = [f'# Informe automático SpVAR — Agro ↔ {NOMBRES_VAR["ind"]} (Colombia)', '',
           f'*Fuente:* {PROCEDENCIA.get("fuente", "")}.', '',
           f'*Datos:* {CONFIG["MODO_DATOS"]} · N={N} departamentos · T={T} años ({ANIOS[0]}–{ANIOS[-1]}) · '
           f'variables en tasas de crecimiento ({TRANSF}) · orden recursivo: {up} → {dn} · '
           f'covariables climáticas: {CONFIG["VARS_SAT"] or "ninguna"}.', '']
    for k in FINAL:
        e, r = ELECCION[k], FINAL[k]
        lin += [f'## Ecuación {NOMBRES_VAR[k]}', '',
                f'Especificación seleccionada: **{r.modelo}** con **W = {r.nombre_W}** '
                f'({WS[r.nombre_W]["desc"]}); criterio {e["criterio"]}, peso de evidencia {e["peso"]:.2f}. '
                f'Árbol de Elhorst en esa W → {e["arbol"][0]}: {e["arbol"][1]}'
                + ('' if e['coincide_IC_arbol'] else ' **Nota:** el árbol y el criterio de información discrepan; '
                   'revise la tabla de pruebas LR.'), '']
        m = MECANISMOS[MECANISMOS.ecuacion == k]
        for mec in m.mecanismo.unique():
            lin.append(f'**{mec}.**')
            for _, f in m[m.mecanismo == mec].iterrows():
                signo = 'positivo' if f.estimacion > 0 else 'negativo'
                lin.append(f'- {f.termino}: {f.estimacion:+.4f} ({signo}, {_sig_txt(f.p)}).')
            lin.append('')
        if r.rho:
            lin.append(f'Multiplicador espacial medio 1/(1−ρ) = {1 / (1 - r.rho):.2f}: un choque común se amplifica '
                       f'{(1 / (1 - r.rho) - 1) * 100:.0f}% por retroalimentación entre departamentos vecinos.')
            lin.append('')
    lin += ['## Dinámica del sistema', '',
            f'Radio espectral de Φ = {RADIO_ESPECTRAL:.3f} ({"sistema estable" if RADIO_ESPECTRAL < 1 else "NO estable"}).']
    for n in IRF_PUNTO:
        for v in ORDEN:
            s = TAB_IRF[(TAB_IRF.choque == n) & (TAB_IRF.respuesta == v)]
            hmax = int(s.loc[s.irf.abs().idxmax(), 'h'])
            lin.append(f'- Choque {n} → {NOMBRES_VAR[v]}: impacto h=0 {s.irf.iloc[0] * 100:+.2f} p.p.; máximo en h={hmax}; '
                       f'efecto acumulado a {H} años {s.acumulada.iloc[-1] * 100:+.2f} p.p.')
    lin += ['', f'Choque localizado en {DPT.nombre[i0]}: en h=1, {(R_LOC[1, N:] > 1e-4).sum()} de los {N} departamentos '
            f'registran una respuesta positiva mayor a 0,01 p.p. en {NOMBRES_VAR[dn]}; la respuesta fuera del origen representa '
            f'{100 * (np.abs(R_LOC[:, N:]).sum() - np.abs(R_LOC[:, N + i0]).sum()) / np.abs(R_LOC[:, N:]).sum():.0f}% '
            f'de la respuesta total acumulada (difusión espacial).']
    return '\n'.join(lin)

INFORME = interpretar()
with open(os.path.join(CONFIG['OUT_DIR'], 'informe_spvar.md'), 'w', encoding='utf-8') as fh:
    fh.write(INFORME)
print(INFORME)

"""## Celda 13 · Visualización: selección, mecanismos, IRF y mapas estáticos"""

def lisa(x, W, n_perm=499, rng=RNG, alfa=0.05):
    z = (x - x.mean()) / x.std()
    lag = W @ z
    Ii = z * lag
    n = len(z); p = np.empty(n)
    for i in range(n):
        otros = np.delete(z, i); w = np.delete(W[i], i)
        perm = np.array([rng.permutation(otros) for _ in range(n_perm)])
        sim = z[i] * (perm @ w)
        p[i] = (1 + np.sum(np.abs(sim) >= abs(Ii[i]))) / (n_perm + 1)
    cuad = np.where(z > 0, np.where(lag > 0, 'AA', 'AB'), np.where(lag > 0, 'BA', 'BB'))
    return pd.DataFrame({'I_local': Ii, 'p': p, 'cluster': np.where(p < alfa, cuad, 'n.s.')})

GDF_M = GDF.set_index('cod_dpto').loc[CODIGOS].reset_index() if GDF is not None else None
CL_COL = {'AA': '#c0392b', 'BB': '#2e6fb7', 'AB': '#f1a9a0', 'BA': '#9ec5ea', 'n.s.': '#e6e6e6'}

def mapa(ax, valores, titulo, cmap='viridis', divergente=False, categorias=None):
    v = np.asarray(valores)
    ax.set_title(titulo, fontsize=10); ax.set_axis_off()
    if categorias is not None:
        colores = [categorias[c] for c in v]
        if GDF_M is not None:
            GDF_M.assign(_c=colores).plot(color=GDF_M.assign(_c=colores)['_c'], ax=ax, edgecolor='white', lw=0.4)
        else:
            ax.scatter(DPT.lon, DPT.lat, c=colores, s=160, edgecolor='k', lw=0.3)
        for c, col in categorias.items():
            ax.scatter([], [], c=col, label=c, s=40)
        ax.legend(fontsize=7, loc='lower left', frameon=False)
        return
    norm = None
    if divergente:
        m = np.nanmax(np.abs(v)) or 1.0
        norm = TwoSlopeNorm(0, -m, m); cmap = 'RdBu_r'
    if GDF_M is not None:
        GDF_M.assign(_v=v).plot(column='_v', cmap=cmap, norm=norm, ax=ax, edgecolor='white', lw=0.4,
                                legend=True, legend_kwds={'shrink': 0.6})
    else:
        sc = ax.scatter(DPT.lon, DPT.lat, c=v, cmap=cmap, norm=norm, s=160, edgecolor='k', lw=0.3)
        plt.colorbar(sc, ax=ax, shrink=0.6)
        for x_, y_, c in zip(DPT.lon, DPT.lat, CODIGOS):
            ax.annotate(c, (x_, y_), fontsize=6, ha='center', va='center')

def guardar(fig, nombre):
    ruta = os.path.join(CONFIG['OUT_DIR'], nombre)
    fig.savefig(ruta, dpi=150, bbox_inches='tight')
    plt.show(); plt.close(fig)
    return ruta

# --- (1) Mapa de calor de la selección W × modelo ---------------------------------------
c = ELECCION[next(iter(ELECCION))]['criterio']
fig, axs = plt.subplots(1, 2, figsize=(12, 4))
for ax, k in zip(axs, FINAL):
    m = TAB_AJ[TAB_AJ.ecuacion == k].pivot(index='W', columns='modelo', values=c)[CONFIG['MODELOS']]
    d = m - m.min().min()
    im = ax.imshow(d.values, cmap='magma_r')
    ax.set_xticks(range(d.shape[1]), d.columns); ax.set_yticks(range(d.shape[0]), d.index)
    for (i, j_), val in np.ndenumerate(d.values):
        sel = d.index[i] == ELECCION[k]['W'] and d.columns[j_] == ELECCION[k]['modelo']
        ax.text(j_, i, ('★ ' if sel else '') + f'{val:.1f}', ha='center', va='center', fontsize=8,
                color='white' if val > d.values.max() / 2 else 'black')
    ax.set_title(f'{NOMBRES_VAR[k]}: Δ{c} vs. mejor (★ = elegido)')
    plt.colorbar(im, ax=ax, shrink=0.8)
guardar(fig, '01_seleccion_W_modelo.png')

# --- (2) Gráfico de bosque de los mecanismos -------------------------------------------
fig, axs = plt.subplots(1, 2, figsize=(13, 6), sharex=False)
for ax, k in zip(axs, FINAL):
    m = MECANISMOS[(MECANISMOS.ecuacion == k)].iloc[::-1]
    col = m.mecanismo.map({u: plt.cm.tab10(i) for i, u in enumerate(MECANISMOS.mecanismo.unique())})
    ax.errorbar(m.estimacion, range(len(m)), xerr=1.96 * m.se, fmt='none', ecolor='grey', lw=1)
    ax.scatter(m.estimacion, range(len(m)), c=list(col), s=30, zorder=3,
               marker='o')
    ax.axvline(0, color='k', lw=0.6)
    ax.set_yticks(range(len(m)), [f'{t[:62]}' for t in m.termino], fontsize=7)
    ax.set_title(f'{NOMBRES_VAR[k]} ({FINAL[k].modelo}, W={FINAL[k].nombre_W})')
fig.suptitle('Mecanismos de transmisión: estimación ± IC95%', y=1.0)
fig.tight_layout()
guardar(fig, '02_mecanismos.png')

# --- (3) Funciones impulso-respuesta agregadas -----------------------------------------
choques = list(IRF_PUNTO)
fig, axs = plt.subplots(len(choques), 2, figsize=(11, 3 * len(choques)), squeeze=False)
for i, n in enumerate(choques):
    for j_, v in enumerate(ORDEN):
        s = TAB_IRF[(TAB_IRF.choque == n) & (TAB_IRF.respuesta == v)]
        ax = axs[i, j_]
        ax.fill_between(s.h, s.p05 * 100, s.p95 * 100, alpha=0.25, color='C0')
        ax.plot(s.h, s.irf * 100, 'o-', color='C0', ms=3)
        ax.axhline(0, color='k', lw=0.5)
        ax.set_title(f'{n} → {NOMBRES_VAR[v]}', fontsize=9); ax.set_ylabel('p.p. de crecimiento')
axs[-1, 0].set_xlabel('años'); axs[-1, 1].set_xlabel('años')
fig.tight_layout()
guardar(fig, '03_irf_agregadas.png')

# --- (4) Mapas estáticos de resultados principales --------------------------------------
up, dn = ORDEN
x_cruce = f'{up}_t' if f'{up}_t' in FINAL[dn].nombres else f'{up}_L1'
S_cruce = matriz_S(FINAL[dn], W_FINAL[dn], x_cruce)
media = PANEL.groupby('cod_dpto')[['agro', 'ind']].mean().loc[CODIGOS]
LISA = {v: lisa(media[v].values, W_FINAL[v]) for v in ['agro', 'ind']}
RES_DPTO = pd.DataFrame({
    'cod_dpto': CODIGOS, 'nombre': DPT.nombre.values,
    'crec_medio_agro': media.agro.values, 'crec_medio_ind': media.ind.values,
    'lisa_agro': LISA['agro'].cluster.values, 'lisa_ind': LISA['ind'].cluster.values,
    f'directo_{x_cruce}': np.diag(S_cruce),
    f'indirecto_recibido_{x_cruce}': S_cruce.sum(1) - np.diag(S_cruce),
    f'indirecto_emitido_{x_cruce}': S_cruce.sum(0) - np.diag(S_cruce),
    **{f'irf_loc_{v}_h{h}': R_LOC[h, b * N:(b + 1) * N] for h in range(4) for b, v in enumerate(ORDEN)},
})
if 'ndvi_anom' in CONFIG['VARS_SAT']:
    RES_DPTO['directo_ndvi_en_agro'] = np.diag(matriz_S(FINAL['agro'], W_FINAL['agro'], 'ndvi_anom'))
if DF_SAT is not None and 'ndvi' in DF_SAT:
    RES_DPTO['ndvi_medio'] = DF_SAT.groupby('cod_dpto').ndvi.mean().reindex(CODIGOS).values

fig, axs = plt.subplots(2, 4, figsize=(18, 9))
mapa(axs[0, 0], RES_DPTO.crec_medio_agro * 100, 'Crecimiento medio agro (%)', divergente=True)
mapa(axs[0, 1], RES_DPTO.crec_medio_ind * 100, f'Crecimiento medio {NOMBRES_VAR["ind"].lower()} (%)', divergente=True)
mapa(axs[0, 2], RES_DPTO.lisa_agro, 'LISA crecimiento agro', categorias=CL_COL)
mapa(axs[0, 3], RES_DPTO.lisa_ind, 'LISA crecimiento industria', categorias=CL_COL)
mapa(axs[1, 0], RES_DPTO[f'directo_{x_cruce}'], f'Impacto directo {x_cruce} → {dn}', cmap='YlGn')
mapa(axs[1, 1], RES_DPTO[f'indirecto_emitido_{x_cruce}'], f'Spillover EMITIDO {x_cruce} → {dn} vecinos', cmap='OrRd')
mapa(axs[1, 2], RES_DPTO[f'indirecto_recibido_{x_cruce}'], f'Spillover RECIBIDO por {dn}', cmap='PuBu')
if 'ndvi_medio' in RES_DPTO:
    mapa(axs[1, 3], RES_DPTO.ndvi_medio, 'NDVI medio (satélite)', cmap='YlGn')
else:
    axs[1, 3].set_axis_off()
fig.suptitle('Resultados espaciales clave del SpVAR', fontsize=13)
guardar(fig, '04_mapas_resultados.png')

# --- (5) Difusión espacio-temporal de un choque localizado -------------------------------
fig, axs = plt.subplots(2, 4, figsize=(18, 8))
for b, v in enumerate(ORDEN):
    vmax = np.abs(R_LOC[:4, b * N:(b + 1) * N]).max()
    for h in range(4):
        mapa(axs[b, h], R_LOC[h, b * N:(b + 1) * N] * 100, f'{NOMBRES_VAR[v]} · h={h} (p.p.)', divergente=True)
fig.suptitle(f'Difusión de un choque de 1 d.e. en {NOMBRES_VAR[up]} originado en {DPT.nombre[i0]}', fontsize=13)
guardar(fig, '05_difusion_choque_local.png')

"""## Celda 14 · Mapa coroplético interactivo (Folium) con capas conmutables"""

def mapa_interactivo():
    if GDF_M is None or folium is None:
        print('Mapa interactivo omitido (requiere geopandas + folium).'); return None
    g = GDF_M[['cod_dpto', 'geometry']].merge(RES_DPTO, on='cod_dpto')
    g = g.assign(**{c: g[c].astype(float).round(5) for c in g.columns if c.startswith(('crec', 'directo', 'indirecto', 'irf', 'ndvi'))})
    capas = [('crec_medio_agro', 'Crecimiento medio agro', 'RdBu_r'),
             ('crec_medio_ind', 'Crecimiento medio industria', 'RdBu_r'),
             (f'directo_{x_cruce}', f'Impacto directo {x_cruce}→{dn}', 'YlGn'),
             (f'indirecto_emitido_{x_cruce}', 'Spillover emitido', 'OrRd'),
             (f'indirecto_recibido_{x_cruce}', 'Spillover recibido', 'PuBu'),
             (f'irf_loc_{dn}_h1', f'Respuesta h=1 a choque en {DPT.nombre[i0]}', 'RdBu_r')]
    if 'ndvi_medio' in g:
        capas.append(('ndvi_medio', 'NDVI medio', 'YlGn'))
    tooltip = ['nombre'] + [c for c, _, _ in capas] + ['lisa_agro', 'lisa_ind']
    m = None
    for i, (col, nombre, cmap) in enumerate(capas):
        kw = dict(column=col, cmap=cmap, name=nombre, tooltip=tooltip, legend=(i == 0), show=(i == 0),
                  style_kwds={'weight': 0.5, 'color': 'white', 'fillOpacity': 0.8})
        if cmap == 'RdBu_r':
            vm = float(np.abs(g[col]).max()); kw.update(vmin=-vm, vmax=vm)
        m = g.explore(tiles=CONFIG.get('FONDO_MAPA'), **kw) if m is None else g.explore(m=m, **kw)
    g.explore(m=m, column='lisa_agro', categorical=True, name='Clusters LISA agro', show=False,
              cmap=[CL_COL[c] for c in sorted(g.lisa_agro.unique())], tooltip=['nombre', 'lisa_agro'])
    xmin, ymin, xmax, ymax = g.total_bounds
    m.fit_bounds([[ymin, xmin], [ymax, xmax]])            # encuadre automático en Colombia
    folium.LayerControl(collapsed=False).add_to(m)
    ruta = os.path.join(CONFIG['OUT_DIR'], 'mapa_interactivo_spvar.html')
    m.save(ruta); print('Mapa interactivo guardado en', ruta)
    return m

MAPA = mapa_interactivo()
MAPA

"""## Celda 15 · Exportación de tablas limpias (Excel + CSV)"""

def exportar():
    ruta = os.path.join(CONFIG['OUT_DIR'], 'resultados_spvar.xlsx')
    hojas = {
        'seleccion_modelos': TAB_AJ.round(5), 'pruebas_LM': TAB_LM.round(5), 'pruebas_LR': TAB_LR.round(5),
        'moran_residuos_MCO': TAB_MORAN.round(5), 'mecanismos': MECANISMOS.round(6),
        'impactos': pd.concat([d.assign(ecuacion=k) for k, d in IMPACTOS.items()]).round(6),
        'irf': TAB_IRF.round(6), 'resultados_dpto': RES_DPTO,
        'matrices_W': resumen_W(WS),
    }
    for k in FINAL:
        hojas[f'coef_{k}'] = FINAL[k].tabla().round(6).reset_index(names='parametro')
    with pd.ExcelWriter(ruta, engine='openpyxl') as xw:
        for nombre, df in hojas.items():
            df.to_excel(xw, sheet_name=nombre[:31], index=False)
            df.to_csv(os.path.join(CONFIG['OUT_DIR'], f'{nombre}.csv'), index=False)
    print('Tablas exportadas a', ruta)
    return ruta

RUTA_XLSX = exportar()

"""## Celda 16 · AUDITORÍA de código, estacionariedad, covarianzas y robustez
Cada verificación devuelve **OK / ADVERTENCIA / FALLA / OMITIDO** y se consolida en un veredicto final.
"""

AUDITORIA = []
def registrar(bloque, prueba, estado, detalle):
    AUDITORIA.append({'bloque': bloque, 'prueba': prueba, 'estado': estado, 'detalle': detalle})

# ---- A. Sintaxis y nombres no definidos ------------------------------------------------
def fuentes_codigo():
    # 1º historial de IPython (instantáneo y siempre disponible en Colab/Jupyter)
    try:
        hist = [c for c in get_ipython().user_ns.get('In', []) if c.strip()]
        if hist:
            return hist, 'historial IPython'
    except Exception:
        pass
    if '__file__' in globals():
        return [open(__file__, encoding='utf-8').read()], 'archivo .py'
    return [], 'no disponible'

def con_limite(fn, segundos, *args, **kw):
    """Ejecuta fn en un hilo; si supera el límite devuelve TimeoutError en vez de bloquear la celda."""
    import concurrent.futures as cf
    ex = cf.ThreadPoolExecutor(max_workers=1)
    fut = ex.submit(fn, *args, **kw)
    try:
        return fut.result(timeout=segundos)
    finally:
        ex.shutdown(wait=False)

T0_AUD = time.time()
def paso(txt):
    print(f'[{time.time() - T0_AUD:6.1f} s] {txt}', flush=True)
paso('A. Revisión de sintaxis')

celdas, origen = fuentes_codigo()
errores = []
for i, c in enumerate(celdas):
    limpio = '\n'.join('pass' if l.lstrip().startswith(('!', '%')) else l for l in c.splitlines())
    try:
        ast.parse(limpio)
    except SyntaxError as e:
        errores.append(f'celda {i}: línea {e.lineno}: {e.msg}')
registrar('A. Código', 'Sintaxis (ast.parse)', 'OK' if celdas and not errores else 'FALLA' if errores else 'OMITIDO',
          f'{len(celdas)} bloques desde {origen}; errores: {errores or "ninguno"}')
try:
    from pyflakes.api import check as _pf_check
    from pyflakes.reporter import Reporter
    import io
    buf = io.StringIO()
    _pf_check('\n\n'.join('\n'.join('pass' if l.lstrip().startswith(('!', '%')) else l for l in c.splitlines())
                          for c in celdas), 'notebook', Reporter(buf, buf))
    indef = [l for l in buf.getvalue().splitlines() if 'undefined name' in l and 'get_ipython' not in l]
    registrar('A. Código', 'Nombres no definidos (pyflakes)', 'OK' if not indef else 'ADVERTENCIA',
              '; '.join(indef[:5]) or 'ninguno')
except Exception as e:
    registrar('A. Código', 'Nombres no definidos (pyflakes)', 'OMITIDO', str(e)[:80])

# ---- B. Integridad de datos y matrices W ------------------------------------------------
paso('B. Integridad de datos y W')
num = PANEL[['agro', 'ind', 'agro_L1', 'ind_L1'] + CONFIG['VARS_SAT']]
registrar('B. Datos', 'Panel balanceado y finito', 'OK' if len(PANEL) == N * T and np.isfinite(num.values).all() else 'FALLA',
          f'NT={len(PANEL)} (= {N}×{T})')
for wn, wd in WS.items():
    Wm = wd['W']
    ok = np.allclose(Wm.sum(1), 1) and np.all(np.diag(Wm) == 0) and np.all(Wm >= 0)
    registrar('B. Datos', f'W {wn}: estandarizada, diagonal nula, sin islas', 'OK' if ok else 'FALLA',
              f'mín. vecinos={int((wd["B"] > 0).sum(1).min())}')
for k, (y, X, nm, _) in ECUACIONES.items():
    rk = np.linalg.matrix_rank(X)
    registrar('B. Datos', f'Rango completo de X ({k})', 'OK' if rk == X.shape[1] else 'FALLA', f'rango {rk}/{X.shape[1]}')

# ---- C. Estacionariedad -------------------------------------------------------------------
paso('C. Estacionariedad')
registrar('C. Estacionariedad', 'Radio espectral de Φ (forma reducida SpVAR)',
          'OK' if RADIO_ESPECTRAL < 0.95 else 'ADVERTENCIA' if RADIO_ESPECTRAL < 1 else 'FALLA', f'{RADIO_ESPECTRAL:.4f}')
for k, r in FINAL.items():
    mot = MotorEspacial(W_FINAL[k], N, T)
    for nom, v in [('ρ', r.rho), ('λ', r.lam)]:
        if v != 0:
            dentro = mot.lo < v < mot.hi
            borde = min(abs(v - mot.lo), abs(mot.hi - v)) < 0.02
            registrar('C. Estacionariedad', f'{nom} admisible ({k})', 'OK' if dentro and not borde else
                      'ADVERTENCIA' if dentro else 'FALLA', f'{v:.4f} ∈ ({mot.lo:.3f}, {mot.hi:.3f})')
if sm_tsa is not None:
    for v in ['agro', 'ind']:
        pv = []
        for c in CODIGOS:
            s = PANEL.loc[PANEL.cod_dpto == c, v].values
            pv.append(sm_tsa.adfuller(s, maxlag=1, autolag=None, regression='c')[1])
        P = -2 * np.sum(np.log(np.clip(pv, 1e-12, 1)))
        p = stats.chi2.sf(P, 2 * len(pv))
        registrar('C. Estacionariedad', f'Raíz unitaria panel Fisher-ADF (Maddala-Wu) en {v}',
                  'OK' if p < CONFIG['ALFA'] else 'ADVERTENCIA', f'P={P:.1f}, p={p:.4f} (H0: raíz unitaria en todos)')
else:
    registrar('C. Estacionariedad', 'Raíz unitaria panel Fisher-ADF', 'OMITIDO', 'statsmodels no disponible')

# ---- D. Matrices de covarianza ------------------------------------------------------------
paso('D. Covarianzas')
for k, r in FINAL.items():
    V = r.vcov
    ev = np.linalg.eigvalsh((V + V.T) / 2)
    simetrica = np.allclose(V, V.T, rtol=1e-6, atol=1e-12)
    cond = ev.max() / ev.min() if ev.min() > 0 else np.inf
    est = 'OK' if (simetrica and ev.min() > 0 and r.vcov_ok and np.isfinite(V).all()) else 'FALLA'
    registrar('D. Covarianzas', f'Var(θ̂) simétrica y definida positiva ({k})', est,
              f'autovalor mín={ev.min():.2e}, número de condición={cond:.2e}')
    mot = MotorEspacial(W_FINAL[k], N, T)
    y, X, nm, dz = ECUACIONES[k]
    Xa = np.column_stack([X, mot.lag(X[:, dz])]) if r.modelo == 'SDM' else X
    cr, cl = 'rho' in r.nombres_params, 'lambda' in r.nombres_params
    f = lambda t: mot.ll_completa(t, y, Xa, cr, cl)
    h = 1e-6 * (np.abs(r.params) + 1e-3)
    grad = np.array([(f(r.params + np.eye(len(h))[i] * h[i]) - f(r.params - np.eye(len(h))[i] * h[i])) / (2 * h[i])
                     for i in range(len(h))])
    gnorm = float(np.abs(grad * np.sqrt(np.diag(V))).max())
    registrar('D. Covarianzas', f'Óptimo MV: gradiente escalado ≈ 0 ({k})', 'OK' if gnorm < 1e-2 else 'ADVERTENCIA',
              f'max |g·se| = {gnorm:.2e}')
E = np.column_stack([FINAL[k].resid for k in ORDEN])
Sig = np.cov(E.T)
corr = Sig[0, 1] / np.sqrt(Sig[0, 0] * Sig[1, 1])
registrar('D. Covarianzas', 'Σ entre ecuaciones definida positiva', 'OK' if np.linalg.eigvalsh(Sig).min() > 0 else 'FALLA',
          f'Σ = {np.round(Sig, 6).tolist()}')
registrar('D. Covarianzas', 'Ortogonalidad de innovaciones (identificación recursiva)',
          'OK' if abs(corr) < 0.2 else 'ADVERTENCIA', f'corr(ε_{ORDEN[0]}, ε_{ORDEN[1]}) = {corr:.3f}')

# ---- E. Diagnóstico de residuos -----------------------------------------------------------
paso('E. Residuos')
for k, r in FINAL.items():
    mor = MotorEspacial(W_FINAL[k], N, T).moran_residuos(r.resid, CONFIG['N_PERM'], RNG)
    registrar('E. Residuos', f'Moran residual sin autocorrelación espacial ({k})',
              'OK' if mor['p_perm'] >= CONFIG['ALFA'] else 'ADVERTENCIA', f"I={mor['I']:.4f}, p={mor['p_perm']:.3f}")
    jb = stats.jarque_bera(r.resid)
    registrar('E. Residuos', f'Normalidad Jarque-Bera ({k})', 'OK' if jb.pvalue >= 0.01 else 'ADVERTENCIA',
              f'JB={jb.statistic:.2f}, p={jb.pvalue:.3f} (la MV es cuasi-MV consistente si falla)')
    registrar('E. Residuos', f'Convergencia del optimizador ({k})', 'OK' if r.convergio else 'FALLA', r.modelo)

# ---- F. Validación cruzada del motor con spreg (opcional) --------------------------------
paso('F. Validación cruzada con spreg (límite 90 s)')
def _rho_spreg():
    from libpysal.weights import full2W
    y, X, nm, _ = ECUACIONES['agro']
    w_nt = full2W(np.kron(np.eye(T), WS[ELECCION['agro']['W']]['W'])); w_nt.transform = 'r'
    m_sp = spreg.ML_Lag(y.reshape(-1, 1), X[:, 1:], w_nt, method='ord')
    return float(np.ravel(m_sp.rho)[0])
try:
    if not CONFIG.get('VALIDAR_SPREG', True) or spreg is None:
        raise RuntimeError('desactivada o spreg no disponible')
    wn = ELECCION['agro']['W']
    rho_sp = con_limite(_rho_spreg, 90)
    rho_mio = AJUSTES[('agro', wn)]['SAR'].rho
    registrar('F. Validación', 'ρ SAR motor propio vs spreg.ML_Lag', 'OK' if abs(rho_sp - rho_mio) < 1e-3 else 'ADVERTENCIA',
              f'propio={rho_mio:.5f}, spreg={rho_sp:.5f}')
except Exception as e:
    registrar('F. Validación', 'ρ SAR motor propio vs spreg.ML_Lag', 'OMITIDO', f'{type(e).__name__}: {str(e)[:70]}')

# ---- G. Robustez --------------------------------------------------------------------------
paso('G. Robustez (W alternativa, submuestras, jackknife)')
def _matrices_sub(panel_sub, k):
    y, X, nm, dz = matrices_ecuacion(panel_sub, k)
    din = [nm[i] for i in dz]
    keep = [i for i in range(X.shape[1]) if np.any(X[:, i] != 0)]
    for pref in ('T_', 'D_'):
        cols = [i for i in keep if nm[i].startswith(pref)]
        if cols and np.allclose(X[:, cols].sum(1), 1):
            keep.remove(cols[0])
    X, nm = X[:, keep], [nm[i] for i in keep]
    return y, X, nm, [nm.index(c) for c in din]

def reestimar(k, modelo, B, D, panel_sub, cods_sub, T_sub):
    B = conectar_islas(B, D)
    mot = MotorEspacial(estandarizar_filas(B), len(cods_sub), T_sub)
    y, X, nm, dz = _matrices_sub(panel_sub, k)
    return mot.ajustar(y, X, nm, modelo, dz)

def clave(k):
    up, dn = ORDEN
    return [x for x in ([f'{k}_L1', 'rho', 'lambda'] + ([f'{up}_t', f'W_{up}_t'] if k == dn else [])
                        + [f'{"ind" if k == "agro" else "agro"}_L1'])
            if x in FINAL[k].nombres_params]

ROBUSTEZ = []
for k, r in FINAL.items():
    base = {c: r.coef(c) for c in clave(k)}
    # (1) W alternativa: segunda mejor W para el mismo modelo
    s = TAB_AJ[(TAB_AJ.ecuacion == k) & (TAB_AJ.modelo == r.modelo) & (TAB_AJ.W != r.nombre_W)]
    if not s.empty:
        w_alt = s.loc[s.BIC.idxmin(), 'W']
        alt = AJUSTES[(k, w_alt)][r.modelo]
        for c in base:
            ROBUSTEZ.append({'ecuacion': k, 'ejercicio': f'W alternativa ({w_alt})', 'parametro': c,
                             'base': base[c], 'min': alt.coef(c), 'max': alt.coef(c),
                             'mismo_signo_%': 100.0 * (np.sign(alt.coef(c)) == np.sign(base[c]))})
    # (2) Submuestras temporales
    mitad = ANIOS[len(ANIOS) // 2]
    vals = {c: [] for c in base}
    for nom, sel in [('1ª mitad', PANEL.anio < mitad), ('2ª mitad', PANEL.anio >= mitad)]:
        ps = PANEL[sel].reset_index(drop=True)
        rr = reestimar(k, r.modelo, WS[r.nombre_W]['B'], WS[r.nombre_W]['D'], ps, CODIGOS, ps.anio.nunique())
        for c in base:
            vals[c].append(rr.coef(c))
    for c in base:
        ROBUSTEZ.append({'ecuacion': k, 'ejercicio': 'Submuestras temporales', 'parametro': c, 'base': base[c],
                         'min': min(vals[c]), 'max': max(vals[c]),
                         'mismo_signo_%': 100 * np.mean(np.sign(vals[c]) == np.sign(base[c]))})
    # (3) Jackknife: excluir un departamento a la vez (reconstruyendo W)
    if CONFIG['JACKKNIFE']:
        vals = {c: [] for c in base}
        paso(f'   jackknife {k}: {N} re-estimaciones {r.modelo}')
        for d in range(N):
            idx = [i for i in range(N) if i != d]
            ps = PANEL[PANEL.cod_dpto != CODIGOS[d]].reset_index(drop=True)
            B = WS[r.nombre_W]['B'][np.ix_(idx, idx)]; Dd = WS[r.nombre_W]['D'][np.ix_(idx, idx)]
            rr = reestimar(k, r.modelo, B, Dd, ps, [CODIGOS[i] for i in idx], T)
            for c in base:
                vals[c].append(rr.coef(c))
        for c in base:
            ROBUSTEZ.append({'ecuacion': k, 'ejercicio': 'Jackknife (N−1)', 'parametro': c, 'base': base[c],
                             'min': min(vals[c]), 'max': max(vals[c]),
                             'mismo_signo_%': 100 * np.mean(np.sign(vals[c]) == np.sign(base[c]))})
ROBUSTEZ = pd.DataFrame(ROBUSTEZ)
for (k, ej), g in ROBUSTEZ.groupby(['ecuacion', 'ejercicio']):
    # solo se exige estabilidad de signo a parámetros significativos en el modelo base
    sig = [c for c in g.parametro if FINAL[k].tabla().loc[c, 'p'] < CONFIG['ALFA']]
    gs = g[g.parametro.isin(sig)]
    estab = gs['mismo_signo_%'].min() if len(gs) else 100.0
    registrar('G. Robustez', f'{ej} ({k})', 'OK' if estab >= 90 else 'ADVERTENCIA',
              f'estabilidad mínima de signo en parámetros significativos: {estab:.0f}%')

# ---- H. Recuperación de parámetros (solo modo demo) ---------------------------------------
paso('H. Recuperación de parámetros')
RECUPERACION = None
if CONFIG['MODO_DATOS'] == 'demo' and DGP_VERDADERO['W'] in WS:
    filas = []
    for k in ['agro', 'ind']:
        v = DGP_VERDADERO[k]
        y, X, nm, dz = ECUACIONES[k]
        rr = MotorEspacial(WS[DGP_VERDADERO['W']]['W'], N, T).ajustar(y, X, nm, v['modelo'], dz)
        verdad = {**v['coef'], 'rho': v['rho']}
        t = rr.tabla()
        for c, val in verdad.items():
            if c in t.index:
                lo, hi = t.loc[c, 'coef'] - 1.96 * t.loc[c, 'se'], t.loc[c, 'coef'] + 1.96 * t.loc[c, 'se']
                filas.append({'ecuacion': k, 'parametro': c, 'verdadero': val, 'estimado': t.loc[c, 'coef'],
                              'IC95_inf': lo, 'IC95_sup': hi, 'cubre': lo <= val <= hi})
        sel_ok = ELECCION[k]['modelo'] == v['modelo'] and ELECCION[k]['W'] == DGP_VERDADERO['W']
        registrar('H. Recuperación', f'Selección identifica el DGP ({k})', 'OK' if sel_ok else 'ADVERTENCIA',
                  f"elegido {ELECCION[k]['modelo']}/{ELECCION[k]['W']} vs verdadero {v['modelo']}/{DGP_VERDADERO['W']}")
    RECUPERACION = pd.DataFrame(filas)
    cob = RECUPERACION.cubre.mean()
    registrar('H. Recuperación', 'Cobertura IC95% de parámetros verdaderos', 'OK' if cob >= 0.8 else 'ADVERTENCIA',
              f'{100 * cob:.0f}% de {len(RECUPERACION)} parámetros')
    display(RECUPERACION.round(4))

# ---- Veredicto -----------------------------------------------------------------------------
paso('Veredicto')
AUD = pd.DataFrame(AUDITORIA)
conteo = AUD.estado.value_counts().to_dict()
veredicto = 'FALLA' if conteo.get('FALLA', 0) else 'APROBADO CON ADVERTENCIAS' if conteo.get('ADVERTENCIA', 0) else 'APROBADO'
with pd.option_context('display.max_colwidth', 110):
    display(AUD)
display(ROBUSTEZ.round(4))
print(f'\nVEREDICTO DE AUDITORÍA: {veredicto} · {conteo}')
with pd.ExcelWriter(RUTA_XLSX, engine='openpyxl', mode='a', if_sheet_exists='replace') as xw:
    AUD.to_excel(xw, sheet_name='auditoria', index=False)
    ROBUSTEZ.to_excel(xw, sheet_name='robustez', index=False)
    if RECUPERACION is not None:
        RECUPERACION.to_excel(xw, sheet_name='recuperacion_DGP', index=False)
AUD.to_csv(os.path.join(CONFIG['OUT_DIR'], 'auditoria.csv'), index=False)

"""## Celda 17 · (Colab) Descargar todos los resultados en un .zip"""

import shutil
ZIP = shutil.make_archive(CONFIG['OUT_DIR'].rstrip('/'), 'zip', CONFIG['OUT_DIR'])
print('Resultados empaquetados en', ZIP)
if EN_COLAB:
    from google.colab import files
    files.download(ZIP)

"""## Celda 18 · App Gradio: datos, figuras e interpretación (Plotly)
Prepara todo lo que muestra la interfaz a partir de los resultados ya estimados arriba (no re-estima nada al interactuar):

* **Coropletas de visión nocturna** sobre los polígonos (sin teselas de fondo): escalas cian, fósforo, térmica y divergente nocturna; los departamentos sin dato quedan sin relleno (no equivalen a cero).
* **Relieve 3D sobre el shape**: cada departamento es su polígono extruido (malla `Mesh3d`); altura y color son una transformación explícita del valor, con escala fija entre años para que la animación sea comparable. El simulador anima la difusión de un choque sobre ese relieve.
* **Realidad aumentada**: exporta la maqueta a GLB (glTF 2.0 escrito sin dependencias) y la muestra en `<model-viewer>`; el botón de RA solo aparece en dispositivos compatibles.
* **Interpretabilidad automatizada**: textos generados solo con las cifras calculadas, etiquetados como *descriptivo*, *asociación condicional*, *respuesta hipotética del modelo* o *identificación*.

Opcional: si coloca la capa de departamentos del **MGN del DANE** en `CONFIG['RUTA_MGN']` (por defecto `/content/MGN_DPTO.zip`, campo `DPTO_CCDGO`), la app usa esas geometrías oficiales.
"""

instalar_faltantes({'plotly': 'plotly', 'gradio': 'gradio', 'mapbox_earcut': 'mapbox-earcut'})
import base64, html as _html, inspect, struct, urllib.request
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import gradio as gr
try:
    import mapbox_earcut as _earcut                   # triangulación robusta (con huecos); opcional
except Exception:
    _earcut = None

# =====================================================================================
# 1. Geometría: polígonos departamentales (MGN del DANE si se proporciona; si no, la capa ya cargada)
# =====================================================================================
RUTA_MGN = CONFIG.get('RUTA_MGN', '/content/MGN_DPTO.zip')   # capa de departamentos del MGN (campo DPTO_CCDGO)

def _gdf_app():
    if gpd is not None and os.path.exists(RUTA_MGN):
        try:
            g = gpd.read_file(f'zip://{RUTA_MGN}' if RUTA_MGN.endswith('.zip') else RUTA_MGN)
            campo = next(c for c in g.columns if c.upper() in ('DPTO_CCDGO', 'DPTO_CODIGO', 'COD_DPTO'))
            g['cod_dpto'] = g[campo].astype(str).str.extract(r'(\d+)')[0].str.zfill(2)
            g = g.dissolve(by='cod_dpto').reset_index().to_crs(4326)
            g = g[g.cod_dpto.isin(CODIGOS)].merge(DPT[['cod_dpto', 'nombre']], on='cod_dpto')
            return g, 'MGN — DANE'
        except Exception as e:
            print('No se pudo leer el MGN del DANE, se usa la capa cargada:', e)
    return GDF_M, 'geoBoundaries ADM1 (simplificada)'

GDF_APP, FUENTE_GEOM = _gdf_app() if GDF_M is not None else (None, 'centroides (sin polígonos)')

def geojson_app(tolerancia=0.01):
    if GDF_APP is not None:
        g = GDF_APP[['cod_dpto', 'nombre', 'geometry']].copy()
        g['geometry'] = g.geometry.simplify(tolerancia, preserve_topology=True)
        return json.loads(g.to_json())
    try:
        with urllib.request.urlopen(URL_GEO, timeout=60) as r:
            gj = json.load(r)
        iso2cod = {k: v[0] for k, v in TABLA_DPTOS.items()}
        feats = []
        for f in gj['features']:
            c = iso2cod.get(f['properties'].get('shapeISO'))
            if c in CODIGOS:
                f['properties']['cod_dpto'] = c
                feats.append(f)
        return {'type': 'FeatureCollection', 'features': feats}
    except Exception as e:
        print('Sin polígonos para la app (se usarán puntos):', e)
        return None

GEOJ = geojson_app(0.01)          # coropletas
GEOJ_3D = geojson_app(0.03)       # relieve 3D y GLB (menos vértices; solo visual)

# =====================================================================================
# 2. Panel largo para mapas, ESDA y 3D
# =====================================================================================
_SAT_COLS = [c for c in ['ndvi', 'ndvi_anom', 'ndvi_crop', 'crop_frac', 'precip', 'precip_anom', 'temp', 'temp_anom'] if DF_SAT is not None and c in DF_SAT.columns]
APP_PANEL = PANEL[['anio', 'cod_dpto', 'agro', 'ind', 'agro_niv', 'ind_niv']].copy()
if _SAT_COLS:
    APP_PANEL = APP_PANEL.merge(DF_SAT[['cod_dpto', 'anio'] + _SAT_COLS], on=['cod_dpto', 'anio'], how='left')
APP_PANEL = APP_PANEL.merge(DPT[['cod_dpto', 'nombre', 'lat', 'lon']], on='cod_dpto')
APP_PANEL['agro_pct'] = APP_PANEL['agro'] * 100
APP_PANEL['ind_pct'] = APP_PANEL['ind'] * 100
APP_PANEL['razon_ind_agro'] = APP_PANEL['ind_niv'] / APP_PANEL['agro_niv']

# =====================================================================================
# 3. Estética de visión nocturna
# =====================================================================================
NV = dict(fondo='#02060c', panel='#06101a', borde='#0f2a3d', texto='#cfefff', tenue='#6f94a8',
          cian='#22d3ee', turquesa='#2dd4bf', ambar='#fbbf24', neg='#ff6b3d', blanco='#e9fdff')
ESCALAS = {
    'Visión nocturna · cian': [[0, '#020a10'], [0.2, '#06303f'], [0.45, '#0b6e7f'], [0.7, '#1fc8d8'],
                               [0.88, '#7af7f0'], [1, '#e6fffd']],
    'Visión nocturna · fósforo': [[0, '#010805'], [0.25, '#06341f'], [0.55, '#0f8a4a'], [0.8, '#39ff88'],
                                  [1, '#d9ffe9']],
    'Térmica nocturna': [[0, '#03040c'], [0.3, '#2a0b4a'], [0.55, '#a3166b'], [0.78, '#ff7a2f'], [1, '#fff3b0']],
    'Divergente nocturna': [[0, '#ffc28a'], [0.18, '#ff6b3d'], [0.38, '#5a1a0e'], [0.5, '#03080f'],
                            [0.62, '#06425a'], [0.82, '#22d3ee'], [1, '#c4fbff']],
}
ESC_DIV, ESC_SEQ = ESCALAS['Divergente nocturna'], ESCALAS['Visión nocturna · cian']
PALETAS = ['Auto'] + list(ESCALAS)
COLORES_LISA = {'Alto-Alto': '#ff5d3a', 'Bajo-Bajo': '#22d3ee', 'Alto-Bajo': '#ffb36b',
                'Bajo-Alto': '#7dd3fc', 'No significativo': '#0f1c26'}
ETQ_LISA = {'AA': 'Alto-Alto', 'BB': 'Bajo-Bajo', 'AB': 'Alto-Bajo', 'BA': 'Bajo-Alto', 'n.s.': 'No significativo'}
FUENTE = 'IBM Plex Sans, Inter, system-ui, sans-serif'

# etiqueta -> (fuente, columna, divergente, escala, ecuación cuya W se usa en ESDA)
VARS_MAPA = {
    'Crecimiento agropecuario (%)': ('panel', 'agro_pct', True, ESC_DIV, 'agro'),
    f'Crecimiento {NOMBRES_VAR["ind"].lower()} (%)': ('panel', 'ind_pct', True, ESC_DIV, 'ind'),
    'Nivel agropecuario': ('panel', 'agro_niv', False, ESC_SEQ, 'agro'),
    f'Nivel {NOMBRES_VAR["ind"].lower()}': ('panel', 'ind_niv', False, ESCALAS['Visión nocturna · fósforo'], 'ind'),
    'Razón industria / agro (niveles)': ('panel', 'razon_ind_agro', False, ESCALAS['Térmica nocturna'], 'ind'),
}
for col, etq, div in [('ndvi', 'NDVI medio anual (satélite)', False), ('ndvi_anom', 'Anomalía NDVI (d.e.)', True),
                      ('ndvi_crop', 'NDVI en zonas de cultivo', False), ('crop_frac', 'Fracción agrícola (MODIS)', False),
                      ('precip', 'Precipitación anual (mm, NASA POWER)', False),
                      ('precip_anom', 'Anomalía de precipitación (d.e.)', True),
                      ('temp', 'Temperatura media anual (°C, NASA POWER)', False),
                      ('temp_anom', 'Anomalía de temperatura (d.e.)', True)]:
    if col in _SAT_COLS:
        VARS_MAPA[etq] = ('panel', col, div, ESC_DIV if div else ESCALAS['Visión nocturna · fósforo'], 'agro')
VARS_MAPA.update({
    f'Impacto directo {x_cruce} → {dn}': ('res', f'directo_{x_cruce}', False, ESC_SEQ, dn),
    'Spillover emitido a vecinos': ('res', f'indirecto_emitido_{x_cruce}', False, ESCALAS['Térmica nocturna'], dn),
    'Spillover recibido de vecinos': ('res', f'indirecto_recibido_{x_cruce}', False, ESCALAS['Térmica nocturna'], dn),
})
VARS_PANEL = [k for k, v in VARS_MAPA.items() if v[0] == 'panel']
NOMBRE_A_COD = dict(zip(DPT.nombre, DPT.cod_dpto))
COD_A_NOMBRE = dict(zip(DPT.cod_dpto, DPT.nombre))
ES_DEMO = CONFIG['MODO_DATOS'] == 'demo'

def estilo(fig, titulo, alto=560):
    fig.update_layout(template='plotly_dark', height=alto, title=dict(text=titulo, x=0.01, font=dict(size=15, color=NV['cian'])),
                      margin=dict(l=8, r=8, t=56, b=8), paper_bgcolor=NV['fondo'], plot_bgcolor=NV['fondo'],
                      font=dict(family=FUENTE, color=NV['texto']), uirevision='fijo',
                      legend=dict(bgcolor='rgba(6,16,26,0.7)', bordercolor=NV['borde'], borderwidth=1),
                      hoverlabel=dict(bgcolor=NV['panel'], bordercolor=NV['cian'], font=dict(color=NV['blanco'])))
    if ES_DEMO:
        fig.add_annotation(text='DATOS SINTÉTICOS DE DEMOSTRACIÓN', xref='paper', yref='paper', x=0.99, y=0.01,
                           xanchor='right', yanchor='bottom', showarrow=False,
                           font=dict(size=10, color=NV['ambar']), opacity=0.85)
    return fig

# =====================================================================================
# 4. Coropletas de visión nocturna
# =====================================================================================
def _geo(fig):
    fig.update_geos(visible=True, bgcolor=NV['fondo'], projection_type='mercator',
                    lonaxis_range=[-82.2, -66.3], lataxis_range=[-4.6, 13.7],
                    showland=True, landcolor='#040b12', showocean=True, oceancolor='#01040a',
                    showcountries=True, countrycolor='#0c2a3a', showcoastlines=True, coastlinecolor='#0b3b4f',
                    showlakes=False, showframe=False)
    return fig

def fig_mapa(d, col, titulo, escala=None, divergente=False, categorica=None, etiqueta=None, alto=600,
             rango=None, resaltar=None):
    """Coropleta sobre polígonos (sin teselas). Los faltantes se dibujan sin relleno: no equivalen a cero."""
    d = d.copy()
    etiqueta = etiqueta or col
    fig = go.Figure()
    if GEOJ is not None and categorica:
        for cat, colr in categorica.items():
            s = d[d[col] == cat]
            if len(s):
                fig.add_trace(go.Choropleth(geojson=GEOJ, featureidkey='properties.cod_dpto', locations=s.cod_dpto,
                                            z=[1] * len(s), colorscale=[[0, colr], [1, colr]], showscale=False,
                                            name=cat, showlegend=True, text=s.nombre,
                                            hovertemplate='%{text}: ' + cat + '<extra></extra>',
                                            marker=dict(line=dict(color='rgba(34,211,238,0.45)', width=0.7))))
    elif GEOJ is not None:
        v = d[col].astype(float)
        if rango is None:
            if divergente:
                m = float(np.nanmax(np.abs(v))) if np.isfinite(v).any() else 1.0
                rango = (-m, m)
            else:
                rango = (float(np.nanmin(v)), float(np.nanmax(v))) if np.isfinite(v).any() else (0, 1)
        ok = np.isfinite(v)
        fig.add_trace(go.Choropleth(
            geojson=GEOJ, featureidkey='properties.cod_dpto', locations=d.cod_dpto[ok], z=v[ok],
            colorscale=escala or (ESC_DIV if divergente else ESC_SEQ), zmin=rango[0], zmax=rango[1],
            marker=dict(line=dict(color='rgba(34,211,238,0.55)', width=0.8)), text=d.nombre[ok],
            colorbar=dict(title=dict(text=etiqueta, font=dict(size=11)), thickness=11, len=0.72,
                          outlinecolor=NV['borde'], tickfont=dict(color=NV['texto'])),
            hovertemplate='<b>%{text}</b><br>' + etiqueta + ': %{z:.4g}<extra></extra>'))
        if (~ok).any():
            fig.add_trace(go.Choropleth(geojson=GEOJ, featureidkey='properties.cod_dpto', locations=d.cod_dpto[~ok],
                                        z=[0] * int((~ok).sum()), colorscale=[[0, '#0a0f14'], [1, '#0a0f14']],
                                        showscale=False, text=d.nombre[~ok], name='sin dato',
                                        marker=dict(line=dict(color='rgba(120,140,150,0.6)', width=0.6)),
                                        hovertemplate='%{text}: sin dato<extra></extra>'))
    else:
        d = d.merge(DPT[['cod_dpto', 'lat', 'lon']], on='cod_dpto', how='left')
        fig.add_trace(go.Scattergeo(lat=d.lat, lon=d.lon, text=d.nombre, mode='markers',
                                    marker=dict(size=15, color=d[col] if not categorica else NV['cian'],
                                                colorscale=escala or ESC_SEQ, line=dict(width=0.6, color=NV['cian']))))
    if resaltar and GEOJ is not None:
        fig.add_trace(go.Choropleth(geojson=GEOJ, featureidkey='properties.cod_dpto', locations=[resaltar], z=[0],
                                    colorscale=[[0, 'rgba(0,0,0,0)'], [1, 'rgba(0,0,0,0)']], showscale=False,
                                    hoverinfo='skip', marker=dict(line=dict(color=NV['blanco'], width=3))))
    return estilo(_geo(fig), titulo, alto)

def fig_ranking(d, col, etiqueta, divergente):
    d = d.dropna(subset=[col]).sort_values(col)
    colores = np.where(d[col] >= 0, NV['cian'], NV['neg']) if divergente else NV['turquesa']
    fig = go.Figure(go.Bar(x=d[col], y=d.nombre, orientation='h', marker=dict(color=colores, line=dict(width=0)),
                           hovertemplate='%{y}: %{x:.4g}<extra></extra>'))
    fig.update_yaxes(tickfont=dict(size=10), gridcolor=NV['borde'])
    fig.update_xaxes(gridcolor=NV['borde'], zerolinecolor=NV['tenue'])
    return estilo(fig, f'Ranking · {etiqueta}', 600)

def datos_variable(etiqueta, anio):
    src, col, div, esc, _ = VARS_MAPA[etiqueta]
    if src == 'panel':
        d = APP_PANEL[APP_PANEL.anio == int(anio)][['cod_dpto', 'nombre', col]]
        d = d.set_index('cod_dpto').reindex(CODIGOS).reset_index()
        d['nombre'] = d.cod_dpto.map(COD_A_NOMBRE)
        tit = f'{etiqueta} · {int(anio)}'
    else:
        d = RES_DPTO[['cod_dpto', 'nombre', col]]
        tit = f'{etiqueta} · resultado del SpVAR (no varía por año)'
    return d, col, div, esc, tit

# =====================================================================================
# 5. ESDA (Moran, Geary, LISA)
# =====================================================================================
def moran_geary(x, Wm, n_perm=999, semilla=CONFIG['SEMILLA']):
    rng = np.random.default_rng(semilla)
    x = np.asarray(x, float); n = len(x); S0 = Wm.sum()
    def I_(v):
        z = v - v.mean(); return n / S0 * (z @ Wm @ z) / (z @ z)
    def C_(v):
        z = v - v.mean(); return (n - 1) * np.sum(Wm * (v[:, None] - v[None, :]) ** 2) / (2 * S0 * (z @ z))
    I, C = I_(x), C_(x)
    perms = [rng.permutation(x) for _ in range(n_perm)]
    Is, Cs = np.array([I_(v) for v in perms]), np.array([C_(v) for v in perms])
    EI = -1 / (n - 1)
    pI = (1 + np.sum(Is >= I if I >= EI else Is <= I)) / (n_perm + 1)
    pC = (1 + np.sum(Cs <= C if C <= 1 else Cs >= C)) / (n_perm + 1)
    return {'I': I, 'E[I]': EI, 'p_I': pI, 'C': C, 'p_C': pC}

def interpretar_moran(r, alfa=CONFIG['ALFA']):
    if r['p_I'] >= alfa:
        return 'Distribución espacial compatible con aleatoriedad'
    return 'Agrupamiento: vecinos similares' if r['I'] > r['E[I]'] else 'Dispersión: vecinos distintos'

def tabla_esda(anio, wn):
    filas = []
    for etq in VARS_PANEL:
        col = VARS_MAPA[etq][1]
        d = APP_PANEL[APP_PANEL.anio == int(anio)].set_index('cod_dpto').reindex(CODIGOS)
        x = d[col].values.astype(float)
        if not np.isfinite(x).all() or np.std(x) == 0:
            continue
        r = moran_geary(x, WS[wn]['W'], 499)
        filas.append({'Variable': etq, 'Moran I': round(r['I'], 4), 'p (Moran)': round(r['p_I'], 4),
                      'Geary C': round(r['C'], 4), 'p (Geary)': round(r['p_C'], 4), 'Lectura': interpretar_moran(r)})
    return pd.DataFrame(filas)

def fig_moran(x, Wm, nombres, clusters, r, etiqueta):
    z = (x - x.mean()) / x.std()
    wz = Wm @ z
    pend = float(z @ wz / (z @ z))
    fig = go.Figure()
    for cat, colr in COLORES_LISA.items():
        m = np.asarray(clusters) == cat
        if m.any():
            fig.add_trace(go.Scatter(x=z[m], y=wz[m], mode='markers', name=cat, text=np.asarray(nombres)[m],
                                     marker=dict(size=12, color=colr if cat != 'No significativo' else '#3d5566',
                                                 line=dict(width=0.6, color=NV['cian'])),
                                     hovertemplate='%{text}<br>z=%{x:.2f} · Wz=%{y:.2f}<extra></extra>'))
    xs = np.linspace(z.min() - 0.3, z.max() + 0.3, 50)
    fig.add_trace(go.Scatter(x=xs, y=pend * xs, mode='lines', name=f'pendiente = {pend:.3f}',
                             line=dict(color=NV['ambar'], width=2, dash='dash')))
    fig.add_hline(y=0, line_color=NV['borde']); fig.add_vline(x=0, line_color=NV['borde'])
    fig.update_xaxes(title=f'{etiqueta} (estandarizada)', gridcolor=NV['borde'])
    fig.update_yaxes(title='Rezago espacial W·z', gridcolor=NV['borde'])
    return estilo(fig, f"Diagrama de Moran · I = {r['I']:.3f} (p = {r['p_I']:.3f})", 560)

def vista_esda(etiqueta, anio, wn):
    col = VARS_MAPA[etiqueta][1]
    d = APP_PANEL[APP_PANEL.anio == int(anio)].set_index('cod_dpto').reindex(CODIGOS).reset_index()
    d['nombre'] = d.cod_dpto.map(COD_A_NOMBRE)
    x = d[col].values.astype(float)
    Wm = WS[wn]['W']
    r = moran_geary(x, Wm)
    lis = lisa(x, Wm, n_perm=499, rng=np.random.default_rng(CONFIG['SEMILLA']))
    d['cluster'] = lis['cluster'].map(ETQ_LISA).values
    mapa_l = fig_mapa(d[['cod_dpto', 'nombre', 'cluster']], 'cluster', f'Clusters LISA · {etiqueta} · {int(anio)}',
                      categorica=COLORES_LISA, etiqueta='Cluster', alto=560)
    txt = interpretar_esda(etiqueta, anio, wn, r, d)
    return fig_moran(x, Wm, d.nombre.values, d.cluster.values, r, etiqueta), mapa_l, tabla_esda(anio, wn), txt

# =====================================================================================
# 6. Relieve 3D sobre los polígonos (extrusión de la forma real de cada departamento)
# =====================================================================================
def _area2(P):
    return 0.5 * np.sum(P[:, 0] * np.roll(P[:, 1], -1) - np.roll(P[:, 0], -1) * P[:, 1])

def _en_triangulo(Q, a, b, c):
    def s(p, q, r):
        return (p[:, 0] - r[0]) * (q[1] - r[1]) - (q[0] - r[0]) * (p[:, 1] - r[1])
    d1, d2, d3 = s(Q, a, b), s(Q, b, c), s(Q, c, a)
    neg = (d1 < 0) | (d2 < 0) | (d3 < 0)
    pos = (d1 > 0) | (d2 > 0) | (d3 > 0)
    return ~(neg & pos)

def _earclip(P):
    """Triangulación por recorte de orejas de un anillo simple (respaldo sin dependencias)."""
    n = len(P)
    idx = list(range(n)) if _area2(P) > 0 else list(range(n))[::-1]
    tris, guarda = [], 0
    while len(idx) > 3 and guarda < 4 * n:
        guarda += 1
        m, hecho = len(idx), False
        for t in range(m):
            i0, i1, i2 = idx[t - 1], idx[t], idx[(t + 1) % m]
            a, b, c = P[i0], P[i1], P[i2]
            cruz = (b[0] - a[0]) * (c[1] - a[1]) - (b[1] - a[1]) * (c[0] - a[0])
            if cruz <= 1e-15:
                continue
            otros = [j for j in idx if j not in (i0, i1, i2)]
            if otros and _en_triangulo(P[otros], a, b, c).any():
                continue
            tris.append((i0, i1, i2)); idx.pop(t); hecho = True
            break
        if not hecho:                      # vértice colineal o anillo degenerado: se retira y continúa
            cruces = [abs((P[idx[t]][0] - P[idx[t - 1]][0]) * (P[idx[(t + 1) % m]][1] - P[idx[t - 1]][1])
                          - (P[idx[t]][1] - P[idx[t - 1]][1]) * (P[idx[(t + 1) % m]][0] - P[idx[t - 1]][0]))
                      for t in range(m)]
            idx.pop(int(np.argmin(cruces)))
    if len(idx) == 3:
        tris.append(tuple(idx))
    return np.array(tris, dtype=np.int64).reshape(-1, 3)

def _triangular(ext, huecos):
    if _earcut is not None:
        V = np.vstack([ext] + huecos).astype(np.float64)
        fin = np.cumsum([len(ext)] + [len(h) for h in huecos]).astype(np.uint32)
        return V, np.asarray(_earcut.triangulate_float64(V, fin), dtype=np.int64).reshape(-1, 3)
    return ext, _earclip(ext)               # sin earcut: los huecos se ignoran en la tapa

def _anillo(c):
    P = np.asarray(c, float)[:, :2]
    return P[:-1] if len(P) > 3 and np.allclose(P[0], P[-1]) else P

def _topologia():
    """Por departamento: vértices 2D, triángulos de la tapa y anillos de borde (para las paredes)."""
    topo = {}
    if GEOJ_3D is None:
        return topo
    for f in GEOJ_3D['features']:
        c = f['properties']['cod_dpto']; g = f['geometry']
        polis = [g['coordinates']] if g['type'] == 'Polygon' else g['coordinates']
        partes = []
        for p in polis:
            ext = _anillo(p[0])
            if len(ext) < 3 or abs(_area2(ext)) < 1e-6:
                continue
            huecos = [_anillo(h) for h in p[1:] if len(h) > 3]
            V, tri = _triangular(ext, huecos)
            partes.append({'V': V, 'tri': tri, 'anillos': [ext] + huecos})
        topo[c] = partes
    return topo

TOPO = _topologia()

def alturas(valores, modo='lineal', ref=None, z_max=1.0):
    """Transformación explícita valor → altura. Devuelve (dict cod→altura, fórmula).
    lineal: h = z_max·v/ref ; raíz: h = z_max·signo(v)·√(|v|/ref) ; nivel: h = z_max·(v−mín)/(máx−mín)."""
    v = np.array([valores.get(c, np.nan) for c in CODIGOS], float)
    fin = np.isfinite(v)
    if not fin.any():
        return {}, 'sin datos'
    if modo == 'nivel':
        lo, hi = ref if ref else (np.nanmin(v), np.nanmax(v))
        h = z_max * (v - lo) / ((hi - lo) or 1)
        f = f'h = {z_max:g}·(x − {lo:.4g}) / ({hi:.4g} − {lo:.4g})'
    else:
        r = ref or float(np.nanmax(np.abs(v))) or 1.0
        if modo == 'raiz':
            h = z_max * np.sign(v) * np.sqrt(np.abs(v) / r)
            f = f'h = {z_max:g}·signo(x)·√(|x| / {r:.4g})'
        else:
            h = z_max * v / r
            f = f'h = {z_max:g}·x / {r:.4g}'
        h = np.clip(h, -1.6 * z_max, 1.6 * z_max)
    return {c: float(x) for c, x in zip(CODIGOS, h) if np.isfinite(x)}, f

def malla(h):
    """Malla extruida: tapas a la altura h del departamento y paredes hasta z = 0."""
    X, Y, Z, I, J, K, cod = [], [], [], [], [], [], []
    base = 0
    for c, partes in TOPO.items():
        if c not in h:
            continue
        for p in partes:
            V, tri = p['V'], p['tri']
            X.append(V[:, 0]); Y.append(V[:, 1]); Z.append(np.full(len(V), h[c])); cod += [c] * len(V)
            I.append(tri[:, 0] + base); J.append(tri[:, 1] + base); K.append(tri[:, 2] + base)
            base += len(V)
            for A in p['anillos']:
                m = len(A)
                B = np.roll(A, -1, axis=0)
                xs = np.column_stack([A[:, 0], B[:, 0], B[:, 0], A[:, 0]]).ravel()
                ys = np.column_stack([A[:, 1], B[:, 1], B[:, 1], A[:, 1]]).ravel()
                zs = np.tile([0.0, 0.0, h[c], h[c]], m)
                q = base + 4 * np.arange(m)
                X.append(xs); Y.append(ys); Z.append(zs); cod += [c] * (4 * m)
                I.append(np.r_[q, q]); J.append(np.r_[q + 1, q + 2]); K.append(np.r_[q + 2, q + 3])
                base += 4 * m
    if not X:
        return None
    cat = lambda L: np.concatenate(L)
    return dict(x=cat(X), y=cat(Y), z=cat(Z), i=cat(I), j=cat(J), k=cat(K), cod=np.array(cod))

def _bordes(h, cods, z0=None):
    xs, ys, zs = [], [], []
    for c in cods:
        for p in TOPO.get(c, []):
            for A in p['anillos']:
                A2 = np.vstack([A, A[:1]])
                xs += list(A2[:, 0]) + [None]; ys += list(A2[:, 1]) + [None]
                zs += [h.get(c, 0.0) if z0 is None else z0] * len(A2) + [None]
    return xs, ys, zs

def fig_relieve(valores, titulo, etiqueta, escala=None, divergente=False, modo='lineal', ref=None,
                rango_color=None, resaltar=None, alto=680, z_max=1.0):
    """Relieve 3D: cada departamento es su polígono extruido; altura y color = valor (o efecto)."""
    if not TOPO:
        return estilo(go.Figure(), 'Relieve 3D no disponible: se requieren polígonos', 300), ''
    h, formula = alturas(valores, modo, ref, z_max)
    M = malla(h)
    fig = go.Figure()
    if M is not None:
        val = np.array([valores[c] for c in M['cod']], float)
        if rango_color is None:
            m = float(np.nanmax(np.abs(val))) or 1.0
            rango_color = (-m, m) if divergente else (float(np.nanmin(val)), float(np.nanmax(val)))
        txt = [f'{COD_A_NOMBRE[c]} ({c})' for c in M['cod']]
        fig.add_trace(go.Mesh3d(
            x=M['x'], y=M['y'], z=M['z'], i=M['i'], j=M['j'], k=M['k'], intensity=val, intensitymode='vertex',
            colorscale=escala or (ESC_DIV if divergente else ESC_SEQ), cmin=rango_color[0], cmax=rango_color[1],
            flatshading=True, text=txt, customdata=val,
            hovertemplate='<b>%{text}</b><br>' + etiqueta + ': %{customdata:.4g}<extra></extra>',
            lighting=dict(ambient=0.55, diffuse=0.75, specular=0.35, roughness=0.55, fresnel=0.25),
            lightposition=dict(x=-30000, y=-60000, z=90000),
            colorbar=dict(title=dict(text=etiqueta, font=dict(size=11)), thickness=11, len=0.65,
                          outlinecolor=NV['borde'], tickfont=dict(color=NV['texto']))))
        bx, by, bz = _bordes(h, list(h))
        fig.add_trace(go.Scatter3d(x=bx, y=by, z=bz, mode='lines', hoverinfo='skip', showlegend=False,
                                   line=dict(color='rgba(34,211,238,0.75)', width=2)))
    faltan = [c for c in CODIGOS if c not in h]
    if faltan:
        bx, by, bz = _bordes(h, faltan, z0=0.0)
        fig.add_trace(go.Scatter3d(x=bx, y=by, z=bz, mode='lines', name='sin dato', hoverinfo='skip',
                                   line=dict(color='rgba(140,160,170,0.6)', width=2, dash='dot')))
    if resaltar and resaltar in TOPO:
        bx, by, bz = _bordes(h, [resaltar])
        fig.add_trace(go.Scatter3d(x=bx, y=by, z=bz, mode='lines', name=COD_A_NOMBRE[resaltar], hoverinfo='skip',
                                   line=dict(color=NV['blanco'], width=6)))
    zr = [-1.7 * z_max, 1.7 * z_max] if divergente else [0, 1.15 * z_max]
    ejes = dict(showbackground=True, backgroundcolor=NV['fondo'], gridcolor='#0b2233', zerolinecolor='#0b2233',
                showspikes=False, color=NV['tenue'])
    fig.update_layout(scene=dict(xaxis=dict(title='Longitud', **ejes), yaxis=dict(title='Latitud', **ejes),
                                 zaxis=dict(title='altura', range=zr, **ejes), bgcolor=NV['fondo'],
                                 aspectmode='manual', aspectratio=dict(x=1, y=1.15, z=0.42),
                                 camera=dict(eye=dict(x=0.85, y=-1.45, z=0.95))))
    return estilo(fig, titulo, alto), formula

# =====================================================================================
# 7. Realidad aumentada: GLB propio (sin dependencias) + visor <model-viewer>
# =====================================================================================
def _hex_rgb(hx):
    hx = hx.lstrip('#'); return np.array([int(hx[i:i + 2], 16) for i in (0, 2, 4)], float) / 255

def color_escala(t, escala):
    t = np.clip(np.nan_to_num(t, nan=0.0), 0, 1)
    pos = np.array([p for p, _ in escala]); cols = np.array([_hex_rgb(c) for _, c in escala])
    return np.column_stack([np.interp(t, pos, cols[:, k]) for k in range(3)])

def glb_bytes(M, rgb, ancho_m=1.2, alto_m=0.18):
    """glTF 2.0 binario: maqueta de mesa de `ancho_m` metros (eje Y hacia arriba)."""
    lat0 = np.radians(4.5)
    ex = (M['x'] - M['x'].mean()) * 111.32 * np.cos(lat0)
    nz = -(M['y'] - M['y'].mean()) * 110.57
    s = ancho_m / max(np.ptp(ex), np.ptp(nz))
    pos = np.column_stack([ex * s, M['z'] * alto_m, nz * s]).astype(np.float32)
    col = np.column_stack([rgb, np.ones(len(rgb))]).astype(np.float32)
    idx = np.column_stack([M['i'], M['j'], M['k']]).astype(np.uint32).ravel()
    blobs = [pos.tobytes(), col.tobytes(), idx.tobytes()]
    vistas, binario, off = [], b'', 0
    for b, tgt in zip(blobs, (34962, 34962, 34963)):
        vistas.append({'buffer': 0, 'byteOffset': off, 'byteLength': len(b), 'target': tgt})
        b += b'\x00' * (-len(b) % 4)
        binario += b; off += len(b)
    gltf = {'asset': {'version': '2.0', 'generator': 'SpVAR Colombia'},
            'scene': 0, 'scenes': [{'nodes': [0]}], 'nodes': [{'mesh': 0, 'name': 'Colombia'}],
            'meshes': [{'primitives': [{'attributes': {'POSITION': 0, 'COLOR_0': 1}, 'indices': 2, 'material': 0}]}],
            'materials': [{'pbrMetallicRoughness': {'baseColorFactor': [1, 1, 1, 1], 'metallicFactor': 0.0,
                                                    'roughnessFactor': 0.8}, 'doubleSided': True}],
            'accessors': [
                {'bufferView': 0, 'componentType': 5126, 'count': len(pos), 'type': 'VEC3',
                 'min': pos.min(0).tolist(), 'max': pos.max(0).tolist()},
                {'bufferView': 1, 'componentType': 5126, 'count': len(col), 'type': 'VEC4'},
                {'bufferView': 2, 'componentType': 5125, 'count': len(idx), 'type': 'SCALAR'}],
            'bufferViews': vistas, 'buffers': [{'byteLength': len(binario)}]}
    js = json.dumps(gltf, separators=(',', ':')).encode()
    js += b' ' * (-len(js) % 4)
    total = 12 + 8 + len(js) + 8 + len(binario)
    return (struct.pack('<4sII', b'glTF', 2, total) + struct.pack('<I4s', len(js), b'JSON') + js
            + struct.pack('<I4s', len(binario), b'BIN\x00') + binario)

def exportar_glb(valores, nombre, escala, divergente, modo='lineal', ref=None):
    h, formula = alturas(valores, modo, ref)
    M = malla(h)
    if M is None:
        return None, formula
    val = np.array([valores[c] for c in M['cod']], float)
    m = float(np.nanmax(np.abs(val))) or 1.0
    t = (val + m) / (2 * m) if divergente else (val - np.nanmin(val)) / ((np.nanmax(val) - np.nanmin(val)) or 1)
    ruta = os.path.join(CONFIG['OUT_DIR'], f'{nombre}.glb')
    with open(ruta, 'wb') as fh:
        fh.write(glb_bytes(M, color_escala(t, escala or (ESC_DIV if divergente else ESC_SEQ))))
    return ruta, formula

def visor_ra(ruta):
    """<model-viewer> con botón de RA. El botón solo aparece si el dispositivo admite WebXR/Scene Viewer/Quick Look;
    en otro caso el visor es 3D convencional y así se indica."""
    src = 'data:model/gltf-binary;base64,' + base64.b64encode(open(ruta, 'rb').read()).decode()
    pagina = f"""<!doctype html><html><head><meta name='viewport' content='width=device-width,initial-scale=1'>
<script type='module' src='https://unpkg.com/@google/model-viewer@3.5.0/dist/model-viewer.min.js'></script>
<style>body{{margin:0;background:{NV['fondo']};color:{NV['texto']};font-family:sans-serif}}
model-viewer{{width:100%;height:560px;background:radial-gradient(circle at 50% 40%,#08202c,{NV['fondo']} 70%)}}
#ra{{position:absolute;right:14px;bottom:14px;padding:9px 16px;border:0;border-radius:8px;background:{NV['cian']};
color:#001018;font-weight:600}}#est{{font-size:12px;padding:6px 12px;color:{NV['tenue']}}}</style></head><body>
<model-viewer src='{src}' ar ar-modes='webxr scene-viewer quick-look' ar-scale='auto' camera-controls
 touch-action='pan-y' shadow-intensity='0.8' exposure='1.15' camera-orbit='20deg 55deg auto'
 alt='Relieve económico departamental de Colombia'><button slot='ar-button' id='ra'>Ver en realidad aumentada</button>
</model-viewer><div id='est'>Comprobando compatibilidad con realidad aumentada…</div>
<script>const mv=document.querySelector('model-viewer');mv.addEventListener('load',()=>{{
document.getElementById('est').textContent=mv.canActivateAR?
'RA disponible: pulse el botón y apunte a una superficie plana (maqueta de ~1,2 m).':
'Este navegador o dispositivo no admite RA: se muestra un visor 3D convencional. Abra la app en un móvil compatible.';}});
</script></body></html>"""
    return (f"<iframe style='width:100%;height:600px;border:1px solid {NV['borde']};border-radius:12px;"
            f"background:{NV['fondo']}' allow='xr-spatial-tracking; fullscreen' srcdoc=\"{_html.escape(pagina, quote=True)}\"></iframe>")

# =====================================================================================
# 8. Simulador y dinámica (forma reducida ya estimada: Φ, Γ); no re-estima nada
# =====================================================================================
ETQ_CLIMA = {'precip_anom': 'precipitación', 'temp_anom': 'temperatura', 'ndvi_anom': 'NDVI', 'ndvi_crop_anom': 'NDVI agrícola'}
TIPOS_CHOQUE = [(f'Choque {NOMBRES_VAR[v].lower()} (+1 d.e.)', v) for v in ORDEN]
for _s in GSAT:
    TIPOS_CHOQUE.append((f'Choque climático: {ETQ_CLIMA.get(_s, _s)} −1 d.e.', 'sat:' + _s))

def simular(origen_nombre, tipo, magnitud, H_, var_resp):
    i = CODIGOS.index(NOMBRE_A_COD[origen_nombre]); H_ = int(H_); mag = float(magnitud)
    if tipo.startswith('sat:'):
        r0 = -mag * GSAT[tipo.split(':', 1)[1]][:, i]
    else:
        b0 = ORDEN.index(tipo); r0 = GAM[:, b0 * N + i] * PAR[tipo]['sigma'] * mag
    R = irf(PHI, r0, H_) * 100                                   # p.p. de crecimiento
    b = ORDEN.index(var_resp); bloque = R[:, b * N:(b + 1) * N]
    return R, b, bloque, i

def _sim(origen_nombre, tipo_etq, magnitud, H_, var_resp_etq):
    tipo = dict(TIPOS_CHOQUE)[tipo_etq]
    var_resp = {NOMBRES_VAR[v]: v for v in ORDEN}[var_resp_etq]
    return (var_resp, tipo) + simular(origen_nombre, tipo, magnitud, H_, var_resp)

def _escala_sim(bloque, i):
    otros = np.delete(np.abs(bloque), i, axis=1)
    return float(otros.max()) * 1.4 if otros.max() > 0 else float(np.abs(bloque).max() or 1)

def fig_difusion(bloque, i, var_resp, origen_nombre, titulo_choque, h=0):
    h = int(min(max(h, 0), bloque.shape[0] - 1))
    d = pd.DataFrame({'cod_dpto': CODIGOS, 'nombre': [COD_A_NOMBRE[c] for c in CODIGOS], 'resp': bloque[h]})
    m = _escala_sim(bloque, i)
    return fig_mapa(d, 'resp', f'{titulo_choque} en {origen_nombre} → {NOMBRES_VAR[var_resp]} · '
                               f'<b>año {h}</b> tras el choque (p.p.)', ESC_DIV, True, etiqueta='p.p.',
                    rango=(-m, m), alto=600, resaltar=CODIGOS[i])

def relieve_difusion(bloque, i, var_resp, origen_nombre, titulo_choque, h=0, modo='raiz'):
    h = int(min(max(h, 0), bloque.shape[0] - 1))
    m = _escala_sim(bloque, i)
    ref = float(np.abs(bloque).max()) or 1.0          # referencia fija en todos los horizontes
    return fig_relieve(dict(zip(CODIGOS, bloque[h])), f'Relieve 3D · {titulo_choque} en {origen_nombre} → '
                       f'{NOMBRES_VAR[var_resp]} · año {h}', 'p.p.', ESC_DIV, True, modo, ref, (-m, m),
                       resaltar=CODIGOS[i], alto=620)

def fig_irf_sim(R, i):
    fig = make_subplots(rows=1, cols=2, subplot_titles=[NOMBRES_VAR[v] for v in ORDEN])
    for b, v in enumerate(ORDEN):
        bl = R[:, b * N:(b + 1) * N]
        vec = WS[ELECCION[v]['W']]['B'][i] > 0
        resto = ~vec; resto[i] = False
        series = [('Origen', bl[:, i], NV['blanco']), ('Vecinos (promedio)', bl[:, vec].mean(1), NV['cian']),
                  ('Resto del país', bl[:, resto].mean(1), NV['turquesa']), ('Promedio nacional', bl.mean(1), NV['ambar'])]
        for nom, y, colr in series:
            fig.add_trace(go.Scatter(x=list(range(len(y))), y=y, mode='lines+markers', name=nom, legendgroup=nom,
                                     showlegend=(b == 0), line=dict(color=colr, width=2.4),
                                     hovertemplate=nom + ' · h=%{x}: %{y:.3f} p.p.<extra></extra>'), 1, b + 1)
        fig.add_hline(y=0, line_color=NV['borde'], row=1, col=b + 1)
    fig.update_xaxes(title='años tras el choque', gridcolor=NV['borde'])
    fig.update_yaxes(title='p.p. de crecimiento', col=1, gridcolor=NV['borde'])
    return estilo(fig, 'Trayectorias: origen, vecinos y resto del país', 420)

def vista_simulador(origen_nombre, tipo_etq, magnitud, H_, var_resp_etq, h=0, modo3d='raiz'):
    var_resp, tipo, R, b, bloque, i = _sim(origen_nombre, tipo_etq, magnitud, H_, var_resp_etq)
    acum = bloque.sum(0)
    top = (pd.DataFrame({'Departamento': [COD_A_NOMBRE[c] for c in CODIGOS], 'Impacto h=0 (p.p.)': bloque[0],
                         'Acumulado (p.p.)': acum})
           .drop(index=i).sort_values('Acumulado (p.p.)', key=np.abs, ascending=False).head(10).round(4))
    tit = tipo_etq.split(' (')[0]
    rel, formula = relieve_difusion(bloque, i, var_resp, origen_nombre, tit, h, modo3d)
    return (fig_difusion(bloque, i, var_resp, origen_nombre, tit, h), rel, fig_irf_sim(R, i),
            interpretar_simulacion(origen_nombre, tipo, tipo_etq, magnitud, H_, var_resp, bloque, i, formula), top)

def cuadro_sim(origen_nombre, tipo_etq, magnitud, H_, var_resp_etq, h, modo3d='raiz'):
    var_resp, tipo, R, b, bloque, i = _sim(origen_nombre, tipo_etq, magnitud, H_, var_resp_etq)
    tit = tipo_etq.split(' (')[0]
    return fig_difusion(bloque, i, var_resp, origen_nombre, tit, h), relieve_difusion(bloque, i, var_resp,
                                                                                         origen_nombre, tit, h, modo3d)[0]

PAUSA_ANIM = 0.9

def reproducir_sim(origen_nombre, tipo_etq, magnitud, H_, var_resp_etq, modo3d='raiz'):
    var_resp, tipo, R, b, bloque, i = _sim(origen_nombre, tipo_etq, magnitud, H_, var_resp_etq)
    tit = tipo_etq.split(' (')[0]
    for h in range(bloque.shape[0]):
        yield (fig_difusion(bloque, i, var_resp, origen_nombre, tit, h),
               relieve_difusion(bloque, i, var_resp, origen_nombre, tit, h, modo3d)[0], h)
        time.sleep(PAUSA_ANIM)

# ---------- Relieve por indicador y año (dinámica temporal) ----------
def _rango_global(col):
    v = APP_PANEL[col].astype(float)
    return float(v.min()), float(v.max())

def vista_relieve(etiqueta, anio, resaltar_nombre=None):
    d, col, div, esc, tit = datos_variable(etiqueta, anio)
    vals = d.set_index('cod_dpto')[col].astype(float).to_dict()
    cod = NOMBRE_A_COD.get(resaltar_nombre) if resaltar_nombre else None
    if VARS_MAPA[etiqueta][0] == 'panel':            # escala fija en todos los años ⇒ años comparables
        lo, hi = _rango_global(col)
        ref = (lo, hi) if not div else max(abs(lo), abs(hi))
        rc = (-ref, ref) if div else (lo, hi)
    else:
        ref, rc = None, None
    fig, formula = fig_relieve(vals, f'Relieve 3D · {tit}', etiqueta, esc, div, 'lineal' if div else 'nivel', ref, rc, cod)
    return fig, interpretar_relieve(etiqueta, anio, d, col, div, formula)

def reproducir_relieve(etiqueta, resaltar_nombre=None):
    if VARS_MAPA[etiqueta][0] != 'panel':
        yield vista_relieve(etiqueta, ANIOS[-1], resaltar_nombre) + (ANIOS[-1],)
        return
    for a in ANIOS:
        yield vista_relieve(etiqueta, a, resaltar_nombre) + (a,)
        time.sleep(PAUSA_ANIM)

# =====================================================================================
# 9. Modelo SpVAR: selección, mecanismos, IRF, coeficientes
# =====================================================================================
def fig_seleccion(k_etq):
    k = {NOMBRES_VAR[v]: v for v in FINAL}[k_etq]
    c = ELECCION[k]['criterio']
    m = TAB_AJ[TAB_AJ.ecuacion == k].pivot(index='W', columns='modelo', values=c)[CONFIG['MODELOS']]
    dlt = (m - m.min().min()).round(1)
    txt = [[('★ ' if (w == ELECCION[k]['W'] and mo == ELECCION[k]['modelo']) else '') + f'{dlt.loc[w, mo]:.1f}'
            for mo in dlt.columns] for w in dlt.index]
    fig = go.Figure(go.Heatmap(z=dlt.values, x=list(dlt.columns), y=list(dlt.index), text=txt,
                               texttemplate='%{text}',
                               colorscale=[[0, '#e6fffd'], [0.15, '#22d3ee'], [0.5, '#0b4f63'], [1, '#02070c']],
                               colorbar=dict(title=f'Δ{c}'), hovertemplate='W=%{y} · %{x}: Δ=%{z}<extra></extra>'))
    return estilo(fig, f'Selección conjunta W × modelo · {k_etq} (★ elegido: '
                       f"{ELECCION[k]['modelo']} con W={ELECCION[k]['W']})", 430)

def fig_mecanismos(k_etq):
    k = {NOMBRES_VAR[v]: v for v in FINAL}[k_etq]
    m = MECANISMOS[MECANISMOS.ecuacion == k].iloc[::-1]
    paleta = [NV['cian'], NV['turquesa'], NV['ambar'], '#a78bfa', NV['neg']]
    fig = go.Figure()
    for j, mec in enumerate(m.mecanismo.unique()):
        s = m[m.mecanismo == mec]
        fig.add_trace(go.Scatter(x=s.estimacion, y=s.termino, mode='markers', name=mec,
                                 error_x=dict(type='data', array=1.96 * s.se.fillna(0), thickness=1.2,
                                              color=paleta[j % 5]),
                                 marker=dict(size=10, color=paleta[j % 5], line=dict(width=0.5, color=NV['blanco'])),
                                 customdata=list(zip(s.p.round(4), s.sig)),
                                 hovertemplate='%{y}<br>%{x:.4f} (p=%{customdata[0]:.3f}) %{customdata[1]}<extra></extra>'))
    fig.add_vline(x=0, line_color=NV['tenue'])
    fig.update_yaxes(tickfont=dict(size=10), gridcolor=NV['borde']); fig.update_xaxes(gridcolor=NV['borde'])
    return estilo(fig, f'Mecanismos de transmisión ± IC95% · {k_etq} ({FINAL[k].modelo})', 620)

def fig_irf_agregada(choque):
    fig = make_subplots(rows=1, cols=2, subplot_titles=[NOMBRES_VAR[v] for v in ORDEN])
    for b, v in enumerate(ORDEN):
        s = TAB_IRF[(TAB_IRF.choque == choque) & (TAB_IRF.respuesta == v)]
        fig.add_trace(go.Scatter(x=list(s.h) + list(s.h[::-1]), y=list(s.p95 * 100) + list(s.p05[::-1] * 100),
                                 fill='toself', fillcolor='rgba(34,211,238,0.16)', line=dict(width=0),
                                 name='Banda 90%', showlegend=(b == 0), hoverinfo='skip'), 1, b + 1)
        fig.add_trace(go.Scatter(x=s.h, y=s.irf * 100, mode='lines+markers', name='IRF', showlegend=(b == 0),
                                 line=dict(color=NV['cian'], width=2.5)), 1, b + 1)
        fig.add_hline(y=0, line_color=NV['borde'], row=1, col=b + 1)
    fig.update_xaxes(title='años', gridcolor=NV['borde']); fig.update_yaxes(title='p.p.', col=1, gridcolor=NV['borde'])
    return estilo(fig, f'Respuesta impulso agregada · {choque}', 420)

def tabla_coef(k_etq):
    k = {NOMBRES_VAR[v]: v for v in FINAL}[k_etq]
    t = FINAL[k].tabla()
    t = t[~t.index.str.match(r'^(T_|D_)')].round(4)
    return t.reset_index(names='parámetro')

def vista_spvar(k_etq):
    return fig_seleccion(k_etq), fig_mecanismos(k_etq), tabla_coef(k_etq), interpretar_ecuacion(k_etq)

def _imp(k, x, efecto):
    s = IMPACTOS[k][(IMPACTOS[k].regresor == x) & (IMPACTOS[k].efecto == efecto)]
    return (float(s.estimacion.iloc[0]), float(s.p.iloc[0])) if len(s) else (np.nan, np.nan)

def html_kpis():
    d_dir, p_dir = _imp(dn, x_cruce, 'Directo')
    d_ind, p_ind = _imp(dn, x_cruce, 'Indirecto')
    tarjetas = [
        ('Estabilidad del sistema', f'{RADIO_ESPECTRAL:.2f}', 'radio espectral de Φ (< 1 = estable)'),
        (f'ρ {NOMBRES_VAR[up]}', f"{FINAL[up].rho:+.2f}", f'{FINAL[up].modelo} · W={FINAL[up].nombre_W}'),
        (f'ρ {NOMBRES_VAR[dn]}', f"{FINAL[dn].rho:+.2f}", f'{FINAL[dn].modelo} · W={FINAL[dn].nombre_W}'),
        (f'Efecto local {x_cruce} → {dn}', f'{d_dir:+.2f}', f'impacto directo (p={p_dir:.3f})'),
        ('Spillover a vecinos', f'{d_ind:+.2f}', f'impacto indirecto (p={p_ind:.3f})'),
        ('Multiplicador espacial', f'{1 / (1 - FINAL[dn].rho):.2f}×', f'1/(1−ρ) en {NOMBRES_VAR[dn].lower()}'),
    ]
    cards = ''.join(f"<div class='kpi'><div class='l'>{l}</div><div class='v'>{v}</div><div class='d'>{dd}</div></div>"
                    for l, v, dd in tarjetas)
    return f"<div class='kpi-grid'>{cards}</div>"

# =====================================================================================
# 10. Interpretabilidad automatizada (solo a partir de resultados calculados)
# =====================================================================================
ETQ = {'D': '`DESCRIPTIVO`', 'A': '`ASOCIACIÓN CONDICIONAL`', 'R': '`RESPUESTA HIPOTÉTICA DEL MODELO`',
       'I': '`IDENTIFICACIÓN`'}
AVISO_DEMO_MD = ('> ⚠️ **Datos sintéticos de demostración** (`MODO_DATOS="demo"`): las cifras ilustran el método y '
                 '**no describen la economía colombiana**. Cargue el panel DANE/EAM para conclusiones reales.\n\n'
                 if ES_DEMO else '')
LEYENDA_EVIDENCIA = (f"Tipos de evidencia: {ETQ['D']} datos observados · {ETQ['A']} coeficientes condicionados a la "
                     f"especificación y a W · {ETQ['R']} simulación ante un choque supuesto · {ETQ['I']} supuestos y límites.")

def _sig(p):
    return 'sin inferencia' if not np.isfinite(p) else (f'significativo, p={p:.3f}' if p < CONFIG['ALFA']
                                                        else f'no significativo, p={p:.3f}')

def _w_de(etiqueta):
    return WS[ELECCION[VARS_MAPA[etiqueta][4]]['W']]['W'], ELECCION[VARS_MAPA[etiqueta][4]]['W']

def interpretar_mapa(etiqueta, anio):
    d, col, div, _, _ = datos_variable(etiqueta, anio)
    s = d.set_index('nombre')[col].astype(float)
    ok = s.dropna()
    if ok.empty:
        return AVISO_DEMO_MD + 'Sin datos para esta selección.'
    lin = [AVISO_DEMO_MD + f'#### {etiqueta} · {int(anio) if VARS_MAPA[etiqueta][0] == "panel" else "SpVAR"}']
    lin.append(f"- {ETQ['D']} {len(ok)}/{len(s)} departamentos con dato; media {ok.mean():.4g}, mediana "
               f"{ok.median():.4g}, desviación {ok.std():.4g}.")
    alto, bajo = ok.sort_values(ascending=False).head(3), ok.sort_values().head(3)
    lin.append(f"- {ETQ['D']} Valores más altos: " + ', '.join(f'{n} ({v:.3g})' for n, v in alto.items())
               + '; más bajos: ' + ', '.join(f'{n} ({v:.3g})' for n, v in bajo.items()) + '.')
    if div:
        lin.append(f"- {ETQ['D']} {int((ok > 0).sum())} departamentos con valor positivo y {int((ok < 0).sum())} negativo.")
    if len(s) - len(ok):
        lin.append(f"- {ETQ['D']} {len(s) - len(ok)} departamentos sin dato se muestran sin relleno (no equivalen a cero).")
    if VARS_MAPA[etiqueta][0] == 'panel' and len(ok) == len(s) and ok.std() > 0:
        Wm, wn = _w_de(etiqueta)
        r = moran_geary(d[col].values.astype(float), Wm, 499)
        lin.append(f"- {ETQ['D']} I de Moran = {r['I']:.3f} (E[I] = {r['E[I]']:.3f}, p = {r['p_I']:.3f}, W = {wn}): "
                   f"{interpretar_moran(r).lower()}.")
        if anio != ANIOS[0]:
            prev = APP_PANEL[APP_PANEL.anio == int(anio) - 1].set_index('cod_dpto')[col].reindex(CODIGOS)
            act = d.set_index('cod_dpto')[col]
            dif = (act - prev).dropna()
            if len(dif):
                lin.append(f"- {ETQ['D']} Frente a {int(anio) - 1}: el valor sube en {int((dif > 0).sum())} y baja en "
                           f"{int((dif < 0).sum())} departamentos (cambio medio {dif.mean():+.3g}).")
    else:
        lin.append(f"- {ETQ['A']} Es un resultado del modelo: depende de la especificación y de W "
                   f"({FINAL[VARS_MAPA[etiqueta][4]].modelo}, W = {FINAL[VARS_MAPA[etiqueta][4]].nombre_W}).")
    return '\n'.join(lin)

def interpretar_esda(etiqueta, anio, wn, r, d):
    cnt = d['cluster'].value_counts()
    nombres = lambda cat: ', '.join(d.loc[d.cluster == cat, 'nombre'].head(6)) or '—'
    return (AVISO_DEMO_MD + f"- {ETQ['D']} {etiqueta} ({int(anio)}, W = {wn}): I = {r['I']:.3f} (p = {r['p_I']:.3f}), "
            f"Geary C = {r['C']:.3f} (p = {r['p_C']:.3f}) → {interpretar_moran(r).lower()}.\n"
            f"- {ETQ['D']} Clusters LISA: Alto-Alto {cnt.get('Alto-Alto', 0)} ({nombres('Alto-Alto')}); "
            f"Bajo-Bajo {cnt.get('Bajo-Bajo', 0)} ({nombres('Bajo-Bajo')}); atípicos Alto-Bajo/Bajo-Alto "
            f"{cnt.get('Alto-Bajo', 0) + cnt.get('Bajo-Alto', 0)}.\n"
            f"- {ETQ['I']} La significancia LISA no está corregida por comparaciones múltiples; con N = {N} "
            "los clusters individuales deben leerse como exploratorios.")

def interpretar_relieve(etiqueta, anio, d, col, div, formula):
    return (interpretar_mapa(etiqueta, anio) +
            f"\n- Altura: `{formula}` (transformación explícita del valor real; escala fija entre años para que sean "
            "comparables). Color = mismo valor. Geometría: " + FUENTE_GEOM + '.')

def interpretar_simulacion(origen, tipo, tipo_etq, magnitud, H_, var_resp, bloque, i, formula):
    acum = bloque.sum(0)
    tot = np.abs(acum).sum()
    fuera = 100 * (tot - abs(acum[i])) / max(tot, 1e-12)
    vec = WS[ELECCION[var_resp]['W']]['B'][i] > 0
    pico = int(np.argmax(np.abs(bloque[:, i])))
    otros = np.delete(np.arange(N), i)
    j = otros[np.argmax(np.abs(acum[otros]))]
    medio = None
    if abs(bloque[0, i]) > 0:
        bajo = np.where(np.abs(bloque[:, i]) <= 0.5 * np.abs(bloque[:, i]).max())[0]
        bajo = bajo[bajo > pico]
        medio = int(bajo[0]) if len(bajo) else None
    ident = ('choque estructural bajo el orden recursivo '
             f"{NOMBRES_VAR[ORDEN[0]]} → {NOMBRES_VAR[ORDEN[1]]} (supuesto de identificación)"
             if not tipo.startswith('sat:') else 'cambio exógeno de la covariable climática (supuesto de exogeneidad del clima)')
    lin = [AVISO_DEMO_MD + f"#### {tipo_etq} × {float(magnitud):g} en {origen} → {NOMBRES_VAR[var_resp]}",
           f"- {ETQ['R']} En el origen: {bloque[0, i]:+.3f} p.p. al impacto, máximo en el año {pico}, "
           f"{acum[i]:+.3f} p.p. acumulados en {int(H_)} años"
           + (f"; la respuesta cae a la mitad del máximo hacia el año {medio}." if medio is not None else '.'),
           f"- {ETQ['R']} Vecinos directos ({int(vec.sum())}): {acum[vec].mean():+.3f} p.p. acumulados en promedio; "
           f"resto del país: {np.delete(acum, np.r_[np.where(vec)[0], i]).mean():+.3f} p.p.",
           f"- {ETQ['R']} {fuera:.0f}% de la respuesta acumulada ocurre fuera de {origen}; el más afectado es "
           f"{COD_A_NOMBRE[CODIGOS[j]]} ({acum[j]:+.3f} p.p.).",
           f"- {ETQ['I']} Tipo de choque: {ident}. Es una respuesta hipotética del modelo estimado, no una predicción; "
           "no incluye bandas de incertidumbre por departamento.",
           f"- Relieve 3D: `{formula}` sobre la respuesta en p.p. (la raíz con signo hace visibles los efectos pequeños "
           "de los vecinos sin ocultar el signo)."]
    return '\n'.join(lin)

def interpretar_ecuacion(k_etq):
    k = {NOMBRES_VAR[v]: v for v in FINAL}[k_etq]
    r, e = FINAL[k], ELECCION[k]
    t = r.tabla()
    lin = [AVISO_DEMO_MD + f"#### Ecuación {k_etq}: {r.modelo} con W = {r.nombre_W}",
           f"- {ETQ['A']} Selección por {e['criterio']} (peso de evidencia {e['peso']:.2f}); el árbol de Elhorst en esa "
           f"W indica {e['arbol'][0]}" + ('' if e['coincide_IC_arbol'] else ' — **discrepa** del criterio de información')
           + '.']
    nombres = {f'{k}_L1': 'persistencia propia (t−1)', 'rho': 'dependencia espacial contemporánea ρ',
               'lambda': 'dependencia espacial del error λ'}
    for p, row in t[~t.index.str.match(r'^(T_|D_|const|sigma2)')].iterrows():
        desc = nombres.get(p, p.replace('W_', 'W·').replace('_L1', ' (t−1)').replace('_t', ' (t)'))
        lin.append(f"- {ETQ['A']} {desc}: {row.coef:+.4f} ({_sig(row.p)}).")
    if r.rho:
        lin.append(f"- {ETQ['A']} Multiplicador espacial medio 1/(1−ρ) = {1 / (1 - r.rho):.2f}. ρ y θ no son efectos "
                   "marginales: use los impactos directos e indirectos (pestaña Modelo → Mecanismos).")
    if CONFIG['ORDEN'].index(k) == 1:
        lin.append(f"- {ETQ['I']} Esta ecuación incluye {ORDEN[0]}_t contemporáneo por el orden recursivo "
                   f"{ORDEN[0]} → {ORDEN[1]}; su lectura causal depende de ese supuesto.")
    if not r.vcov_ok:
        lin.append(f"- {ETQ['I']} La matriz de covarianzas no es definida positiva: los p-valores no son fiables.")
    return '\n'.join(lin)

def interpretar_panorama():
    lin = [AVISO_DEMO_MD + '### Panorama del sistema', LEYENDA_EVIDENCIA, '',
           f"- {ETQ['D']} Panel de estimación: N = {N} departamentos, T = {T} años ({ANIOS[0]}–{ANIOS[-1]}), "
           f"transformación {TRANSF}; geometría {FUENTE_GEOM}.",
           f"- {ETQ['A']} Radio espectral de Φ = {RADIO_ESPECTRAL:.3f}: "
           + ('el sistema es estable y los choques se disipan.' if RADIO_ESPECTRAL < 1 else
              '**no estable**; las respuestas dinámicas no deben interpretarse.')]
    for k in FINAL:
        lin.append(f"- {ETQ['A']} {NOMBRES_VAR[k]}: {FINAL[k].modelo} con W = {FINAL[k].nombre_W}, ρ = {FINAL[k].rho:+.3f}.")
    sig = MECANISMOS[(MECANISMOS.p < CONFIG['ALFA']) & (MECANISMOS.tipo == 'impacto')]
    sig = sig.reindex(sig.estimacion.abs().sort_values(ascending=False).index).head(5)
    for f in sig.itertuples():
        lin.append(f"- {ETQ['A']} {NOMBRES_VAR[f.ecuacion]} · {f.mecanismo}: {f.termino} = {f.estimacion:+.3f} ({_sig(f.p)}).")
    for n in IRF_PUNTO:
        s = TAB_IRF[(TAB_IRF.choque == n) & (TAB_IRF.respuesta == ORDEN[1])]
        if len(s):
            excl = (s.p05.iloc[-1] > 0) or (s.p95.iloc[-1] < 0)
            lin.append(f"- {ETQ['R']} Choque {n} → {NOMBRES_VAR[ORDEN[1]]}: acumulado a {H} años "
                       f"{s.acumulada.iloc[-1] * 100:+.2f} p.p.; en h = {H} la banda 90% "
                       f"{'excluye' if excl else 'incluye'} el cero.")
    if PROCEDENCIA.get('fuente'):
        lin.insert(3, f"- {ETQ['D']} Fuente: {PROCEDENCIA['fuente']}."
                   + (f" Años provisionales/preliminares: {PROCEDENCIA['anios_preliminares']}." if PROCEDENCIA.get('anios_preliminares') else ''))
    if CONFIG['MODO_DATOS'] == 'dane':
        lin.append(f"- {ETQ['I']} {NOMBRES_VAR['ind']} es el total manufacturero del departamento: incluye la agroindustria "
                   "pero también otras ramas; los efectos agro → manufactura no son exclusivos de la industria de alimentos.")
    if PROCEDENCIA.get('clima'):
        lin.append(f"- {ETQ['D']} Clima: {PROCEDENCIA['clima']['fuente']}; {PROCEDENCIA['clima']['agregacion']}.")
    lin.append(f"- {ETQ['I']} Identificación recursiva {ORDEN[0]} → {ORDEN[1]}: supone que {NOMBRES_VAR[ORDEN[1]].lower()} "
               f"no afecta a {NOMBRES_VAR[ORDEN[0]].lower()} dentro del mismo año. Ninguna relación es causal fuera de ese supuesto.")
    if 'AUD' in globals():
        c = AUD.estado.value_counts().to_dict()
        lin.append(f"- {ETQ['I']} Auditoría: {c}.")
    return '\n'.join(lin)

SECCIONES_INTERP = ['Panorama'] + [f'Ecuación {NOMBRES_VAR[k]}' for k in FINAL] + ['Informe completo']

def vista_interpretacion(seccion):
    if seccion == 'Panorama':
        return interpretar_panorama()
    if seccion == 'Informe completo':
        return AVISO_DEMO_MD + INFORME
    return interpretar_ecuacion(seccion.replace('Ecuación ', ''))

# =====================================================================================
# 11. Vistas de mapas (con interpretación sincronizada)
# =====================================================================================
def vista_mapas(etiqueta, anio, paleta, resaltar_nombre=None):
    d, col, div, esc, tit = datos_variable(etiqueta, anio)
    esc = esc if paleta == 'Auto' else ESCALAS[paleta]
    cod = NOMBRE_A_COD.get(resaltar_nombre) if resaltar_nombre else None
    return (fig_mapa(d, col, tit, esc, div, etiqueta=etiqueta, resaltar=cod), fig_ranking(d, col, etiqueta, div),
            interpretar_mapa(etiqueta, anio))

def reproducir_mapas(etiqueta, paleta, resaltar_nombre=None):
    for a in ANIOS:
        yield vista_mapas(etiqueta, a, paleta, resaltar_nombre) + (a,)
        time.sleep(PAUSA_ANIM)

def vista_ra(fuente, etiqueta, anio, origen, tipo_etq, magnitud, var_resp_etq, h):
    if fuente == 'Indicador por año':
        d, col, div, esc, tit = datos_variable(etiqueta, anio)
        vals = d.set_index('cod_dpto')[col].astype(float).to_dict()
        ruta, formula = exportar_glb(vals, f'ra_{col}_{int(anio)}', esc, div, 'lineal' if div else 'nivel')
        desc = f'{tit}'
    else:
        var_resp, tipo, R, b, bloque, i = _sim(origen, tipo_etq, magnitud, CONFIG['HORIZONTE_IRF'], var_resp_etq)
        hh = int(min(h, bloque.shape[0] - 1))
        ruta, formula = exportar_glb(dict(zip(CODIGOS, bloque[hh])), f'ra_choque_{CODIGOS[i]}_h{hh}', ESC_DIV, True,
                                     'raiz', float(np.abs(bloque).max()))
        desc = f'{tipo_etq} en {origen} → {var_resp_etq}, año {hh}'
    if ruta is None:
        return None, '<p>Sin polígonos: no se puede generar el modelo 3D.</p>', None, ''
    nota = (AVISO_DEMO_MD + f"**Modelo exportado:** {desc}. Altura: `{formula}`; maqueta de 1,2 m de ancho, "
            f"0,18 m por unidad de altura. Geometría: {FUENTE_GEOM}. El archivo GLB puede abrirse también en "
            "visores externos (p. ej. Scene Viewer en Android).")
    return ruta, visor_ra(ruta), ruta, nota

AVISO_DEMO = ("<div class='aviso'>⚠️ <b>Datos de demostración sintéticos.</b> Los resultados ilustran el método y no "
              "describen la economía colombiana; para conclusiones reales cargue el panel DANE/EAM (Celda 2: "
              "MODO_DATOS='csv').</div>" if ES_DEMO else '')
print('Datos y figuras de la app listos.', '| Polígonos:', GEOJ is not None, '| Geometría:', FUENTE_GEOM,
      '| Relieve 3D:', len(TOPO), 'departamentos', '| Triangulación:', 'earcut' if _earcut else 'recorte de orejas')

"""## Celda 19 · App Gradio: lanzar
Abre la app dentro de Colab y genera un enlace público temporal (`share=True`). Para probar la realidad aumentada, abra ese enlace desde un móvil compatible (Android con ARCore o iOS con Quick Look).
"""

CSS_APP = """
.gradio-container {background: radial-gradient(1000px 520px at 12% -10%, #06263a 0%, transparent 60%),
  radial-gradient(900px 480px at 100% 0%, #031a1c 0%, transparent 55%),
  linear-gradient(170deg, #01050a 0%, #02070d 50%, #030a12 100%) !important; color: #cfefff;}
.hero h1 {font-size: 2.0rem !important; letter-spacing: .01em; color: #e9fdff !important; margin-bottom: 0 !important;
  text-shadow: 0 0 14px rgba(34,211,238,.55);}
.hero p {color: #7fb2c4 !important; font-size: 1rem;}
.kpi-grid {display: grid; grid-template-columns: repeat(auto-fit, minmax(165px, 1fr)); gap: 10px; margin: 8px 0 4px;}
.kpi {background: rgba(6,16,26,.75); border: 1px solid rgba(34,211,238,.25); border-radius: 12px; padding: 12px 14px;
      box-shadow: inset 0 0 18px rgba(34,211,238,.06);}
.kpi .l {font-size: .72rem; color: #6f94a8; text-transform: uppercase; letter-spacing: .06em;}
.kpi .v {font-size: 1.65rem; font-weight: 700; color: #7af7f0; margin: 2px 0; text-shadow: 0 0 10px rgba(45,212,191,.45);}
.kpi .d {font-size: .74rem; color: #5d7f90;}
.aviso {background: rgba(251,191,36,.08); border: 1px solid rgba(251,191,36,.45); color: #fcd88a;
        border-radius: 10px; padding: 10px 14px; margin: 6px 0;}
.nv {position: relative; border-radius: 12px; overflow: hidden; border: 1px solid #0f2a3d;}
.nv::after {content: ''; position: absolute; inset: 0; pointer-events: none;
  background: repeating-linear-gradient(0deg, rgba(34,211,238,.035) 0px, rgba(34,211,238,.035) 1px, transparent 1px, transparent 3px),
              radial-gradient(ellipse at center, transparent 62%, rgba(0,0,0,.45) 100%);}
.interp {background: rgba(6,16,26,.6); border-left: 3px solid #22d3ee; border-radius: 8px; padding: 6px 14px;}
.interp code {color: #7af7f0; background: rgba(34,211,238,.08);}
footer {display: none !important;}
"""
JS_OSCURO = "() => { document.body.classList.add('dark'); }"
TEMA_APP = gr.themes.Base(primary_hue=gr.themes.colors.cyan, secondary_hue=gr.themes.colors.teal,
                          neutral_hue=gr.themes.colors.slate,
                          font=[gr.themes.GoogleFont('IBM Plex Sans'), 'system-ui', 'sans-serif'])
try:
    TEMA_APP = TEMA_APP.set(body_background_fill_dark=NV['fondo'], block_background_fill_dark=NV['panel'],
                            block_border_color_dark=NV['borde'], body_text_color_dark=NV['texto'],
                            input_background_fill_dark='#081624', button_primary_background_fill_dark=NV['cian'],
                            button_primary_text_color_dark='#001018', block_label_text_color_dark=NV['tenue'])
except Exception as e:                                   # versiones de Gradio con otras variables de tema
    print('Tema: se usan los valores por defecto ->', e)

# Compatibilidad Gradio 4/5 (theme/css/js en Blocks) y Gradio 6 (en launch)
try:
    _pb = inspect.signature(gr.Blocks.__init__).parameters
    _pl = inspect.signature(gr.Blocks.launch).parameters
except (TypeError, ValueError):
    _pb, _pl = {'theme': 1, 'css': 1, 'js': 1}, {}
_estilo = dict(theme=TEMA_APP, css=CSS_APP, js=JS_OSCURO)
KW_BLOCKS = {k: v for k, v in _estilo.items() if k in _pb}
KW_LAUNCH = {k: v for k, v in _estilo.items() if k not in _pb and k in _pl}
if 'allowed_paths' in _pl:
    KW_LAUNCH['allowed_paths'] = [CONFIG['OUT_DIR']]

def al_cambiar(comp, fn, entradas, salidas):
    """Los sliders se actualizan al soltar (evita saturar la cola); el resto, al cambiar."""
    ev = getattr(comp, 'release', None) if type(comp).__name__ == 'Slider' else None
    (ev or comp.change)(fn, entradas, salidas)

def interp(valor=''):
    return gr.Markdown(valor, elem_classes=['interp'])

def grafico(valor=None, etiqueta=''):
    return gr.Plot(value=valor, label=etiqueta, elem_classes=['nv'])

DEPTOS = list(DPT.nombre)
ORIGEN0 = COD_A_NOMBRE.get(CONFIG['DPTO_CHOQUE'], DEPTOS[0])
HMAX = CONFIG['HORIZONTE_IRF']
ARGS0 = (ORIGEN0, TIPOS_CHOQUE[0][0], 1.0, HMAX, NOMBRES_VAR[dn])       # escenario de portada
VAR0 = VARS_PANEL[0]
ECUS = [NOMBRES_VAR[v] for v in FINAL]
MODOS_3D = {'Raíz con signo (realza efectos pequeños)': 'raiz', 'Lineal': 'lineal'}
ARCHIVOS = [os.path.join(CONFIG['OUT_DIR'], f) for f in
            ['resultados_spvar.xlsx', 'informe_spvar.md', 'auditoria.csv', 'mapa_interactivo_spvar.html',
             '04_mapas_resultados.png', '05_difusion_choque_local.png', 'panel_dane.csv']]
ARCHIVOS = [f for f in ARCHIVOS if os.path.exists(f)]

with gr.Blocks(title=f'SpVAR Colombia · Agro ↔ {NOMBRES_VAR["ind"]}', **KW_BLOCKS) as APP:
    gr.HTML(f"""<div class='hero'><h1>SpVAR Colombia · Agro ↔ {NOMBRES_VAR['ind']}</h1>
        <p>Transmisión dinámica y espacial entre la actividad agropecuaria y la industria de alimentos ·
        {N} departamentos · {ANIOS[0]}–{ANIOS[-1]} · geometría: {FUENTE_GEOM}<br>
        Datos: {PROCEDENCIA.get('fuente', '')}{' · clima: NASA POWER' if PROCEDENCIA.get('clima') else ''}</p></div>""")
    gr.HTML(AVISO_DEMO)
    gr.HTML(html_kpis())

    with gr.Tabs():
        # ---- 1. Inicio ----
        with gr.Tab('Inicio'):
            i0 = vista_simulador(*ARGS0, 0)
            with gr.Row():
                with gr.Column(scale=1):
                    i_mapa = grafico(i0[0], 'Difusión del choque (mapa)')
                with gr.Column(scale=1):
                    i_rel = grafico(i0[1], 'Difusión del choque (relieve 3D)')
            with gr.Row():
                i_btn = gr.Button('▶ Reproducir la difusión', variant='primary', scale=1)
                i_h = gr.Slider(0, HMAX, value=0, step=1, label='Año tras el choque', scale=3)
            interp(interpretar_panorama())
            def _portada():
                yield from reproducir_sim(*ARGS0)
            i_btn.click(_portada, None, [i_mapa, i_rel, i_h])
            al_cambiar(i_h, lambda h: cuadro_sim(*ARGS0, h), i_h, [i_mapa, i_rel])

        # ---- 2. Mapas coropléticos de visión nocturna ----
        with gr.Tab('Mapas nocturnos'):
            with gr.Row():
                m_var = gr.Dropdown(list(VARS_MAPA), value=VAR0, label='Variable')
                m_anio = gr.Slider(ANIOS[0], ANIOS[-1], value=ANIOS[-1], step=1, label='Año')
                m_pal = gr.Dropdown(PALETAS, value='Auto', label='Escala cromática')
                m_dep = gr.Dropdown(['(ninguno)'] + DEPTOS, value='(ninguno)', label='Resaltar departamento')
            m0 = vista_mapas(VAR0, ANIOS[-1], 'Auto')
            with gr.Row():
                with gr.Column(scale=3):
                    m_mapa = grafico(m0[0], 'Mapa coroplético')
                    m_btn = gr.Button(f'▶ Recorrer {ANIOS[0]}–{ANIOS[-1]}', variant='primary')
                with gr.Column(scale=2):
                    m_rank = grafico(m0[1], 'Ranking')
            m_txt = interp(m0[2])
            ent_m = [m_var, m_anio, m_pal, m_dep]
            for comp in ent_m:
                al_cambiar(comp, vista_mapas, ent_m, [m_mapa, m_rank, m_txt])
            m_btn.click(reproducir_mapas, [m_var, m_pal, m_dep], [m_mapa, m_rank, m_txt, m_anio])

        # ---- 3. Relieve 3D sobre los polígonos ----
        with gr.Tab('Relieve 3D'):
            gr.Markdown('Cada departamento es su **polígono real extruido**: la altura y el color son una transformación '
                        'explícita del valor (fórmula debajo). Arrastre para girar, rueda para acercar.')
            with gr.Row():
                r_var = gr.Dropdown(list(VARS_MAPA), value=VAR0, label='Variable')
                r_anio = gr.Slider(ANIOS[0], ANIOS[-1], value=ANIOS[-1], step=1, label='Año')
                r_dep = gr.Dropdown(['(ninguno)'] + DEPTOS, value='(ninguno)', label='Resaltar departamento')
            r0 = vista_relieve(VAR0, ANIOS[-1])
            r_fig = grafico(r0[0], 'Relieve económico 3D')
            r_btn = gr.Button(f'▶ Dinámica temporal {ANIOS[0]}–{ANIOS[-1]}', variant='primary')
            r_txt = interp(r0[1])
            for comp in [r_var, r_anio, r_dep]:
                al_cambiar(comp, vista_relieve, [r_var, r_anio, r_dep], [r_fig, r_txt])
            r_btn.click(reproducir_relieve, [r_var, r_dep], [r_fig, r_txt, r_anio])

        # ---- 4. Simulador ----
        with gr.Tab('Simulador de perturbaciones'):
            with gr.Row():
                s_origen = gr.Dropdown(DEPTOS, value=ORIGEN0, label='Departamento de origen')
                s_tipo = gr.Dropdown([t for t, _ in TIPOS_CHOQUE], value=TIPOS_CHOQUE[0][0], label='Tipo de choque')
                s_resp = gr.Radio([NOMBRES_VAR[v] for v in ORDEN], value=NOMBRES_VAR[dn], label='Respuesta de')
            with gr.Row():
                s_mag = gr.Slider(-3, 3, value=1, step=0.25, label='Magnitud (múltiplos de 1 d.e.)')
                s_hz = gr.Slider(2, 12, value=HMAX, step=1, label='Horizonte (años)')
                s_m3 = gr.Radio(list(MODOS_3D), value=list(MODOS_3D)[0], label='Altura del relieve')
            s_out = vista_simulador(*ARGS0, 0)
            with gr.Row():
                with gr.Column(scale=1):
                    s_mapa = grafico(s_out[0], 'Mapa de difusión')
                with gr.Column(scale=1):
                    s_rel = grafico(s_out[1], 'Relieve 3D de la respuesta')
            with gr.Row():
                s_btn = gr.Button('▶ Reproducir', variant='primary', scale=1)
                s_ver = gr.Slider(0, 12, value=0, step=1, label='Año tras el choque (avanzar / retroceder)', scale=3)
            with gr.Row():
                with gr.Column(scale=3):
                    s_irf = grafico(s_out[2], 'Trayectorias')
                with gr.Column(scale=2):
                    s_top = gr.Dataframe(value=s_out[4], label='10 departamentos más afectados (fuera del origen)')
            s_txt = interp(s_out[3])
            ent_s = [s_origen, s_tipo, s_mag, s_hz, s_resp]

            def _vs(o, t, m, hz, r, h, m3):
                return vista_simulador(o, t, m, hz, r, h, MODOS_3D[m3])

            def _cs(o, t, m, hz, r, h, m3):
                return cuadro_sim(o, t, m, hz, r, h, MODOS_3D[m3])

            def _rs(o, t, m, hz, r, m3):
                yield from reproducir_sim(o, t, m, hz, r, MODOS_3D[m3])

            for comp in ent_s + [s_m3]:
                al_cambiar(comp, _vs, ent_s + [s_ver, s_m3], [s_mapa, s_rel, s_irf, s_txt, s_top])
            al_cambiar(s_ver, _cs, ent_s + [s_ver, s_m3], [s_mapa, s_rel])
            s_btn.click(_rs, ent_s + [s_m3], [s_mapa, s_rel, s_ver])

        # ---- 5. ESDA ----
        with gr.Tab('ESDA'):
            with gr.Row():
                e_var = gr.Dropdown(VARS_PANEL, value=VAR0, label='Variable')
                e_anio = gr.Slider(ANIOS[0], ANIOS[-1], value=ANIOS[-1], step=1, label='Año')
                e_w = gr.Dropdown(list(WS), value=ELECCION[up]['W'], label='Matriz de pesos W')
            e0 = vista_esda(VAR0, ANIOS[-1], ELECCION[up]['W'])
            with gr.Row():
                e_moran = grafico(e0[0], 'Diagrama de Moran')
                e_lisa = grafico(e0[1], 'Clusters LISA')
            e_txt = interp(e0[3])
            e_tab = gr.Dataframe(value=e0[2], label='Autocorrelación espacial global (permutaciones)')
            for comp in [e_var, e_anio, e_w]:
                al_cambiar(comp, vista_esda, [e_var, e_anio, e_w], [e_moran, e_lisa, e_tab, e_txt])

        # ---- 6. Realidad aumentada ----
        with gr.Tab('Realidad aumentada'):
            gr.Markdown('Genera una **maqueta 3D (GLB)** con los polígonos extruidos. En móviles compatibles '
                        '(WebXR / Scene Viewer / Quick Look) aparece el botón *Ver en realidad aumentada*; en los demás '
                        'dispositivos el visor es 3D convencional y así lo indica.')
            with gr.Row():
                a_fuente = gr.Radio(['Indicador por año', 'Respuesta a un choque'], value='Indicador por año',
                                    label='Contenido de la maqueta')
                a_var = gr.Dropdown(list(VARS_MAPA), value=VAR0, label='Variable')
                a_anio = gr.Slider(ANIOS[0], ANIOS[-1], value=ANIOS[-1], step=1, label='Año')
            with gr.Row():
                a_origen = gr.Dropdown(DEPTOS, value=ORIGEN0, label='Origen del choque')
                a_tipo = gr.Dropdown([t for t, _ in TIPOS_CHOQUE], value=TIPOS_CHOQUE[0][0], label='Tipo de choque')
                a_mag = gr.Slider(-3, 3, value=1, step=0.25, label='Magnitud (d.e.)')
                a_resp = gr.Radio([NOMBRES_VAR[v] for v in ORDEN], value=NOMBRES_VAR[dn], label='Respuesta de')
                a_h = gr.Slider(0, HMAX, value=1, step=1, label='Año tras el choque')
            a_btn = gr.Button('Generar maqueta 3D / RA', variant='primary')
            a_nota = interp('')
            with gr.Row():
                with gr.Column(scale=3):
                    a_html = gr.HTML()
                with gr.Column(scale=2):
                    try:
                        a_mod = gr.Model3D(label='Visor 3D (Gradio)', clear_color=(0.01, 0.02, 0.05, 1.0))
                    except TypeError:
                        a_mod = gr.Model3D(label='Visor 3D (Gradio)')
                    a_file = gr.File(label='Descargar GLB')
            a_btn.click(vista_ra, [a_fuente, a_var, a_anio, a_origen, a_tipo, a_mag, a_resp, a_h],
                        [a_mod, a_html, a_file, a_nota])

        # ---- 7. Modelo SpVAR ----
        with gr.Tab('Modelo SpVAR'):
            p_ec = gr.Radio(ECUS, value=ECUS[-1], label='Ecuación')
            p0 = vista_spvar(ECUS[-1])
            p_sel = grafico(p0[0], 'Selección W × modelo')
            with gr.Row():
                with gr.Column(scale=3):
                    p_mec = grafico(p0[1], 'Mecanismos')
                with gr.Column(scale=2):
                    p_coef = gr.Dataframe(value=p0[2], label='Coeficientes (MV)')
            p_txt = interp(p0[3])
            p_ec.change(vista_spvar, p_ec, [p_sel, p_mec, p_coef, p_txt])
            p_choque = gr.Dropdown(list(IRF_PUNTO), value=list(IRF_PUNTO)[0], label='Choque agregado (todos los departamentos)')
            p_irf = grafico(fig_irf_agregada(list(IRF_PUNTO)[0]), 'IRF agregada')
            p_choque.change(fig_irf_agregada, p_choque, p_irf)
            gr.Markdown("""#### Los cuatro mecanismos de transmisión
1. **Autodinámica espacial** — persistencia propia en el tiempo (φ) y su eco en los vecinos (ψ).
2. **Cruce local** — efecto del agro sobre la industria del mismo departamento, contemporáneo (β) y rezagado.
3. **Relación espacial directa** — contagio de la misma variable desde los vecinos (ρ, λ).
4. **Relación espacial cruzada** — el agro de los vecinos mueve la industria propia (θ, ψ cruzado).""")
            if 'AUD' in globals():
                with gr.Accordion('Auditoría del modelo', open=False):
                    gr.Dataframe(value=AUD, label='Verificaciones')
            if ARCHIVOS:
                gr.File(value=ARCHIVOS, file_count='multiple', label='Descargar resultados')

        # ---- 8. Interpretabilidad automatizada ----
        with gr.Tab('Interpretabilidad'):
            gr.Markdown('Texto generado exclusivamente a partir de las estimaciones y estadísticas calculadas en este '
                        'cuaderno. Cada afirmación indica su tipo de evidencia. Las pestañas de mapas, relieve, '
                        'simulador, ESDA y modelo incluyen además su interpretación sincronizada con los controles.')
            x_sec = gr.Radio(SECCIONES_INTERP, value='Panorama', label='Sección')
            x_txt = interp(vista_interpretacion('Panorama'))
            x_sec.change(vista_interpretacion, x_sec, x_txt)

    gr.Markdown(f"<center style='color:#4d7284'>SpVAR por máxima verosimilitud · selección {CONFIG['CRITERIO']} · "
                f"geometría {FUENTE_GEOM} · Plotly · Gradio</center>")

APP.queue().launch(share=True, debug=False, **KW_LAUNCH)